In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import mne
from scipy.io import loadmat
import os
import scipy
from scipy import signal
from scipy.stats import skew, kurtosis
import pywt
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import catboost
from catboost import CatBoostClassifier
import shap
import pandas as pd
from scipy.signal import hilbert, find_peaks, iirnotch
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
from multiprocessing import Pool
from tqdm import tqdm
import tempfile
import time
from sklearn.metrics import confusion_matrix, classification_report
import psutil

# Verify versions to confirm environment
print(f"NumPy version: {np.__version__}")
print(f"SciPy version: {scipy.__version__}")
print(f"CatBoost version: {catboost.__version__}")
print(f"MNE version: {mne.__version__}")
print(f"PyWavelets version: {pywt.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"SHAP version: {shap.__version__}")

# Function to monitor memory usage
def print_memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024 / 1024:.2f} MB")

# Global counter for Butterworth filter calls
butterworth_call_count = 0

# Helper function for error handling
def safe_compute(func, default, error_msg):
    try:
        return func()
    except Exception as e:
        print(f"{error_msg}: {e}")
        return default

# Preprocessing Pipeline
def load_and_segment_file(file_path, window_size=1024, stride=512, expected_channels=19):
    try:
        mat_data = loadmat(file_path)
        print(f"Loaded {file_path}: Contents {list(mat_data.keys())}")

        var_names = [k for k in mat_data.keys() if not k.startswith('__')]
        if not var_names:
            raise ValueError(f"No non-metadata variables found in {file_path}")

        possible_var_names = ['data', 'eeg', 'signal', 'EEG', 'Data'] + var_names
        data = None
        for var_name in possible_var_names:
            if var_name in mat_data:
                data = mat_data[var_name]
                print(f"Selected variable '{var_name}' from {file_path}")
                break
        if data is None:
            raise ValueError(f"No valid EEG data variable found in {file_path}. Tried {possible_var_names}")

        print(f"Initial data shape for {file_path}: {data.shape}, dtype: {data.dtype}")

        if data.ndim == 1:
            print(f"Data in {file_path} is 1D with shape {data.shape}. Reshaping to (1, n_samples, 1).")
            data = data.reshape(1, -1, 1)
        elif data.ndim == 2:
            print(f"Data in {file_path} is 2D with shape {data.shape}. Reshaping to (1, n_samples, n_channels).")
            data = data[np.newaxis, :, :]
        elif data.ndim == 3:
            print(f"Data in {file_path} is 3D with shape {data.shape}. Keeping as (n_segments, n_samples, n_channels).")
        else:
            raise ValueError(f"Unexpected data dimensions {data.ndim} in {file_path}. Shape: {data.shape}")

        n_segments, n_samples, n_channels = data.shape
        print(f"Processed data shape for {file_path}: {data.shape} (n_segments={n_segments}, n_samples={n_samples}, n_channels={n_channels})")

        if n_channels != expected_channels:
            if n_channels < expected_channels:
                print(f"Warning: {file_path} has {n_channels} channels, expected {expected_channels}. Padding with zeros.")
                data = np.pad(data, ((0, 0), (0, 0), (0, expected_channels - n_channels)), mode='constant')
            else:
                print(f"Warning: {file_path} has {n_channels} channels, expected {expected_channels}. Truncating channels.")
                data = data[:, :, :expected_channels]
            n_channels = expected_channels

        max_abs_value = np.max(np.abs(data))
        if max_abs_value > 1000:
            print(f"Data appears to be in mV with large scaling. Converting to µV (dividing by 1000).")
            data = data / 1000.0
            max_abs_value = np.max(np.abs(data))
            print(f"After conversion to µV: Min {np.min(data)}, Max {np.max(data)}")

        if max_abs_value > 100:
            print(f"Data exceeds typical EEG range (±100 µV). Normalizing to ±100 µV.")
            scaling_factor = 100 / max_abs_value
            data = data * scaling_factor
            print(f"After scaling: Min {np.min(data)}, Max {np.max(data)}")

        all_segments = []
        for seg in range(n_segments):
            seg_data = data[seg]
            seg_samples = seg_data.shape[0]
            if seg_samples < window_size:
                padded_data = np.pad(seg_data, ((0, window_size - seg_samples), (0, 0)), mode='constant')
                print(f"Padded segment {seg+1} in {file_path} to {window_size} samples, {n_channels} channels")
                all_segments.append(padded_data[np.newaxis, :, :])
                continue

            n_seg_segments = ((seg_samples - window_size + stride - 1) // stride) + 1
            segments = np.zeros((n_seg_segments, window_size, n_channels))
            for i in range(n_seg_segments):
                start = i * stride
                end = min(start + window_size, seg_samples)
                segment_length = end - start
                if segment_length == window_size:
                    segments[i] = seg_data[start:end, :]
                else:
                    segments[i] = np.pad(seg_data[start:end, :],
                                       ((0, window_size - segment_length), (0, 0)),
                                       mode='constant')
                print(f"Segment {i+1} from segment {seg+1} in {file_path} has shape {segments[i].shape}")
            all_segments.append(segments)

        if not all_segments:
            print(f"No segments created for {file_path}")
            return None

        final_segments = np.concatenate(all_segments, axis=0)
        print(f"Segmented {file_path} into {final_segments.shape[0]} segments with shape {final_segments.shape}")
        return final_segments

    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")
        return None

def load_from_folder(folder_path, window_size=1024, stride=512, expected_channels=19):
    if not os.path.exists(folder_path):
        print(f"Directory {folder_path} does not exist. Skipping.")
        return None

    mat_files = [f for f in os.listdir(folder_path) if f.endswith('.mat')]
    if not mat_files:
        print(f"No .mat files found in {folder_path}. Skipping.")
        return None

    all_segments = []
    print(f"Scanning {folder_path} for .mat files... Found {len(mat_files)} files.")
    for file in mat_files:
        file_path = os.path.join(folder_path, file)
        print(f"Attempting to process {file_path}")
        segments = load_and_segment_file(file_path, window_size, stride, expected_channels)
        if segments is not None:
            all_segments.append(segments)
            print(f"Successfully processed {file}: {len(segments)} segments with shape {segments.shape}")
        else:
            print(f"Failed to process {file}, segment data is None")
    
    if not all_segments:
        print(f"No valid segments found in {folder_path} after processing all files")
        return None
    
    try:
        concatenated_segments = np.concatenate(all_segments, axis=0)
        print(f"Concatenated {len(all_segments)} segment arrays from {folder_path} with shape {concatenated_segments.shape}")
        return concatenated_segments
    except ValueError as e:
        print(f"Error concatenating segments in {folder_path}: {str(e)}. Segment shapes: {[seg.shape for seg in all_segments if seg is not None]}")
        return None

def load_all_data(window_size=1024, stride=512, expected_channels=19):
    # Update these paths to your local directory structure
    adhd_path1 = r"C:\Users\ADMIN\Downloads\Calcium signalling research\ADHD_part1\ADHD_part1"# Replace with actual path
    adhd_path2 = r"C:\Users\ADMIN\Downloads\Calcium signalling research\ADHD_part2\ADHD_part2"  # Replace with actual path
    control_path1 = r"C:\Users\ADMIN\Downloads\Calcium signalling research\Control_part1\Control_part1"  # Replace with actual path
    control_path2 = r"C:\Users\ADMIN\Downloads\Calcium signalling research\Control_part2" # Replace with actual path

    for path in [adhd_path1, adhd_path2, control_path1, control_path2]:
        if not os.path.exists(path):
            print(f"Warning: Path {path} does not exist. Please update the path.")
        else:
            mat_files = [f for f in os.listdir(path) if f.endswith('.mat')]
            print(f"Directory {path} contains {len(mat_files)} .mat files.")

    print("\nProcessing ADHD data...")
    adhd_segments1 = load_from_folder(adhd_path1, window_size, stride, expected_channels)
    print(f"Debug: adhd_segments1 shape: {adhd_segments1.shape if adhd_segments1 is not None else 'None'}")
    adhd_segments2 = load_from_folder(adhd_path2, window_size, stride, expected_channels)
    print(f"Debug: adhd_segments2 shape: {adhd_segments2.shape if adhd_segments2 is not None else 'None'}")

    adhd_segments = None
    if adhd_segments1 is not None and adhd_segments2 is not None:
        try:
            adhd_segments = np.concatenate([adhd_segments1, adhd_segments2], axis=0)
            print(f"Successfully concatenated ADHD segments with shape {adhd_segments.shape}")
        except ValueError as e:
            print(f"Error concatenating ADHD segments: {str(e)}. Shapes - adhd_segments1: {adhd_segments1.shape}, adhd_segments2: {adhd_segments2.shape}")
            adhd_segments = adhd_segments1
    elif adhd_segments1 is not None:
        print("Warning: No valid segments from ADHD_part2, proceeding with ADHD_part1 only.")
        adhd_segments = adhd_segments1
    elif adhd_segments2 is not None:
        print("Warning: No valid segments from ADHD_part1, proceeding with ADHD_part2 only.")
        adhd_segments = adhd_segments2
    else:
        print("Warning: No valid ADHD data loaded from either ADHD_part1 or ADHD_part2. Proceeding with Control data if available.")

    print("\nProcessing Control data...")
    control_segments1 = load_from_folder(control_path1, window_size, stride, expected_channels)
    control_segments2 = load_from_folder(control_path2, window_size, stride, expected_channels)

    control_segments = None
    if control_segments1 is not None and control_segments2 is not None:
        try:
            control_segments = np.concatenate([control_segments1, control_segments2], axis=0)
            print(f"Successfully concatenated Control segments with shape {control_segments.shape}")
        except ValueError as e:
            print(f"Error concatenating Control segments: {str(e)}. Shapes - control_segments1: {control_segments1.shape}, control_segments2: {control_segments2.shape}")
            control_segments = control_segments1
    elif control_segments1 is not None:
        print("Warning: No valid segments from Control_part2, proceeding with Control_part1 only.")
        control_segments = control_segments1
    elif control_segments2 is not None:
        print("Warning: No valid segments from Control_part1, proceeding with Control_part2 only.")
        control_segments = control_segments2
    else:
        print("Warning: No valid Control data loaded from either Control_part1 or Control_part2. Proceeding with ADHD data if available.")

    if adhd_segments is None and control_segments is None:
        raise ValueError("No valid data loaded from any directory. Please check the data paths and file contents.")

    y_adhd = np.ones(len(adhd_segments), dtype=int) if adhd_segments is not None else np.array([])
    y_control = np.zeros(len(control_segments), dtype=int) if control_segments is not None else np.array([])

    X = np.concatenate([seg for seg in [adhd_segments, control_segments] if seg is not None], axis=0) if (adhd_segments is not None or control_segments is not None) else np.array([])
    y = np.concatenate([y_adhd, y_control]) if len(y_adhd) > 0 or len(y_control) > 0 else np.array([])

    print(f"\nTotal ADHD segments: {len(adhd_segments) if adhd_segments is not None else 0}")
    print(f"Total Control segments: {len(control_segments) if control_segments is not None else 0}")
    print(f"Final data shape: {X.shape if X.size > 0 else 'Empty'}")
    print(f"Class balance: {np.bincount(y) if y.size > 0 else 'Empty'}")

    if X.size == 0:
        raise ValueError("No segments available after concatenation. Cannot proceed with empty data.")

    X = notch_filter(X)
    X = clip_spikes(X, threshold=500)
    X = butterworth_filter(X)
    X = apply_ica_and_plot(X, segments_to_plot=3, channels_to_plot=expected_channels)

    print(f"Validating X shape: {X.shape}")
    assert X.ndim == 3, f"Expected 3D array, got {X.ndim}D array with shape {X.shape}"
    assert X.shape[1] == 1024, f"Expected 1024 samples per segment in X, got {X.shape[1]}"
    assert X.shape[2] == expected_channels, f"Expected {expected_channels} channels, but found {X.shape[2]} channels in X"

    return X, y

def notch_filter(data, fs=128, freqs=[50, 60], Q=30):
    print(f"Applying notch filter to remove {freqs} Hz noise...")
    assert data.ndim == 3, f"Expected 3D array for notch_filter, got {data.ndim}D with shape {data.shape}"
    filtered_data = data.copy()
    for freq in freqs:
        b, a = signal.iirnotch(freq / (fs / 2), Q)
        filtered_data = signal.filtfilt(b, a, filtered_data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    print(f"Notch filter applied, output shape: {filtered_data.shape}")
    return filtered_data

def clip_spikes(data, threshold=500):
    print(f"Clipping spikes with absolute amplitude > {threshold} µV...")
    assert data.ndim == 3, f"Expected 3D array for clip_spikes, got {data.ndim}D with shape {data.shape}"
    clipped_data = data.copy()
    clipped_data[np.abs(clipped_data) > threshold] = np.sign(clipped_data[np.abs(clipped_data) > threshold]) * threshold
    fp2_threshold = 30
    clipped_data[:, :, 1] = np.where(np.abs(clipped_data[:, :, 1]) > fp2_threshold,
                                   np.sign(clipped_data[:, :, 1]) * fp2_threshold,
                                   clipped_data[:, :, 1])
    print(f"Fp2 after clipping: Min {np.min(clipped_data[:, :, 1])}, Max {np.max(clipped_data[:, :, 1])} µV")
    print(f"Clip spikes applied, output shape: {clipped_data.shape}")
    return clipped_data

def butterworth_filter(data, fs=128, lowcut=1, highcut=45, order=4):
    global butterworth_call_count
    butterworth_call_count += 1
    assert data.ndim == 3, f"Expected 3D array for butterworth_filter, got {data.ndim}D with shape {data.shape}"
    nyquist = fs / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = signal.butter(order, [low, high], btype='band')
    filtered_data = signal.filtfilt(b, a, data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    print(f"Butterworth filter applied (call {butterworth_call_count}), output shape: {filtered_data.shape}")
    return filtered_data

def apply_ica_and_plot(data, fs=128, n_components=19, segments_to_plot=5, channels_to_plot=19):
    print("Applying ICA for artifact removal...")
    n_segments, n_samples, n_channels = data.shape
    total_samples = n_segments * n_samples
    concatenated_data = data.reshape(total_samples, n_channels)

    ch_names = [f"EEG {i}" for i in range(n_channels)]
    montage_labels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
    theta_values = [-18, 18, -54, -39, 0, 39, 54, -90, -90, 90, 90, 90, -126, -141, 180, 141, 126, -162, 162]
    radius_values_cm = [51.1, 51.1, 51.1, 33.3, 25.6, 33.3, 51.1, 51.1, 25.6, 0, 25.6, 51.1, 51.1, 33.3, 25.6, 33.3, 51.1, 51.1, 51.1]
    radius_values_m = [r / 100 for r in radius_values_cm]

    info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types="eeg")
    raw = mne.io.RawArray(concatenated_data.T, info)
    channel_mapping = {f"EEG {i}": montage_labels[i] for i in range(n_channels)}
    raw.rename_channels(channel_mapping)

    try:
        ch_pos = {label: [radius * np.cos(np.deg2rad(theta)), radius * np.sin(np.deg2rad(theta)), 0.0] 
                  for label, theta, radius in zip(montage_labels, theta_values, radius_values_m)}
        montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='head')
        raw.set_montage(montage)
    except Exception as e:
        print(f"Error applying custom montage: {str(e)}. Using standard 10-20.")
        montage = mne.channels.make_standard_montage('standard_1020')
        raw.set_montage(montage)

    raw.filter(l_freq=0.5, h_freq=None, method='iir')
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=42, max_iter=500)
    ica.fit(raw)
    ica_sources = ica.get_sources(raw).get_data().T

    kurtosis_values = np.array([kurtosis(ica_sources[:, i]) for i in range(n_components)])
    variance_values = np.var(ica_sources, axis=0)
    peak_amplitude = np.max(np.abs(ica_sources), axis=0)
    fp2_weight = np.max(np.abs(ica_sources[total_samples // 2:total_samples // 2 + n_samples, 1])) / np.max(peak_amplitude)
    kurtosis_norm = (kurtosis_values - kurtosis_values.min()) / (kurtosis_values.max() - kurtosis_values.min() + 1e-10)
    variance_norm = (variance_values - variance_values.min()) / (variance_values.max() - variance_values.min() + 1e-10)
    peak_norm = (peak_amplitude - peak_amplitude.min()) / (peak_amplitude.max() - peak_amplitude.min() + 1e-10)
    combined_score = 0.45 * kurtosis_norm + 0.25 * variance_norm + 0.2 * peak_norm + 0.1 * fp2_weight
    top_artifact_indices = np.argsort(combined_score)[-10:]
    ica.exclude = top_artifact_indices.tolist()

    raw_clean = ica.apply(raw.copy())
    cleaned_data_raw = raw_clean.get_data()
    cleaned_data = cleaned_data_raw.T.reshape(n_segments, n_samples, n_channels)

    for ch in [6, 15, 16]:
        plt.figure(figsize=(10, 5))
        plt.plot(data[0, :, ch], label=f'Raw ch{ch}')
        plt.plot(cleaned_data[0, :, ch], label=f'Cleaned ch{ch}')
        plt.title(f"Raw vs. Cleaned Data for Channel {ch}")
        plt.legend()
        plt.savefig(f"ica_comparison_ch{ch}.png")
        plt.close()

    return cleaned_data

# Feature Extraction Functions
def normalize_data(data):
    scaler = StandardScaler()
    return scaler.fit_transform(data)

def calculate_engagement_index(data, fs=128):
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    alpha = calculate_psd(data, fs, freq_range=(8, 13))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    return np.mean((theta + alpha) / beta)

def calculate_psd(data, fs=128, freq_range=None):
    f, psd = signal.welch(data, fs=fs, nperseg=256, axis=0)
    if freq_range:
        freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
        return np.mean(psd[freq_mask], axis=0)
    return psd.mean(axis=0)

def phase_amplitude_coupling(data, fs=128):
    theta = butterworth_filter(data, fs, 4, 8)
    alpha = butterworth_filter(data, fs, 8, 13)
    beta = butterworth_filter(data, fs, 13, 30)

    theta_phase = np.angle(hilbert(theta))
    alpha_amp = np.abs(hilbert(alpha))
    beta_amp = np.abs(hilbert(beta))

    pac_theta_alpha = np.mean(np.abs(np.mean(alpha_amp * np.exp(1j * theta_phase), axis=0)))
    pac_theta_beta = np.mean(np.abs(np.mean(beta_amp * np.exp(1j * theta_phase), axis=0)))
    pac_alpha_beta = np.mean(np.abs(np.mean(beta_amp * np.exp(1j * np.angle(hilbert(alpha))), axis=0)))

    return np.array([pac_theta_alpha, pac_theta_beta, pac_alpha_beta])

def calculate_sprint(data, fs=128):
    f, t, Zxx = signal.stft(data, fs=fs, nperseg=256, noverlap=128)
    power = np.abs(Zxx)**2
    return np.mean(power, axis=(0, 1))

def wavelet_features(data):
    coeffs = pywt.wavedec(data, 'db4', level=5, axis=0)
    features = []
    for c in coeffs:
        features.extend([np.mean(np.abs(c)), np.std(c), np.max(np.abs(c))])
    return np.array(features)

def microstate_analysis(data, fs=128, n_states=4):
    gfp = np.std(data, axis=1)
    peaks, _ = find_peaks(gfp, distance=fs//10)

    if len(peaks) < n_states:
        peak_data = data
    else:
        peak_data = data[peaks]

    kmeans = KMeans(n_clusters=n_states, random_state=42)
    kmeans.fit(peak_data)
    templates = kmeans.cluster_centers_

    distances = np.zeros((data.shape[0], n_states))
    for i, template in enumerate(templates):
        template_norm = np.linalg.norm(template)
        data_norm = np.linalg.norm(data, axis=1)
        distances[:, i] = np.abs(np.sum(data * template, axis=1) / (data_norm * template_norm + 1e-10))

    labels = np.argmax(distances, axis=1)

    durations, occurrences = [], []
    for state in range(n_states):
        state_indices = np.where(labels == state)[0]
        if len(state_indices) == 0:
            durations.append(0)
            occurrences.append(0)
            continue
        diff = np.diff(state_indices)
        breaks = np.where(diff > 1)[0]
        segment_lengths = np.diff(np.concatenate([[0], breaks + 1, [len(state_indices)]]))
        durations.append(np.mean(segment_lengths) / fs * 1000)
        occurrences.append(len(breaks) + 1)

    coverage = np.bincount(labels, minlength=n_states) / len(labels)
    transitions = np.zeros((n_states, n_states))
    for t in range(len(labels)-1):
        transitions[labels[t], labels[t+1]] += 1
    transitions = transitions / (transitions.sum(axis=1, keepdims=True) + 1e-10)

    return np.concatenate([durations, occurrences, coverage, transitions.flatten()])

def calculate_hurst_exponent(time_series, max_lag=20):
    n = len(time_series)
    if n < max_lag:
        return 0.5
    lags = range(2, max_lag + 1)
    rs = []
    for lag in lags:
        blocks = [time_series[i:i + lag] for i in range(0, n - lag + 1, lag) if i + lag <= n]
        if not blocks:
            continue
        rs_block = [((np.max(np.cumsum(block - np.mean(block))) - np.min(np.cumsum(block - np.mean(block)))) / np.std(block)) 
                    for block in blocks if np.std(block) > 0]
        if rs_block:
            rs.append(np.mean(rs_block))
    if not rs or len(lags) < 2:
        return 0.5
    coeffs = np.polyfit(np.log(lags), np.log(rs), 1)
    return np.clip(coeffs[0], 0.0, 1.0)

def extract_features_single(args):
    global butterworth_call_count
    sample, idx, temp_dir, prev_features = args  # Added prev_features to pass previous segment's features
    print(f"Processing segment {idx+1}...")

    start_time = time.time()
    n_channels = 19
    filtered = butterworth_filter(sample)
    normalized = normalize_data(filtered)

    microstates = microstate_analysis(normalized)
    channel_features = []
    feature_types = ['engagement', 'theta', 'alpha', 'beta', 'pac_theta_alpha', 'pac_theta_beta', 
                     'pac_alpha_beta', 'sprint', 'wavelet_mean1', 'wavelet_std1', 'wavelet_max1',
                     'wavelet_mean2', 'wavelet_std2', 'wavelet_max2', 'wavelet_mean3', 'wavelet_std3', 
                     'wavelet_max3', 'wavelet_mean4', 'wavelet_std4', 'wavelet_max4', 'wavelet_mean5', 
                     'wavelet_std5', 'wavelet_max5', 'wavelet_mean6', 'wavelet_std6', 'wavelet_max6', 
                     'skewness', 'kurtosis', 'hurst']
    expected_features_per_channel = len(feature_types)  # 29 features per channel
    expected_total_features = n_channels * expected_features_per_channel + len(microstates)

    for ch in range(n_channels):
        channel_data = normalized[:, ch]
        features = []
        
        # Calculate each feature with error handling
        engagement = safe_compute(lambda: calculate_engagement_index(channel_data), 0, f"Segment {idx+1}, ch{ch}: Engagement index failed")
        theta = safe_compute(lambda: calculate_psd(channel_data, freq_range=(4, 8)), 0, f"Segment {idx+1}, ch{ch}: Theta PSD failed")
        alpha = safe_compute(lambda: calculate_psd(channel_data, freq_range=(8, 13)), 0, f"Segment {idx+1}, ch{ch}: Alpha PSD failed")
        beta = safe_compute(lambda: calculate_psd(channel_data, freq_range=(13, 30)), 0, f"Segment {idx+1}, ch{ch}: Beta PSD failed")
        pac = safe_compute(lambda: phase_amplitude_coupling(channel_data), np.zeros(3), f"Segment {idx+1}, ch{ch}: PAC failed")
        sprint = safe_compute(lambda: calculate_sprint(channel_data), 0, f"Segment {idx+1}, ch{ch}: Sprint failed")
        wavelet = safe_compute(lambda: wavelet_features(channel_data), np.zeros(18), f"Segment {idx+1}, ch{ch}: Wavelet features failed")
        skewness = safe_compute(lambda: skew(channel_data), 0, f"Segment {idx+1}, ch{ch}: Skewness failed")
        kurt = safe_compute(lambda: kurtosis(channel_data), 0, f"Segment {idx+1}, ch{ch}: Kurtosis failed")
        hurst = safe_compute(lambda: calculate_hurst_exponent(channel_data), 0.5, f"Segment {idx+1}, ch{ch}: Hurst exponent failed")

        ch_features = np.concatenate([
            [engagement], [theta], [alpha], [beta], pac, [sprint],
            wavelet, [skewness], [kurt], [hurst]
        ])
        channel_features.append(ch_features)

    # Combine channel features and microstates
    channel_features_flat = np.array(channel_features).flatten()
    sample_features = np.concatenate([channel_features_flat, microstates])

    # Verify features (your requested tracking line)
    if len(channel_features) == n_channels and len(sample_features) == expected_total_features:
        print(f"Segment {idx+1}: All features extracted successfully. Filter applied: {butterworth_call_count > 0}. 19 channels present: Yes")
    else:
        print(f"Segment {idx+1}: Feature extraction incomplete. Expected {expected_total_features}, got {len(sample_features)}")
        if prev_features is not None and len(prev_features) == expected_total_features:
            missing_count = expected_total_features - len(sample_features)
            print(f"Segment {idx+1}: Borrowing {missing_count} missing features from previous segment")
            sample_features = np.concatenate([sample_features, prev_features[-missing_count:]])
        else:
            print(f"Segment {idx+1}: No previous features available to borrow. Padding with zeros")
            sample_features = np.pad(sample_features, (0, expected_total_features - len(sample_features)), mode='constant')

    temp_file = os.path.join(temp_dir, f"segment_{idx}.npy")
    np.save(temp_file, sample_features)
    end_time = time.time()
    print(f"Finished processing segment {idx+1} in {end_time - start_time:.2f} seconds")
    return temp_file, sample_features  # Return features for use as prev_features in next iteration

def extract_all_features(data, n_channels=19, chunk_size=20):
    n_samples = data.shape[0]
    print(f"Starting feature extraction for {n_samples} segments with {os.cpu_count()} cores")

    with tempfile.TemporaryDirectory() as temp_dir:
        feature_files = []
        prev_features = None  # Initialize previous features
        with Pool(processes=os.cpu_count()) as pool:
            for i in range(0, n_samples, chunk_size):
                chunk = data[i:i + chunk_size]
                chunk_indices = range(i, min(i + chunk_size, n_samples))
                tasks = [(chunk[j], idx, temp_dir, prev_features) for j, idx in enumerate(chunk_indices)]

                results = []
                for task in tasks:
                    result = pool.apply_async(extract_features_single, (task,))
                    results.append(result)

                for r in tqdm(results, total=len(tasks), desc=f"Processing segments {i+1}-{min(i+chunk_size, n_samples)}"):
                    temp_file, current_features = r.get()
                    feature_files.append(temp_file)
                    prev_features = current_features  # Update prev_features for next iteration

        print("Loading features from disk...")
        feature_matrix = []
        for temp_file in tqdm(feature_files, desc="Loading features"):
            features = np.load(temp_file)
            feature_matrix.append(features)

    return np.array(feature_matrix)

# Feature Selection
def select_features(X, y, feature_names):
    from sklearn.feature_selection import f_classif
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

    initial_model = CatBoostClassifier(random_state=42, verbose=0)
    initial_model.fit(X_train, y_train)

    explainer = shap.TreeExplainer(initial_model)
    initial_shap_values = explainer.shap_values(X_train)
    shap_importance = np.abs(initial_shap_values).mean(axis=0)

    n_features_to_select = int(X.shape[1] * 0.25)
    important_indices = np.argsort(shap_importance)[-n_features_to_select:]
    X_shap_selected = X[:, important_indices]

    f_values, p_values = f_classif(X_shap_selected, y)
    significant_features = p_values < 0.01
    X_anova_selected = X_shap_selected[:, significant_features]

    pca = PCA(n_components=0.95)
    X_final = pca.fit_transform(X_anova_selected)
    X_train_pca = pca.transform(X_train[:, important_indices][:, significant_features])
    X_val_pca = pca.transform(X_val[:, important_indices][:, significant_features])
    X_test_pca = pca.transform(X_test[:, important_indices][:, significant_features])

    selected_feature_names = np.array(feature_names)[important_indices]
    selected_features = selected_feature_names[significant_features]
    pca_loadings = pca.components_
    pca_loadings_df = pd.DataFrame(pca_loadings, columns=selected_features, index=[f"PCA_{i}" for i in range(pca.n_components_)])

    return (X_final, initial_model, initial_shap_values, selected_features, important_indices,
            significant_features, pca, X_train_pca, X_val_pca, X_test_pca, X_train, X_val, X_test, y_train, y_val, y_test, pca_loadings_df)

# Evaluation Metrics
def evaluate_model(model, X, y, dataset_name="Dataset"):
    y_pred = model.predict(X)
    cm = confusion_matrix(y, y_pred)
    report = classification_report(y, y_pred, output_dict=True)

    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = report['weighted avg']['f1-score']
    recall = report['weighted avg']['recall']

    print(f"\n{dataset_name} Confusion Matrix:")
    print(cm)
    print(f"\n{dataset_name} Evaluation Metrics:")
    print(f"Sensitivity: {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Recall: {recall:.4f}")
    return cm, sensitivity, specificity, f1, recall

# Hyperparameter Tuning for CatBoost
def tune_catboost(X_train, y_train):
    param_dist = {
        'iterations': [100, 200, 300],
        'depth': [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1],
        'l2_leaf_reg': [1, 3, 5],
        'border_count': [32, 50, 100]
    }
    catboost = CatBoostClassifier(random_state=42, verbose=0)
    search = RandomizedSearchCV(catboost, param_distributions=param_dist, n_iter=20, cv=5,
                                scoring='accuracy', random_state=42, n_jobs=-1)
    search.fit(X_train, y_train)
    print(f"Best CatBoost parameters: {search.best_params_}")
    print(f"Best CatBoost CV accuracy: {search.best_score_:.4f}")
    return search.best_estimator_

# Main Function
def main():
    print("Loading and preprocessing data...")
    X, y = load_all_data()

    print("Extracting features...")
    start_time = time.time()
    X_features = extract_all_features(X)
    end_time = time.time()
    print(f"Feature extraction completed in {int((end_time - start_time) / 60)} minutes {int((end_time - start_time) % 60)} seconds")

    n_channels = 19
    feature_types = ['engagement', 'theta', 'alpha', 'beta',
                    'pac_theta_alpha', 'pac_theta_beta', 'pac_alpha_beta',
                    'sprint',
                    'wavelet_mean1', 'wavelet_std1', 'wavelet_max1',
                    'wavelet_mean2', 'wavelet_std2', 'wavelet_max2',
                    'wavelet_mean3', 'wavelet_std3', 'wavelet_max3',
                    'wavelet_mean4', 'wavelet_std4', 'wavelet_max4',
                    'wavelet_mean5', 'wavelet_std5', 'wavelet_max5',
                    'wavelet_mean6', 'wavelet_std6', 'wavelet_max6',
                    'skewness', 'kurtosis',
                    'hurst']
    n_states = 4
    microstate_features = ([f'ms_duration_{i}' for i in range(n_states)] +
                         [f'ms_occurrence_{i}' for i in range(n_states)] +
                         [f'ms_coverage_{i}' for i in range(n_states)] +
                         [f'ms_trans_{i}_{j}' for i in range(n_states) for j in range(n_states)])
    feature_names = ([f"{ft}_ch{ch}" for ch in range(n_channels) for ft in feature_types] +
                    microstate_features)

    print("Selecting features...")
    (X_final, initial_model, initial_shap_values, selected_features, shap_indices,
     anova_mask, pca, X_train_pca, X_val_pca, X_test_pca, X_train, X_val, X_test, y_train, y_val, y_test, pca_loadings_df) = select_features(X_features, y, feature_names)

    print("\nPCA Loadings for top components (e.g., PCA_0, PCA_8, PCA_10):")
    for comp in ['PCA_0', 'PCA_8', 'PCA_10']:
        if comp in pca_loadings_df.index:
            print(f"{comp} Loadings:")
            print(pca_loadings_df.loc[comp].sort_values(ascending=False).head(5))

    print("\nTuning CatBoost...")
    catboost_model = tune_catboost(X_train_pca, y_train)

    print("\nPerforming 10-fold cross-validation with StratifiedKFold...")
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    cv_scores = cross_val_score(catboost_model, X_final, y, cv=cv, scoring='accuracy')
    print(f"CatBoost CV scores: {cv_scores}")
    print(f"CatBoost Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

    print("\nTraining CatBoost on train set...")
    catboost_model.fit(X_train_pca, y_train)

    print("\nEvaluating CatBoost on validation set...")
    cm_val, sens_val, spec_val, f1_val, recall_val = evaluate_model(catboost_model, X_val_pca, y_val, "Validation")

    print("\nEvaluating CatBoost on test set...")
    cm_test, sens_test, spec_test, f1_test, recall_test = evaluate_model(catboost_model, X_test_pca, y_test, "Test")

    print("\nComputing SHAP values for CatBoost on validation set...")
    explainer = shap.TreeExplainer(catboost_model)
    shap_values = explainer.shap_values(X_val_pca)

    shap.summary_plot(shap_values, X_val_pca, feature_names=[f"PCA_{i}" for i in range(X_val_pca.shape[1])], show=False)
    plt.show()
    X_features_df = pd.DataFrame(X_features, columns=feature_names)
    shap.summary_plot(initial_shap_values, X_features_df, plot_type="bar", max_display=20, show=False)
    plt.show()

    print("\nTop 10 selected features:")
    for i, feature in enumerate(selected_features[:10]):
        print(f"{i+1}. {feature}")

if __name__ == "__main__":
    main()

NumPy version: 1.26.4
SciPy version: 1.13.1
CatBoost version: 1.2.7
MNE version: 1.8.0
PyWavelets version: 1.7.0
Scikit-learn version: 1.5.1
SHAP version: 0.46.0
Loading and preprocessing data...
Directory C:\Users\ADMIN\Downloads\Calcium signalling research\ADHD_part1\ADHD_part1 contains 30 .mat files.
Directory C:\Users\ADMIN\Downloads\Calcium signalling research\ADHD_part2\ADHD_part2 contains 31 .mat files.
Directory C:\Users\ADMIN\Downloads\Calcium signalling research\Control_part1\Control_part1 contains 30 .mat files.
Directory C:\Users\ADMIN\Downloads\Calcium signalling research\Control_part2 contains 30 .mat files.

Processing ADHD data...
Scanning C:\Users\ADMIN\Downloads\Calcium signalling research\ADHD_part1\ADHD_part1 for .mat files... Found 30 files.
Attempting to process C:\Users\ADMIN\Downloads\Calcium signalling research\ADHD_part1\ADHD_part1\v10p.mat
Loaded C:\Users\ADMIN\Downloads\Calcium signalling research\ADHD_part1\ADHD_part1\v10p.mat: Contents ['__header__', '__ve

In [38]:
import numpy as np
from scipy.io import loadmat
import os
from scipy import signal
from scipy.stats import skew, kurtosis
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from sklearn.decomposition import PCA
import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import shap
import pandas as pd
from scipy.signal import hilbert, find_peaks
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import mne
from tqdm import tqdm
import tempfile
import time
import psutil
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

def print_memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024 / 1024:.2f} MB")

def load_and_segment_file(file_path, window_size=1024, stride=512, expected_channels=19):
    mat_data = loadmat(file_path)
    var_names = [k for k in mat_data.keys() if not k.startswith('__')]
    if not var_names:
        raise ValueError(f"No non-metadata variables found in {file_path}")
    possible_var_names = ['data', 'eeg', 'signal', 'EEG', 'Data'] + var_names
    data = None
    for var_name in possible_var_names:
        if var_name in mat_data:
            data = mat_data[var_name]
            break
    if data is None:
        raise ValueError(f"No valid EEG data variable found in {file_path}. Tried {possible_var_names}")
    if data.ndim == 1:
        data = data.reshape(1, -1, 1)
    elif data.ndim == 2:
        data = data[np.newaxis, :, :]
    elif data.ndim == 3:
        pass
    else:
        raise ValueError(f"Unexpected data dimensions {data.ndim} in {file_path}. Shape: {data.shape}")
    n_segments, n_samples, n_channels = data.shape
    if n_channels != expected_channels:
        if n_channels < expected_channels:
            data = np.pad(data, ((0, 0), (0, 0), (0, expected_channels - n_channels)), mode='constant')
        else:
            data = data[:, :, :expected_channels]
        n_channels = expected_channels
    max_abs_value = np.max(np.abs(data))
    if max_abs_value > 1000:
        data = data / 1000.0
    if max_abs_value > 100:
        scaling_factor = 100 / max_abs_value
        data = data * scaling_factor
    all_segments = []
    for seg in range(n_segments):
        seg_data = data[seg]
        seg_samples = seg_data.shape[0]
        if seg_samples < window_size:
            padded_data = np.pad(seg_data, ((0, window_size - seg_samples), (0, 0)), mode='constant')
            all_segments.append(padded_data[np.newaxis, :, :])
            continue
        n_seg_segments = ((seg_samples - window_size + stride - 1) // stride) + 1
        segments = np.zeros((n_seg_segments, window_size, n_channels))
        for i in range(n_seg_segments):
            start = i * stride
            end = min(start + window_size, seg_samples)
            segment_length = end - start
            if segment_length == window_size:
                segments[i] = seg_data[start:end, :]
            else:
                segments[i] = np.pad(seg_data[start:end, :], ((0, window_size - segment_length), (0, 0)), mode='constant')
        all_segments.append(segments)
    if not all_segments:
        return None
    final_segments = np.concatenate(all_segments, axis=0)
    return final_segments

def load_from_folder(folder_path, window_size=1024, stride=512, expected_channels=19):
    if not os.path.exists(folder_path):
        return None
    mat_files = [f for f in os.listdir(folder_path) if f.endswith('.mat')]
    if not mat_files:
        return None
    all_segments = []
    for file in mat_files:
        file_path = os.path.join(folder_path, file)
        segments = load_and_segment_file(file_path, window_size, stride, expected_channels)
        if segments is not None:
            all_segments.append(segments)
    if not all_segments:
        return None
    concatenated_segments = np.concatenate(all_segments, axis=0)
    return concatenated_segments

def load_all_data(window_size=1024, stride=512, expected_channels=19):
    adhd_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1"
    adhd_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part2/ADHD_part2"
    control_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part1/Control_part1"
    control_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part2"
    adhd_segments1 = load_from_folder(adhd_path1, window_size, stride, expected_channels)
    adhd_segments2 = load_from_folder(adhd_path2, window_size, stride, expected_channels)
    if adhd_segments1 is None or adhd_segments2 is None:
        raise ValueError("Failed to load ADHD data")
    adhd_segments = np.concatenate([adhd_segments1, adhd_segments2], axis=0)
    control_segments1 = load_from_folder(control_path1, window_size, stride, expected_channels)
    control_segments2 = load_from_folder(control_path2, window_size, stride, expected_channels)
    if control_segments1 is None or control_segments2 is None:
        raise ValueError("Failed to load Control data")
    control_segments = np.concatenate([control_segments1, control_segments2], axis=0)
    y_adhd = np.ones(len(adhd_segments))
    y_control = np.zeros(len(control_segments))
    X = np.concatenate([adhd_segments, control_segments], axis=0)
    y = np.concatenate([y_adhd, y_control])
    X = notch_filter(X)
    X = clip_spikes(X)
    X = butterworth_filter(X)
    X = apply_ica_and_plot(X, segments_to_plot=3, channels_to_plot=expected_channels)
    print("Data loading and preprocessing completed.")
    return X, y

def notch_filter(data, fs=128, freqs=[50, 60], Q=30):
    filtered_data = data.copy()
    for freq in freqs:
        b, a = signal.iirnotch(freq / (fs / 2), Q)
        filtered_data = signal.filtfilt(b, a, filtered_data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def clip_spikes(data, threshold=500):
    clipped_data = data.copy()
    clipped_data[np.abs(clipped_data) > threshold] = np.sign(clipped_data[np.abs(clipped_data) > threshold]) * threshold
    fp2_threshold = 30
    clipped_data[:, :, 1] = np.where(np.abs(clipped_data[:, :, 1]) > fp2_threshold,
                                     np.sign(clipped_data[:, :, 1]) * fp2_threshold,
                                     clipped_data[:, :, 1])
    return clipped_data

def butterworth_filter(data, fs=128, lowcut=1, highcut=45, order=4):
    nyquist = fs / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = signal.butter(order, [low, high], btype='band')
    filtered_data = signal.filtfilt(b, a, data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def apply_ica_and_plot(data, fs=128, n_components=19, segments_to_plot=5, channels_to_plot=19):
    n_segments, n_samples, n_channels = data.shape
    total_samples = n_segments * n_samples
    concatenated_data = data.reshape(total_samples, n_channels)
    ch_names = [f"EEG {i}" for i in range(n_channels)]
    montage_labels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
    info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types="eeg")
    raw = mne.io.RawArray(concatenated_data.T, info)
    channel_mapping = {f"EEG {i}": montage_labels[i] for i in range(n_channels)}
    raw.rename_channels(channel_mapping)
    try:
        theta_values = [-18, 18, -54, -39, 0, 39, 54, -90, -90, 90, 90, 90, -126, -141, 180, 141, 126, -162, 162]
        radius_values_cm = [51.1, 51.1, 51.1, 33.3, 25.6, 33.3, 51.1, 51.1, 25.6, 0, 25.6, 51.1, 51.1, 33.3, 25.6, 33.3, 51.1, 51.1, 51.1]
        radius_values_m = [r / 100 for r in radius_values_cm]
        ch_pos = {label: [radius * np.cos(np.deg2rad(theta)), radius * np.sin(np.deg2rad(theta)), 0.0] 
                  for label, theta, radius in zip(montage_labels, theta_values, radius_values_m)}
        montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='head')
        raw.set_montage(montage)
    except Exception:
        montage = mne.channels.make_standard_montage('standard_1020')
        raw.set_montage(montage)
    raw.filter(l_freq=0.5, h_freq=None, method='iir')
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=42, max_iter=500)
    ica.fit(raw)
    ica_sources = ica.get_sources(raw).get_data().T
    kurtosis_values = np.array([kurtosis(ica_sources[:, i]) for i in range(n_components)])
    variance_values = np.var(ica_sources, axis=0)
    peak_amplitude = np.max(np.abs(ica_sources), axis=0)
    fp2_weight = np.max(np.abs(ica_sources[total_samples // 2:total_samples // 2 + n_samples, 1])) / np.max(peak_amplitude)
    kurtosis_norm = (kurtosis_values - kurtosis_values.min()) / (kurtosis_values.max() - kurtosis_values.min() + 1e-10)
    variance_norm = (variance_values - variance_values.min()) / (variance_values.max() - variance_values.min() + 1e-10)
    peak_norm = (peak_amplitude - peak_amplitude.min()) / (peak_amplitude.max() - peak_amplitude.min() + 1e-10)
    combined_score = 0.45 * kurtosis_norm + 0.25 * variance_norm + 0.2 * peak_norm + 0.1 * fp2_weight
    top_artifact_indices = np.argsort(combined_score)[-10:]
    ica.exclude = top_artifact_indices.tolist()
    raw_clean = ica.apply(raw.copy())
    cleaned_data_raw = raw_clean.get_data()
    cleaned_data = cleaned_data_raw.T.reshape(n_segments, n_samples, n_channels)
    for ch in [6, 15, 16]:
        plt.figure(figsize=(10, 5))
        plt.plot(data[0, :, ch], label=f'Raw ch{ch}')
        plt.plot(cleaned_data[0, :, ch], label=f'Cleaned ch{ch}')
        plt.title(f"Raw vs. Cleaned Data for Channel {ch}")
        plt.legend()
        plt.savefig(f"ica_comparison_ch{ch}.png")
        plt.close()
    return cleaned_data

# Feature Extraction Functions
def normalize_data(data):
    scaler = StandardScaler()
    return scaler.fit_transform(data)

def calculate_engagement_index(data, fs=128):
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    alpha = calculate_psd(data, fs, freq_range=(8, 13))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    return np.mean((theta + alpha) / beta)

def calculate_psd(data, fs=128, freq_range=None):
    f, psd = signal.welch(data, fs=fs, nperseg=1024, axis=0)
    if freq_range:
        freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
        return np.mean(psd[freq_mask], axis=0)
    return psd.mean(axis=0)

def phase_amplitude_coupling(data, fs=128):
    theta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 4, 8)[0, :, 0]
    alpha = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 8, 13)[0, :, 0]
    beta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 13, 30)[0, :, 0]
    theta_phase = np.angle(hilbert(theta))
    alpha_amp = np.abs(hilbert(alpha))
    beta_amp = np.abs(hilbert(beta))
    pac_theta_alpha = np.mean(np.abs(np.mean(alpha_amp * np.exp(1j * theta_phase))))
    pac_theta_beta = np.mean(np.abs(np.mean(beta_amp * np.exp(1j * theta_phase))))
    pac_alpha_beta = np.mean(np.abs(np.mean(beta_amp * np.exp(1j * np.angle(hilbert(alpha))))))
    return np.array([pac_theta_alpha, pac_theta_beta, pac_alpha_beta])

def calculate_sprint(data, fs=128):
    f, t, Zxx = signal.stft(data, fs=fs, nperseg=1024, noverlap=512)
    power = np.abs(Zxx)**2
    return np.mean(power)

def wavelet_features(data):
    coeffs = pywt.wavedec(data, 'db4', level=5, axis=0)
    features = []
    for c in coeffs:
        features.extend([np.mean(np.abs(c)), np.std(c), np.max(np.abs(c))])
    return np.array(features)

def hjorth_parameters(data):
    diff1 = np.diff(data)
    diff2 = np.diff(diff1)
    activity = np.var(data)
    mobility = np.sqrt(np.var(diff1) / activity) if activity > 0 else 0
    complexity = np.sqrt(np.var(diff2) / np.var(diff1)) / mobility if mobility > 0 else 0
    return np.array([activity, mobility, complexity])

def microstate_analysis(data, fs=128, n_states=4):
    t0 = time.time()
    gfp = np.std(data, axis=1)
    peaks, _ = find_peaks(gfp, distance=fs//10)
    if len(peaks) < n_states:
        peak_data = data
    else:
        peak_data = data[peaks]
    kmeans = KMeans(n_clusters=n_states, random_state=42, n_init=10, max_iter=300)
    kmeans.fit(peak_data)
    templates = kmeans.cluster_centers_
    distances = np.zeros((data.shape[0], n_states))
    for i, template in enumerate(templates):
        template_norm = np.linalg.norm(template)
        data_norm = np.linalg.norm(data, axis=1)
        distances[:, i] = np.abs(np.sum(data * template, axis=1) / (data_norm * template_norm + 1e-10))
    labels = np.argmax(distances, axis=1)
    durations = np.zeros(n_states)
    occurrences = np.zeros(n_states)
    for state in range(n_states):
        state_indices = np.where(labels == state)[0]
        if len(state_indices) > 0:
            diff = np.diff(state_indices)
            breaks = np.where(diff > 1)[0]
            segment_lengths = np.diff(np.concatenate([[0], breaks + 1, [len(state_indices)]]))
            durations[state] = np.mean(segment_lengths) / fs * 1000 if len(segment_lengths) > 0 else 0
            occurrences[state] = len(breaks) + 1 if len(breaks) > 0 else 1 if len(state_indices) > 0 else 0
    coverage = np.bincount(labels, minlength=n_states).astype(float) / len(labels)
    transitions = np.zeros((n_states, n_states))
    for t in range(len(labels)-1):
        transitions[labels[t], labels[t+1]] += 1
    transition_sums = transitions.sum(axis=1, keepdims=True)
    transitions = transitions / (transition_sums + 1e-10)
    microstate_features = np.hstack([durations, occurrences, coverage, transitions.ravel()])
    if len(microstate_features) != 32:
        microstate_features = np.pad(microstate_features, (0, 32 - len(microstate_features)), mode='constant', constant_values=0)
    return microstate_features

def extract_features_single(args):
    sample, idx, temp_dir = args
    n_channels = 19
    normalized = normalize_data(sample)
    microstates = microstate_analysis(normalized)
    channel_features = []
    for ch in range(n_channels):
        channel_data = normalized[:, ch]
        engagement = calculate_engagement_index(channel_data)
        theta = calculate_psd(channel_data, freq_range=(4, 8))
        alpha = calculate_psd(channel_data, freq_range=(8, 13))
        beta = calculate_psd(channel_data, freq_range=(13, 30))
        pac = phase_amplitude_coupling(channel_data)
        sprint = calculate_sprint(channel_data)
        wavelet = wavelet_features(channel_data)
        skewness = skew(channel_data)
        kurt = kurtosis(channel_data)
        hjorth = hjorth_parameters(channel_data)
        ch_features = np.concatenate([[engagement], [theta], [alpha], [beta], pac, [sprint], wavelet, [skewness], [kurt], hjorth])
        if len(ch_features) != 31:
            raise ValueError(f"Channel {ch} features length {len(ch_features)} != 31")
        channel_features.append(ch_features)
    sample_features = np.concatenate([np.array(channel_features).flatten(), microstates])
    if len(sample_features) != 621:
        raise ValueError(f"Total features {len(sample_features)} != 621")
    temp_file = os.path.join(temp_dir, f"segment_{idx}.npy")
    if os.path.exists(temp_file):
        os.remove(temp_file)
    np.save(temp_file, sample_features)
    return temp_file, sample_features

def extract_all_features(data, n_channels=19):
    n_samples = data.shape[0]
    feature_matrix = []
    with tempfile.TemporaryDirectory() as temp_dir:
        for i in tqdm(range(n_samples), desc="Processing segments"):
            temp_file, features = extract_features_single((data[i], i, temp_dir))
            feature_matrix.append(features)
    print("Feature extraction completed.")
    return np.array(feature_matrix)

def select_features(X, y, feature_names):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    initial_model = xgb.XGBClassifier(random_state=42)
    initial_model.fit(X_train, y_train)
    explainer = shap.TreeExplainer(initial_model)
    initial_shap_values = explainer.shap_values(X_train)
    shap_importance = np.abs(initial_shap_values).mean(axis=0)
    n_features_to_select = int(X.shape[1] * 0.5)
    important_indices = np.argsort(shap_importance)[-n_features_to_select:]
    X_shap_selected = X[:, important_indices]
    f_values, p_values = f_classif(X_shap_selected, y)
    significant_features = p_values < 0.05
    X_anova_selected = X_shap_selected[:, significant_features]
    pca = PCA(n_components=0.95)
    X_final = pca.fit_transform(X_anova_selected)
    selected_feature_names = np.array(feature_names)[important_indices]
    selected_features = selected_feature_names[significant_features]
    pca_loadings = pca.components_
    pca_loadings_df = pd.DataFrame(pca_loadings, columns=selected_features, index=[f"PCA_{i}" for i in range(pca.n_components_)])
    return (X_final, initial_model, initial_shap_values, selected_features, important_indices,
            significant_features, pca, X_train, X_test, y_train, y_test, pca_loadings_df)

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = report['weighted avg']['f1-score']
    recall = report['weighted avg']['recall']
    print("\nConfusion Matrix:")
    print(cm)
    print(f"Sensitivity: {sensitivity:.4f}, Specificity: {specificity:.4f}, F1 Score: {f1:.4f}, Recall: {recall:.4f}")
    return cm, sensitivity, specificity, f1, recall

def main():
    print("Starting data loading and preprocessing...")
    X, y = load_all_data()
    print("Starting feature extraction...")
    start_time = time.time()
    X_features = extract_all_features(X)
    end_time = time.time()
    print(f"Feature extraction completed in {int((end_time - start_time) / 60)} minutes {int((end_time - start_time) % 60)} seconds")
    n_channels = 19
    feature_types = ['engagement', 'theta', 'alpha', 'beta',
                    'pac_theta_alpha', 'pac_theta_beta', 'pac_alpha_beta',
                    'sprint',
                    'wavelet_mean1', 'wavelet_std1', 'wavelet_max1',
                    'wavelet_mean2', 'wavelet_std2', 'wavelet_max2',
                    'wavelet_mean3', 'wavelet_std3', 'wavelet_max3',
                    'wavelet_mean4', 'wavelet_std4', 'wavelet_max4',
                    'wavelet_mean5', 'wavelet_std5', 'wavelet_max5',
                    'wavelet_mean6', 'wavelet_std6', 'wavelet_max6',
                    'skewness', 'kurtosis',
                    'hjorth_activity', 'hjorth_mobility', 'hjorth_complexity']
    n_states = 4
    microstate_features = ([f'ms_duration_{i}' for i in range(n_states)] +
                         [f'ms_occurrence_{i}' for i in range(n_states)] +
                         [f'ms_coverage_{i}' for i in range(n_states)] +
                         [f'ms_trans_{i}_{j}' for i in range(n_states) for j in range(n_states)])
    feature_names = ([f"{ft}_ch{ch}" for ch in range(n_channels) for ft in feature_types] +
                    microstate_features)
    print("Selecting features...")
    (X_final, initial_model, initial_shap_values, selected_features, shap_indices,
     anova_mask, pca, X_train, X_test, y_train, y_test, pca_loadings_df) = select_features(X_features, y, feature_names)
    print("Performing hyperparameter tuning for XGBoost...")
    xgb_param_dist = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0]
    }
    xgb_base = xgb.XGBClassifier(random_state=42)
    xgb_search = RandomizedSearchCV(xgb_base, param_distributions=xgb_param_dist, n_iter=20, cv=5,
                                    scoring='accuracy', random_state=42, n_jobs=-1)
    xgb_search.fit(X_final, y)
    xgb_model = xgb_search.best_estimator_
    print(f"Best XGBoost parameters: {xgb_search.best_params_}")
    print(f"Best XGBoost CV accuracy: {xgb_search.best_score_:.4f}")
    print("Performing hyperparameter tuning for CatBoost...")
    cat_param_dist = {
        'iterations': [100, 200, 300],
        'depth': [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1],
        'l2_leaf_reg': [1, 3, 5]
    }
    cat_base = CatBoostClassifier(random_state=42, verbose=0)
    cat_search = RandomizedSearchCV(cat_base, param_distributions=cat_param_dist, n_iter=20, cv=5,
                                    scoring='accuracy', random_state=42, n_jobs=-1)
    cat_search.fit(X_final, y)
    cat_model = cat_search.best_estimator_
    print(f"Best CatBoost parameters: {cat_search.best_params_}")
    print(f"Best CatBoost CV accuracy: {cat_search.best_score_:.4f}")
    print("Performing 10-fold cross-validation for individual models...")
    cv_scores_xgb = cross_val_score(xgb_model, X_final, y, cv=10, scoring='accuracy')
    cv_scores_cat = cross_val_score(cat_model, X_final, y, cv=10, scoring='accuracy')
    print(f"XGBoost Mean CV accuracy: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std() * 2:.4f})")
    print(f"CatBoost Mean CV accuracy: {cv_scores_cat.mean():.4f} (+/- {cv_scores_cat.std() * 2:.4f})")
    print("Training individual models...")
    xgb_model.fit(X_train, y_train)
    cat_model.fit(X_train, y_train)
    lgb_model = LGBMClassifier(random_state=42)
    lgb_model.fit(X_train, y_train)
    print("Evaluating individual models...")
    print("XGBoost:")
    evaluate_model(xgb_model, X_test, y_test)
    print("CatBoost:")
    evaluate_model(cat_model, X_test, y_test)
    print("LightGBM:")
    evaluate_model(lgb_model, X_test, y_test)
    print("Training stacked ensemble...")
    estimators = [('xgb', xgb_model), ('cat', cat_model), ('lgb', lgb_model)]
    stack_model = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(), cv=5)
    cv_scores_stack = cross_val_score(stack_model, X_final, y, cv=10, scoring='accuracy')
    print(f"Stacked Model Mean CV accuracy: {cv_scores_stack.mean():.4f} (+/- {cv_scores_stack.std() * 2:.4f})")
    stack_model.fit(X_train, y_train)
    print("Evaluating stacked model...")
    evaluate_model(stack_model, X_test, y_test)
    print("Computing SHAP values...")
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test)
    plt.figure()
    shap.summary_plot(shap_values, X_test, feature_names=[f"PCA_{i}" for i in range(X_test.shape[1])], show=False)
    plt.title("SHAP Summary Plot (PCA Features)")
    plt.savefig("shap_summary_pca.png")
    plt.close()
    X_features_df = pd.DataFrame(X_features, columns=feature_names)
    plt.figure()
    shap.summary_plot(initial_shap_values, X_features_df, plot_type="bar", max_display=20, show=False)
    plt.title("SHAP Bar Plot (Mean Impact on Model Prediction)")
    plt.savefig("shap_bar_original.png")
    plt.close()
    print("\nTop 10 selected features:")
    for i, feature in enumerate(selected_features[:10]):
        print(f"{i+1}. {feature}")
    print_memory_usage()

if __name__ == "__main__":
    main()

Starting data loading and preprocessing...
Creating RawArray with float64 data, n_channels=19, n_times=4272128
    Range : 0 ... 4272127 =      0.000 ... 33375.992 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 0.5 Hz

IIR filter parameters
---------------------
Butterworth highpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 0.50 Hz: -6.02 dB

Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Fitting ICA took 541.7s.
Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 10 ICA components
    Projecting back using 19 PCA components
Data loading and preprocessing completed.
Starting feature extraction...


Processing segments: 100%|█████████████████████████████████████████████████████████| 4172/4172 [25:25<00:00,  2.74it/s]


Feature extraction completed.
Feature extraction completed in 25 minutes 26 seconds
Selecting features...
Performing hyperparameter tuning for XGBoost...
Best XGBoost parameters: {'subsample': 0.6, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.6}
Best XGBoost CV accuracy: 0.6867
Performing hyperparameter tuning for CatBoost...
Best CatBoost parameters: {'learning_rate': 0.01, 'l2_leaf_reg': 1, 'iterations': 300, 'depth': 6}
Best CatBoost CV accuracy: 0.6865
Performing 10-fold cross-validation for individual models...
XGBoost Mean CV accuracy: 0.6862 (+/- 0.1373)
CatBoost Mean CV accuracy: 0.6807 (+/- 0.1470)
Training individual models...
[LightGBM] [Info] Number of positive: 1835, number of negative: 1502
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.047181 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 151923
[LightGBM] [Info] Number of data points in the trai

KeyboardInterrupt: 

In [ ]:
adhd_path1 = r"C:\Users\ADMIN\Downloads\Calcium signalling research\ADHD_part1\ADHD_part1"# Replace with actual path
    adhd_path2 = r"C:\Users\ADMIN\Downloads\Calcium signalling research\ADHD_part2\ADHD_part2"  # Replace with actual path
    control_path1 = r"C:\Users\ADMIN\Downloads\Calcium signalling research\Control_part1\Control_part1"  # Replace with actual path
    control_path2 = r"C:\Users\ADMIN\Downloads\Calcium signalling research\Control_part2" # Replace with actual path

In [ ]:
import numpy as np
from scipy.io import loadmat
import os
from scipy import signal
from scipy.stats import skew, kurtosis
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping
import shap
import pandas as pd
from scipy.signal import hilbert, find_peaks
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.ensemble import VotingClassifier
import mne
from tqdm import tqdm
import tempfile
import time
import psutil
import matplotlib.pyplot as plt

def print_memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024 / 1024:.2f} MB")

# Preprocessing Functions
def load_and_segment_file(file_path, window_size=512, stride=256, expected_channels=19):
    mat_data = loadmat(file_path)
    var_names = [k for k in mat_data.keys() if not k.startswith('__')]
    if not var_names:
        raise ValueError(f"No non-metadata variables found in {file_path}")
    possible_var_names = ['data', 'eeg', 'signal', 'EEG', 'Data'] + var_names
    data = None
    for var_name in possible_var_names:
        if var_name in mat_data:
            data = mat_data[var_name]
            break
    if data is None:
        raise ValueError(f"No valid EEG data variable found in {file_path}")
    if data.ndim == 1:
        data = data.reshape(1, -1, 1)
    elif data.ndim == 2:
        data = data[np.newaxis, :, :]
    elif data.ndim == 3:
        pass
    else:
        raise ValueError(f"Unexpected data dimensions {data.ndim}")
    n_segments, n_samples, n_channels = data.shape
    if n_channels != expected_channels:
        if n_channels < expected_channels:
            data = np.pad(data, ((0, 0), (0, 0), (0, expected_channels - n_channels)), mode='constant')
        else:
            data = data[:, :, :expected_channels]
    max_abs_value = np.max(np.abs(data))
    if max_abs_value > 1000:
        data = data / 1000.0
    if max_abs_value > 100:
        scaling_factor = 100 / max_abs_value
        data = data * scaling_factor
    all_segments = []
    for seg in range(n_segments):
        seg_data = data[seg]
        seg_samples = seg_data.shape[0]
        if seg_samples < window_size:
            padded_data = np.pad(seg_data, ((0, window_size - seg_samples), (0, 0)), mode='constant')
            all_segments.append(padded_data[np.newaxis, :, :])
            continue
        n_seg_segments = ((seg_samples - window_size + stride - 1) // stride) + 1
        segments = np.zeros((n_seg_segments, window_size, n_channels))
        for i in range(n_seg_segments):
            start = i * stride
            end = min(start + window_size, seg_samples)
            segment_length = end - start
            if segment_length == window_size:
                segments[i] = seg_data[start:end, :]
            else:
                segments[i] = np.pad(seg_data[start:end, :], ((0, window_size - segment_length), (0, 0)), mode='constant')
        all_segments.append(segments)
    if not all_segments:
        return None
    return np.concatenate(all_segments, axis=0)

def load_from_folder(folder_path, window_size=512, stride=256, expected_channels=19):
    if not os.path.exists(folder_path):
        return None
    mat_files = [f for f in os.listdir(folder_path) if f.endswith('.mat')]
    if not mat_files:
        return None
    all_segments = []
    for file in mat_files:
        file_path = os.path.join(folder_path, file)
        segments = load_and_segment_file(file_path, window_size, stride, expected_channels)
        if segments is not None:
            all_segments.append(segments)
    if not all_segments:
        return None
    return np.concatenate(all_segments, axis=0)

def load_all_data(window_size=512, stride=256, expected_channels=19):
    adhd_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1"
    adhd_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part2/ADHD_part2"
    control_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part1/Control_part1"
    control_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part2"
    adhd_segments1 = load_from_folder(adhd_path1, window_size, stride, expected_channels)
    adhd_segments2 = load_from_folder(adhd_path2, window_size, stride, expected_channels)
    if adhd_segments1 is None or adhd_segments2 is None:
        raise ValueError("Failed to load ADHD data")
    adhd_segments = np.concatenate([adhd_segments1, adhd_segments2], axis=0)
    control_segments1 = load_from_folder(control_path1, window_size, stride, expected_channels)
    control_segments2 = load_from_folder(control_path2, window_size, stride, expected_channels)
    if control_segments1 is None or control_segments2 is None:
        raise ValueError("Failed to load Control data")
    control_segments = np.concatenate([control_segments1, control_segments2], axis=0)
    y_adhd = np.ones(len(adhd_segments))
    y_control = np.zeros(len(control_segments))
    X = np.concatenate([adhd_segments, control_segments], axis=0)
    y = np.concatenate([y_adhd, y_control])
    X = notch_filter(X)
    X = clip_spikes(X)
    X = butterworth_filter(X)
    X = apply_ica_and_plot(X)
    print("Data loading and preprocessing completed.")
    return X, y

def notch_filter(data, fs=128, freqs=[50, 60], Q=30):
    filtered_data = data.copy()
    for freq in freqs:
        b, a = signal.iirnotch(freq / (fs / 2), Q)
        filtered_data = signal.filtfilt(b, a, filtered_data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def clip_spikes(data, threshold=500):
    clipped_data = data.copy()
    clipped_data[np.abs(clipped_data) > threshold] = np.sign(clipped_data[np.abs(clipped_data) > threshold]) * threshold
    return clipped_data

def butterworth_filter(data, fs=128, lowcut=1, highcut=45, order=4):
    nyquist = fs / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = signal.butter(order, [low, high], btype='band')
    filtered_data = signal.filtfilt(b, a, data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def apply_ica_and_plot(data, fs=128, n_components=19, segments_to_plot=3, channels_to_plot=19):
    n_segments, n_samples, n_channels = data.shape
    total_samples = n_segments * n_samples
    concatenated_data = data.reshape(total_samples, n_channels)
    ch_names = [f"EEG {i}" for i in range(n_channels)]
    montage_labels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
    info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types="eeg")
    raw = mne.io.RawArray(concatenated_data.T, info)
    channel_mapping = {f"EEG {i}": montage_labels[i] for i in range(n_channels)}
    raw.rename_channels(channel_mapping)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage)
    raw.filter(l_freq=0.5, h_freq=None, method='iir')
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=42, max_iter=500)
    ica.fit(raw)
    ica_sources = ica.get_sources(raw).get_data().T
    kurtosis_values = np.array([kurtosis(ica_sources[:, i]) for i in range(n_components)])
    variance_values = np.var(ica_sources, axis=0)
    peak_amplitude = np.max(np.abs(ica_sources), axis=0)
    kurtosis_norm = (kurtosis_values - kurtosis_values.min()) / (kurtosis_values.max() - kurtosis_values.min() + 1e-10)
    variance_norm = (variance_values - variance_values.min()) / (variance_values.max() - variance_values.min() + 1e-10)
    peak_norm = (peak_amplitude - peak_amplitude.min()) / (peak_amplitude.max() - peak_amplitude.min() + 1e-10)
    combined_score = 0.5 * kurtosis_norm + 0.3 * variance_norm + 0.2 * peak_norm
    top_artifact_indices = np.argsort(combined_score)[-10:]
    ica.exclude = top_artifact_indices.tolist()
    raw_clean = ica.apply(raw.copy())
    cleaned_data_raw = raw_clean.get_data()
    cleaned_data = cleaned_data_raw.T.reshape(n_segments, n_samples, n_channels)
    for ch in [6, 15, 16]:
        plt.figure(figsize=(10, 5))
        plt.plot(data[0, :, ch], label=f'Raw ch{ch}')
        plt.plot(cleaned_data[0, :, ch], label=f'Cleaned ch{ch}')
        plt.title(f"Raw vs. Cleaned Data for Channel {ch}")
        plt.legend()
        plt.savefig(f"ica_comparison_ch{ch}.png")
        plt.close()
    return cleaned_data

# Feature Extraction Functions
def normalize_data(data):
    scaler = StandardScaler()
    return scaler.fit_transform(data)

def calculate_engagement_index(data, fs=128):
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    alpha = calculate_psd(data, fs, freq_range=(8, 13))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    return np.mean((theta + alpha) / beta)

def calculate_psd(data, fs=128, freq_range=None):
    f, psd = signal.welch(data, fs=fs, nperseg=512, axis=0)
    if freq_range:
        freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
        return np.mean(psd[freq_mask], axis=0)
    return psd.mean(axis=0)

def phase_amplitude_coupling(data, fs=128):
    theta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 4, 8)[0, :, 0]
    alpha = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 8, 13)[0, :, 0]
    beta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 13, 30)[0, :, 0]
    theta_phase = np.angle(hilbert(theta))
    alpha_amp = np.abs(hilbert(alpha))
    beta_amp = np.abs(hilbert(beta))
    pac_theta_alpha = np.mean(np.abs(np.mean(alpha_amp * np.exp(1j * theta_phase))))
    pac_theta_beta = np.mean(np.abs(np.mean(beta_amp * np.exp(1j * theta_phase))))
    pac_alpha_beta = np.mean(np.abs(np.mean(beta_amp * np.exp(1j * np.angle(hilbert(alpha))))))
    return np.array([pac_theta_alpha, pac_theta_beta, pac_alpha_beta])

def calculate_sprint(data, fs=128):
    f, t, Zxx = signal.stft(data, fs=fs, nperseg=512, noverlap=256)
    power = np.abs(Zxx)**2
    return np.mean(power)

def wavelet_features(data):
    scales = np.arange(1, 7)  # 6 scales = 18 features
    coeffs, _ = pywt.cwt(data, scales, 'mexh', axis=0)
    features = []
    for c in coeffs:
        features.extend([np.mean(np.abs(c)), np.std(c), np.max(np.abs(c))])
    return np.array(features)

def hjorth_parameters(data):
    diff1 = np.diff(data)
    diff2 = np.diff(diff1)
    activity = np.var(data)
    mobility = np.sqrt(np.var(diff1) / activity) if activity > 0 else 0
    complexity = np.sqrt(np.var(diff2) / np.var(diff1)) / mobility if mobility > 0 else 0
    return np.array([activity, mobility, complexity])

def microstate_analysis(data, fs=128, n_states=4):
    gfp = np.std(data, axis=1)
    peaks, _ = find_peaks(gfp, distance=fs//10)
    if len(peaks) < n_states:
        peak_data = data
    else:
        peak_data = data[peaks]
    kmeans = KMeans(n_clusters=n_states, random_state=42, n_init=10, max_iter=300)
    kmeans.fit(peak_data)
    templates = kmeans.cluster_centers_
    distances = np.zeros((data.shape[0], n_states))
    for i, template in enumerate(templates):
        template_norm = np.linalg.norm(template)
        data_norm = np.linalg.norm(data, axis=1)
        distances[:, i] = np.abs(np.sum(data * template, axis=1) / (data_norm * template_norm + 1e-10))
    labels = np.argmax(distances, axis=1)
    durations = np.zeros(n_states)
    occurrences = np.zeros(n_states)
    for state in range(n_states):
        state_indices = np.where(labels == state)[0]
        if len(state_indices) > 0:
            diff = np.diff(state_indices)
            breaks = np.where(diff > 1)[0]
            segment_lengths = np.diff(np.concatenate([[0], breaks + 1, [len(state_indices)]]))
            durations[state] = np.mean(segment_lengths) / fs * 1000 if len(segment_lengths) > 0 else 0
            occurrences[state] = len(breaks) + 1 if len(breaks) > 0 else 1 if len(state_indices) > 0 else 0
    coverage = np.bincount(labels, minlength=n_states).astype(float) / len(labels)
    transitions = np.zeros((n_states, n_states))
    for t in range(len(labels)-1):
        transitions[labels[t], labels[t+1]] += 1
    transition_sums = transitions.sum(axis=1, keepdims=True)
    transitions = transitions / (transition_sums + 1e-10)
    microstate_features = np.hstack([durations, occurrences, coverage, transitions.ravel()])
    if len(microstate_features) != 32:
        microstate_features = np.pad(microstate_features, (0, 32 - len(microstate_features)), mode='constant', constant_values=0)
    return microstate_features

def extract_features_single(args):
    sample, idx, temp_dir = args
    n_channels = 19
    normalized = normalize_data(sample)
    microstates = microstate_analysis(normalized)
    channel_features = []
    for ch in range(n_channels):
        channel_data = normalized[:, ch]
        engagement = calculate_engagement_index(channel_data)
        theta = calculate_psd(channel_data, freq_range=(4, 8))
        alpha = calculate_psd(channel_data, freq_range=(8, 13))
        beta = calculate_psd(channel_data, freq_range=(13, 30))
        pac = phase_amplitude_coupling(channel_data)
        sprint = calculate_sprint(channel_data)
        wavelet = wavelet_features(channel_data)
        skewness = skew(channel_data)
        kurt = kurtosis(channel_data)
        hjorth = hjorth_parameters(channel_data)
        ch_features = np.concatenate([[engagement], [theta], [alpha], [beta], pac, [sprint], wavelet, [skewness], [kurt], hjorth])
        if ch_features.shape[0] != 31:
            raise ValueError(f"Channel {ch} features length {ch_features.shape[0]} != 31")
        channel_features.append(ch_features)
    sample_features = np.concatenate([np.array(channel_features).flatten(), microstates])
    temp_file = os.path.join(temp_dir, f"segment_{idx}.npy")
    if os.path.exists(temp_file):
        os.remove(temp_file)
    np.save(temp_file, sample_features)
    return temp_file, sample_features

def extract_all_features(data, n_channels=19):
    n_samples = data.shape[0]
    feature_matrix = []
    with tempfile.TemporaryDirectory() as temp_dir:
        for i in tqdm(range(n_samples), desc="Processing segments"):
            temp_file, features = extract_features_single((data[i], i, temp_dir))
            feature_matrix.append(features)
    print("Feature extraction completed.")
    return np.array(feature_matrix)

def select_features(X, y, feature_names):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    initial_model = LGBMClassifier(random_state=42)
    initial_model.fit(X_train, y_train)
    explainer = shap.TreeExplainer(initial_model)
    shap_values = explainer.shap_values(X_train)
    shap_importance = np.abs(shap_values[1]).mean(axis=0) if isinstance(shap_values, list) else np.abs(shap_values).mean(axis=0)
    n_features_to_select = min(100, int(X.shape[1] * 0.8))
    important_indices = np.argsort(shap_importance)[-n_features_to_select:]
    if max(important_indices) >= X.shape[1]:
        raise ValueError(f"SHAP selected index {max(important_indices)} exceeds feature count {X.shape[1]}")
    X_shap_selected = X[:, important_indices]
    f_values, p_values = f_classif(X_shap_selected, y)
    significant_features = p_values < 0.01
    X_final = X_shap_selected[:, significant_features]
    if len(feature_names) != X.shape[1]:
        print(f"Warning: Feature names length ({len(feature_names)}) != X.shape[1] ({X.shape[1]}). Adjusting names.")
        feature_names = [f"feature_{i}" for i in range(X.shape[1])]
    selected_feature_names = np.array(feature_names)[important_indices]
    selected_features = selected_feature_names[significant_features]
    print(f"Selected {X_final.shape[1]} features after SHAP and ANOVA.")
    return X_final, initial_model, selected_features, X_train, X_test, y_train, y_test

def evaluate_model(model, X_train, X_test, y_train, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = report['weighted avg']['f1-score']
    print(f"\n{name}:")
    print(f"Accuracy: {accuracy:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print(f"Sensitivity: {sensitivity:.4f}, Specificity: {specificity:.4f}, F1 Score: {f1:.4f}")
    return accuracy, sensitivity, specificity, f1

def main():
    print("Starting data loading and preprocessing...")
    X, y = load_all_data(window_size=512, stride=256)
    print("Starting feature extraction...")
    start_time = time.time()
    X_features = extract_all_features(X)
    end_time = time.time()
    print(f"Feature extraction completed in {int((end_time - start_time) / 60)} minutes {int((end_time - start_time) % 60)} seconds")
    print(f"X_features shape: {X_features.shape}")
    
    n_channels = 19
    feature_types = [
        'engagement',
        'theta',
        'alpha',
        'beta',
        'pac_theta_alpha',
        'pac_theta_beta',
        'pac_alpha_beta',
        'sprint',
        'wavelet_mean1', 'wavelet_std1', 'wavelet_max1',
        'wavelet_mean2', 'wavelet_std2', 'wavelet_max2',
        'wavelet_mean3', 'wavelet_std3', 'wavelet_max3',
        'wavelet_mean4', 'wavelet_std4', 'wavelet_max4',
        'wavelet_mean5', 'wavelet_std5', 'wavelet_max5',
        'wavelet_mean6', 'wavelet_std6', 'wavelet_max6',
        'skewness',
        'kurtosis',
        'hjorth_activity',
        'hjorth_mobility',
        'hjorth_complexity'
    ]
    n_states = 4
    microstate_features = (
        [f'ms_duration_{i}' for i in range(n_states)] +
        [f'ms_occurrence_{i}' for i in range(n_states)] +
        [f'ms_coverage_{i}' for i in range(n_states)] +
        [f'ms_trans_{i}_{j}' for i in range(n_states) for j in range(n_states)]
    )
    feature_names = ([f"{ft}_ch{ch}" for ch in range(n_channels) for ft in feature_types] + microstate_features)
    print(f"Feature names length: {len(feature_names)}")
    if len(feature_names) != X_features.shape[1]:
        print(f"Adjusting feature names to match X_features.shape[1] ({X_features.shape[1]})")
        feature_names = [f"feature_{i}" for i in range(X_features.shape[1])]
    
    print("Selecting features...")
    X_final, initial_model, selected_features, X_train, X_test, y_train, y_test = select_features(X_features, y, feature_names)
    
    class_weight = {0: 1.3, 1: 1.0}
    
    print("Tuning LightGBM...")
    lgb_param_dist = {
        'n_estimators': [500, 700, 1000],
        'max_depth': [5, 10, 15, -1],
        'learning_rate': [0.005, 0.01, 0.05],
        'num_leaves': [31, 50, 70, 100],
        'min_child_samples': [10, 20, 30],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0],
        'min_gain_to_split': [0.0, 0.01]
    }
    lgb_base = LGBMClassifier(random_state=42, class_weight=class_weight)
    lgb_search = RandomizedSearchCV(lgb_base, lgb_param_dist, n_iter=30, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
    lgb_search.fit(X_final, y, eval_set=[(X_test, y_test)], callbacks=[early_stopping(stopping_rounds=10)])
    lgb_model = lgb_search.best_estimator_
    print(f"Best LightGBM parameters: {lgb_search.best_params_}")
    print(f"Best LightGBM CV accuracy: {lgb_search.best_score_:.4f}")
    
    print("Tuning XGBoost...")
    xgb_param_dist = {
        'n_estimators': [300, 500, 700],
        'max_depth': [7, 10, 15],
        'learning_rate': [0.005, 0.01, 0.05],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0],
        'gamma': [0, 1, 2],
        'reg_lambda': [1, 5, 10],
        'min_child_weight': [1, 5, 10],
        'scale_pos_weight': [0.79]
    }
    xgb_base = xgb.XGBClassifier(random_state=42)
    xgb_search = RandomizedSearchCV(xgb_base, xgb_param_dist, n_iter=30, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
    xgb_search.fit(X_final, y)
    xgb_model = xgb_search.best_estimator_
    print(f"Best XGBoost parameters: {xgb_search.best_params_}")
    print(f"Best XGBoost CV accuracy: {xgb_search.best_score_:.4f}")
    
    print("Tuning CatBoost...")
    cat_param_dist = {
        'iterations': [700, 1000, 1200],
        'depth': [6, 8, 10, 12],
        'learning_rate': [0.001, 0.005, 0.01],
        'l2_leaf_reg': [3, 5, 7],
        'border_count': [32, 64, 128],
        'class_weights': [{0: 1.3, 1: 1.0}]
    }
    cat_base = CatBoostClassifier(random_state=42, verbose=0)
    cat_search = RandomizedSearchCV(cat_base, cat_param_dist, n_iter=30, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
    cat_search.fit(X_final, y)
    cat_model = cat_search.best_estimator_
    print(f"Best CatBoost parameters: {cat_search.best_params_}")
    print(f"Best CatBoost CV accuracy: {cat_search.best_score_:.4f}")
    
    print("Training Voting Ensemble...")
    voting_model = VotingClassifier(
        estimators=[('lgb', lgb_model), ('xgb', xgb_model), ('cat', cat_model)],
        voting='soft'
    )
    
    print("Performing 10-fold cross-validation...")
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    cv_scores_lgb = cross_val_score(lgb_model, X_final, y, cv=skf, scoring='accuracy')
    cv_scores_xgb = cross_val_score(xgb_model, X_final, y, cv=skf, scoring='accuracy')
    cv_scores_cat = cross_val_score(cat_model, X_final, y, cv=skf, scoring='accuracy')
    cv_scores_voting = cross_val_score(voting_model, X_final, y, cv=skf, scoring='accuracy')
    print(f"LightGBM Mean CV accuracy: {cv_scores_lgb.mean():.4f} (+/- {cv_scores_lgb.std() * 2:.4f})")
    print(f"XGBoost Mean CV accuracy: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std() * 2:.4f})")
    print(f"CatBoost Mean CV accuracy: {cv_scores_cat.mean():.4f} (+/- {cv_scores_cat.std() * 2:.4f})")
    print(f"Voting Ensemble Mean CV accuracy: {cv_scores_voting.mean():.4f} (+/- {cv_scores_voting.std() * 2:.4f})")
    
    print("Evaluating models on test set...")
    evaluate_model(lgb_model, X_train, X_test, y_train, y_test, "LightGBM")
    evaluate_model(xgb_model, X_train, X_test, y_train, y_test, "XGBoost")
    evaluate_model(cat_model, X_train, X_test, y_train, y_test, "CatBoost")
    evaluate_model(voting_model, X_train, X_test, y_train, y_test, "Voting Ensemble")
    
    print("Computing SHAP values for LightGBM...")
    explainer = shap.TreeExplainer(lgb_model)
    shap_values = explainer.shap_values(X_test)
    plt.figure()
    shap.summary_plot(shap_values[1] if isinstance(shap_values, list) else shap_values, X_test, feature_names=selected_features, show=False)
    plt.title("SHAP Summary Plot (LightGBM)")
    plt.savefig("shap_summary_lgb.png")
    plt.close()
    
    top_feature_idx = np.argmax(np.abs(shap_values[1] if isinstance(shap_values, list) else shap_values).mean(0))
    plt.figure()
    shap.dependence_plot(top_feature_idx, shap_values[1] if isinstance(shap_values, list) else shap_values, X_test, feature_names=selected_features, show=False)
    plt.title(f"SHAP Dependence Plot for {selected_features[top_feature_idx]}")
    plt.savefig("shap_dependence_top_feature.png")
    plt.close()
    
    print("\nTop 10 selected features:")
    for i, feature in enumerate(selected_features[:10]):
        print(f"{i+1}. {feature}")
    print_memory_usage()

if __name__ == "__main__":
    main()

Starting data loading and preprocessing...
Creating RawArray with float64 data, n_channels=19, n_times=4300288
    Range : 0 ... 4300287 =      0.000 ... 33595.992 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 0.5 Hz

IIR filter parameters
---------------------
Butterworth highpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 0.50 Hz: -6.02 dB

Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Fitting ICA took 175.9s.
Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 10 ICA components
    Projecting back using 19 PCA components
Data loading and preprocessing completed.
Starting feature extraction...


Processing segments: 100%|█████████████████████████████████████████████████████████| 8399/8399 [24:35<00:00,  5.69it/s]


Feature extraction completed.
Feature extraction completed in 24 minutes 37 seconds
X_features shape: (8399, 621)
Feature names length: 617
Adjusting feature names to match X_features.shape[1] (621)
Selecting features...
[LightGBM] [Info] Number of positive: 3745, number of negative: 2974
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035670 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 151660
[LightGBM] [Info] Number of data points in the train set: 6719, number of used features: 598
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.557375 -> initscore=0.230514
[LightGBM] [Info] Start training from score 0.230514


C:\Users\ADMIN\AppData\Roaming\Python\Python312\site-packages\shap\explainers\_tree.py:448: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn('LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray')


Selected 89 features after SHAP and ANOVA.
Tuning LightGBM...
[LightGBM] [Warning] min_gain_to_split is set=0.01, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.01
[LightGBM] [Warning] min_gain_to_split is set=0.01, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.01
[LightGBM] [Info] Number of positive: 4682, number of negative: 3717
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006759 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 22513
[LightGBM] [Info] Number of data points in the train set: 8399, number of used features: 89
[LightGBM] [Warning] min_gain_to_split is set=0.01, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.01
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.492112 -> initscore=-0.031556
[LightGBM] [Info] Start training from score -0.031556
Training until validation scores don't improve for 10 rounds
Early s

In [ ]:
import numpy as np
from scipy.io import loadmat
import os
from scipy import signal
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif, RFE
import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping
import shap
import pandas as pd
from scipy.signal import hilbert, find_peaks
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import mne
from tqdm import tqdm
import tempfile
import time
import psutil
import matplotlib.pyplot as plt
from hurst import compute_Hc
from scipy.stats import kurtosis  # Added this import

def print_memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024 / 1024:.2f} MB")

# Preprocessing Functions (unchanged)
def load_and_segment_file(file_path, window_size=512, stride=256, expected_channels=19):
    mat_data = loadmat(file_path)
    var_names = [k for k in mat_data.keys() if not k.startswith('__')]
    if not var_names:
        raise ValueError(f"No non-metadata variables found in {file_path}")
    possible_var_names = ['data', 'eeg', 'signal', 'EEG', 'Data'] + var_names
    data = None
    for var_name in possible_var_names:
        if var_name in mat_data:
            data = mat_data[var_name]
            break
    if data is None:
        raise ValueError(f"No valid EEG data variable found in {file_path}")
    if data.ndim == 1:
        data = data.reshape(1, -1, 1)
    elif data.ndim == 2:
        data = data[np.newaxis, :, :]
    elif data.ndim == 3:
        pass
    else:
        raise ValueError(f"Unexpected data dimensions {data.ndim}")
    n_segments, n_samples, n_channels = data.shape
    if n_channels != expected_channels:
        if n_channels < expected_channels:
            data = np.pad(data, ((0, 0), (0, 0), (0, expected_channels - n_channels)), mode='constant')
        else:
            data = data[:, :, :expected_channels]
    max_abs_value = np.max(np.abs(data))
    if max_abs_value > 1000:
        data = data / 1000.0
    if max_abs_value > 100:
        scaling_factor = 100 / max_abs_value
        data = data * scaling_factor
    all_segments = []
    for seg in range(n_segments):
        seg_data = data[seg]
        seg_samples = seg_data.shape[0]
        if seg_samples < window_size:
            padded_data = np.pad(seg_data, ((0, window_size - seg_samples), (0, 0)), mode='constant')
            all_segments.append(padded_data[np.newaxis, :, :])
            continue
        n_seg_segments = ((seg_samples - window_size + stride - 1) // stride) + 1
        segments = np.zeros((n_seg_segments, window_size, n_channels))
        for i in range(n_seg_segments):
            start = i * stride
            end = min(start + window_size, seg_samples)
            segment_length = end - start
            if segment_length == window_size:
                segments[i] = seg_data[start:end, :]
            else:
                segments[i] = np.pad(seg_data[start:end, :], ((0, window_size - segment_length), (0, 0)), mode='constant')
        all_segments.append(segments)
    if not all_segments:
        return None
    return np.concatenate(all_segments, axis=0)

def load_from_folder(folder_path, window_size=512, stride=256, expected_channels=19):
    if not os.path.exists(folder_path):
        return None
    mat_files = [f for f in os.listdir(folder_path) if f.endswith('.mat')]
    if not mat_files:
        return None
    all_segments = []
    for file in mat_files:
        file_path = os.path.join(folder_path, file)
        segments = load_and_segment_file(file_path, window_size, stride, expected_channels)
        if segments is not None:
            all_segments.append(segments)
    if not all_segments:
        return None
    return np.concatenate(all_segments, axis=0)

def load_all_data(window_size=512, stride=256, expected_channels=19):
    adhd_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1"
    adhd_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part2/ADHD_part2"
    control_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part1/Control_part1"
    control_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part2"
    adhd_segments1 = load_from_folder(adhd_path1, window_size, stride, expected_channels)
    adhd_segments2 = load_from_folder(adhd_path2, window_size, stride, expected_channels)
    if adhd_segments1 is None or adhd_segments2 is None:
        raise ValueError("Failed to load ADHD data")
    adhd_segments = np.concatenate([adhd_segments1, adhd_segments2], axis=0)
    control_segments1 = load_from_folder(control_path1, window_size, stride, expected_channels)
    control_segments2 = load_from_folder(control_path2, window_size, stride, expected_channels)
    if control_segments1 is None or control_segments2 is None:
        raise ValueError("Failed to load Control data")
    control_segments = np.concatenate([control_segments1, control_segments2], axis=0)
    y_adhd = np.ones(len(adhd_segments))
    y_control = np.zeros(len(control_segments))
    X = np.concatenate([adhd_segments, control_segments], axis=0)
    y = np.concatenate([y_adhd, y_control])
    X = notch_filter(X)
    X = clip_spikes(X)
    X = butterworth_filter(X)
    X = apply_ica_and_plot(X)
    print("Data loading and preprocessing completed.")
    return X, y

def notch_filter(data, fs=128, freqs=[50, 60], Q=30):
    filtered_data = data.copy()
    for freq in freqs:
        b, a = signal.iirnotch(freq / (fs / 2), Q)
        filtered_data = signal.filtfilt(b, a, filtered_data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def clip_spikes(data, threshold=500):
    clipped_data = data.copy()
    clipped_data[np.abs(clipped_data) > threshold] = np.sign(clipped_data[np.abs(clipped_data) > threshold]) * threshold
    return clipped_data

def butterworth_filter(data, fs=128, lowcut=1, highcut=45, order=4):
    nyquist = fs / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = signal.butter(order, [low, high], btype='band')
    filtered_data = signal.filtfilt(b, a, data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def apply_ica_and_plot(data, fs=128, n_components=19, segments_to_plot=3, channels_to_plot=19):
    n_segments, n_samples, n_channels = data.shape
    total_samples = n_segments * n_samples
    concatenated_data = data.reshape(total_samples, n_channels)
    ch_names = [f"EEG {i}" for i in range(n_channels)]
    montage_labels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
    info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types="eeg")
    raw = mne.io.RawArray(concatenated_data.T, info)
    channel_mapping = {f"EEG {i}": montage_labels[i] for i in range(n_channels)}
    raw.rename_channels(channel_mapping)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage)
    raw.filter(l_freq=0.5, h_freq=None, method='iir')
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=42, max_iter=500)
    ica.fit(raw)
    ica_sources = ica.get_sources(raw).get_data().T
    kurtosis_values = np.array([kurtosis(ica_sources[:, i]) for i in range(n_components)])
    variance_values = np.var(ica_sources, axis=0)
    peak_amplitude = np.max(np.abs(ica_sources), axis=0)
    kurtosis_norm = (kurtosis_values - kurtosis_values.min()) / (kurtosis_values.max() - kurtosis_values.min() + 1e-10)
    variance_norm = (variance_values - variance_values.min()) / (variance_values.max() - variance_values.min() + 1e-10)
    peak_norm = (peak_amplitude - peak_amplitude.min()) / (peak_amplitude.max() - peak_amplitude.min() + 1e-10)
    combined_score = 0.5 * kurtosis_norm + 0.3 * variance_norm + 0.2 * peak_norm
    top_artifact_indices = np.argsort(combined_score)[-10:]
    ica.exclude = top_artifact_indices.tolist()
    raw_clean = ica.apply(raw.copy())
    cleaned_data_raw = raw_clean.get_data()
    cleaned_data = cleaned_data_raw.T.reshape(n_segments, n_samples, n_channels)
    for ch in [6, 15, 16]:
        plt.figure(figsize=(10, 5))
        plt.plot(data[0, :, ch], label=f'Raw ch{ch}')
        plt.plot(cleaned_data[0, :, ch], label=f'Cleaned ch{ch}')
        plt.title(f"Raw vs. Cleaned Data for Channel {ch}")
        plt.legend()
        plt.savefig(f"ica_comparison_ch{ch}.png")
        plt.close()
    return cleaned_data

# Feature Extraction Functions
def normalize_data(data):
    scaler = StandardScaler()
    return scaler.fit_transform(data)

def calculate_engagement_index(data, fs=128):
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    alpha = calculate_psd(data, fs, freq_range=(8, 13))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    return np.mean((theta + alpha) / beta)

def calculate_psd(data, fs=128, freq_range=None):
    f, psd = signal.welch(data, fs=fs, nperseg=512, axis=0)
    if freq_range:
        freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
        return np.mean(psd[freq_mask], axis=0)
    return psd.mean(axis=0)

def phase_amplitude_coupling(data, fs=128):
    theta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 4, 8)[0, :, 0]
    alpha = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 8, 13)[0, :, 0]
    beta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 13, 30)[0, :, 0]
    theta_phase = np.angle(hilbert(theta))
    alpha_amp = np.abs(hilbert(alpha))
    beta_amp = np.abs(hilbert(beta))
    pac_theta_alpha = np.mean(np.abs(np.mean(alpha_amp * np.exp(1j * theta_phase))))
    pac_theta_beta = np.mean(np.abs(np.mean(beta_amp * np.exp(1j * theta_phase))))
    pac_alpha_beta = np.mean(np.abs(np.mean(beta_amp * np.exp(1j * np.angle(hilbert(alpha))))))
    return np.array([pac_theta_alpha, pac_theta_beta, pac_alpha_beta])

def calculate_sprint(data, fs=128):
    f, t, Zxx = signal.stft(data, fs=fs, nperseg=512, noverlap=256)
    power = np.abs(Zxx)**2
    return np.mean(power)

def wavelet_features(data):
    scales = np.arange(1, 7)  # 6 scales = 18 features
    coeffs, _ = pywt.cwt(data, scales, 'mexh', axis=0)
    features = []
    for c in coeffs:
        features.extend([np.mean(np.abs(c)), np.std(c), np.max(np.abs(c))])
    return np.array(features)

def hjorth_parameters(data):
    diff1 = np.diff(data)
    diff2 = np.diff(diff1)
    activity = np.var(data)
    mobility = np.sqrt(np.var(diff1) / activity) if activity > 0 else 0
    complexity = np.sqrt(np.var(diff2) / np.var(diff1)) / mobility if mobility > 0 else 0
    return np.array([activity, mobility, complexity])

def line_length(data):
    return np.sum(np.abs(np.diff(data)))

def hurst_exponent(data):
    H, _, _ = compute_Hc(data, kind='change', simplified=True)
    return H

def zero_crossing_rate(data):
    return len(np.where(np.diff(np.sign(data)))[0]) / len(data)

def channel_correlation(data):
    n_channels = data.shape[1]
    corr_features = []
    for i in range(n_channels):
        for j in range(i + 1, n_channels):
            corr = np.corrcoef(data[:, i], data[:, j])[0, 1]
            corr_features.append(corr)
    return np.array(corr_features)

def microstate_analysis(data, fs=128, n_states=4):
    gfp = np.std(data, axis=1)
    peaks, _ = find_peaks(gfp, distance=fs//10)
    if len(peaks) < n_states:
        peak_data = data
    else:
        peak_data = data[peaks]
    kmeans = KMeans(n_clusters=n_states, random_state=42, n_init=10, max_iter=300)
    kmeans.fit(peak_data)
    templates = kmeans.cluster_centers_
    distances = np.zeros((data.shape[0], n_states))
    for i, template in enumerate(templates):
        template_norm = np.linalg.norm(template)
        data_norm = np.linalg.norm(data, axis=1)
        distances[:, i] = np.abs(np.sum(data * template, axis=1) / (data_norm * template_norm + 1e-10))
    labels = np.argmax(distances, axis=1)
    durations = np.zeros(n_states)
    occurrences = np.zeros(n_states)
    gfp_peaks = gfp[peaks] if len(peaks) > 0 else gfp
    gfp_mean = np.mean(gfp_peaks)
    gfp_std = np.std(gfp_peaks)
    for state in range(n_states):
        state_indices = np.where(labels == state)[0]
        if len(state_indices) > 0:
            diff = np.diff(state_indices)
            breaks = np.where(diff > 1)[0]
            segment_lengths = np.diff(np.concatenate([[0], breaks + 1, [len(state_indices)]]))
            durations[state] = np.mean(segment_lengths) / fs * 1000 if len(segment_lengths) > 0 else 0
            occurrences[state] = len(breaks) + 1 if len(breaks) > 0 else 1 if len(state_indices) > 0 else 0
    coverage = np.bincount(labels, minlength=n_states).astype(float) / len(labels)
    transitions = np.zeros((n_states, n_states))
    for t in range(len(labels)-1):
        transitions[labels[t], labels[t+1]] += 1
    transition_sums = transitions.sum(axis=1, keepdims=True)
    transitions = transitions / (transition_sums + 1e-10)
    microstate_features = np.hstack([durations, occurrences, coverage, transitions.ravel(), [gfp_mean, gfp_std]])
    return microstate_features

def extract_features_single(args):
    sample, idx, temp_dir = args
    n_channels = sample.shape[1]  # Dynamically get from data
    normalized = normalize_data(sample)
    microstates = microstate_analysis(normalized)
    correlations = channel_correlation(normalized)
    channel_features = []
    for ch in range(n_channels):
        channel_data = normalized[:, ch]
        engagement = calculate_engagement_index(channel_data)
        theta = calculate_psd(channel_data, freq_range=(4, 8))
        alpha = calculate_psd(channel_data, freq_range=(8, 13))
        beta = calculate_psd(channel_data, freq_range=(13, 30))
        pac = phase_amplitude_coupling(channel_data)
        sprint = calculate_sprint(channel_data)
        wavelet = wavelet_features(channel_data)
        ll = line_length(channel_data)
        he = hurst_exponent(channel_data)
        zcr = zero_crossing_rate(channel_data)
        hjorth = hjorth_parameters(channel_data)
        ch_features = np.concatenate([[engagement], [theta], [alpha], [beta], pac, [sprint], wavelet, [ll], [he], [zcr], hjorth])
        channel_features.append(ch_features)
    per_channel_features = np.array(channel_features)
    sample_features = np.concatenate([per_channel_features.flatten(), microstates, correlations])
    print(f"Segment {idx}: Per-channel features shape: {per_channel_features.shape}, Total features: {sample_features.shape[0]}")
    temp_file = os.path.join(temp_dir, f"segment_{idx}.npy")
    if os.path.exists(temp_file):
        os.remove(temp_file)
    np.save(temp_file, sample_features)
    return temp_file, sample_features

def extract_all_features(data):
    n_samples = data.shape[0]
    feature_matrix = []
    with tempfile.TemporaryDirectory() as temp_dir:
        for i in tqdm(range(n_samples), desc="Processing segments"):
            temp_file, features = extract_features_single((data[i], i, temp_dir))
            feature_matrix.append(features)
    feature_matrix = np.array(feature_matrix)
    print("Feature extraction completed.")
    print(f"Feature matrix shape: {feature_matrix.shape}")
    return feature_matrix

def select_features(X, y, feature_names):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    initial_model = LGBMClassifier(random_state=42, n_estimators=100)
    initial_model.fit(X_train, y_train)
    explainer = shap.TreeExplainer(initial_model)
    shap_values = explainer.shap_values(X_train)
    shap_importance = np.abs(shap_values[1]).mean(axis=0) if isinstance(shap_values, list) else np.abs(shap_values).mean(axis=0)
    top_n = min(300, X.shape[1])
    shap_indices = np.argsort(shap_importance)[-top_n:]
    X_shap = X[:, shap_indices]
    shap_feature_names = np.array(feature_names)[shap_indices]
    rfe_model = LGBMClassifier(random_state=42, n_estimators=100)
    rfe = RFE(estimator=rfe_model, n_features_to_select=150, step=10)
    rfe.fit(X_shap, y)
    X_final = rfe.transform(X_shap)
    X_train_final = rfe.transform(X_train[:, shap_indices])
    X_test_final = rfe.transform(X_test[:, shap_indices])
    selected_features = shap_feature_names[rfe.support_]
    print(f"Selected {X_final.shape[1]} features after SHAP + RFE.")
    print(f"X_final shape: {X_final.shape}, X_train_final shape: {X_train_final.shape}, X_test_final shape: {X_test_final.shape}")
    return X_final, initial_model, selected_features, X_train_final, X_test_final, y_train, y_test

def evaluate_model(model, X_train, X_test, y_train, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = report['weighted avg']['f1-score']
    print(f"\n{name}:")
    print(f"Accuracy: {accuracy:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print(f"Sensitivity: {sensitivity:.4f}, Specificity: {specificity:.4f}, F1 Score: {f1:.4f}")
    return accuracy, sensitivity, specificity, f1

def main():
    print("Starting data loading and preprocessing...")
    X, y = load_all_data(window_size=512, stride=256)
    print("Starting feature extraction...")
    start_time = time.time()
    X_features = extract_all_features(X)
    end_time = time.time()
    print(f"Feature extraction completed in {int((end_time - start_time) / 60)} minutes {int((end_time - start_time) % 60)} seconds")
    
    n_channels = X.shape[2]
    feature_types = [
        'engagement', 'theta', 'alpha', 'beta',
        'pac_theta_alpha', 'pac_theta_beta', 'pac_alpha_beta',
        'sprint',
        'wavelet_mean1', 'wavelet_std1', 'wavelet_max1',
        'wavelet_mean2', 'wavelet_std2', 'wavelet_max2',
        'wavelet_mean3', 'wavelet_std3', 'wavelet_max3',
        'wavelet_mean4', 'wavelet_std4', 'wavelet_max4',
        'wavelet_mean5', 'wavelet_std5', 'wavelet_max5',
        'wavelet_mean6', 'wavelet_std6', 'wavelet_max6',
        'line_length', 'hurst_exponent', 'zero_crossing_rate',
        'hjorth_activity', 'hjorth_mobility', 'hjorth_complexity'
    ]
    n_states = 4
    microstate_features = (
        [f'ms_duration_{i}' for i in range(n_states)] +
        [f'ms_occurrence_{i}' for i in range(n_states)] +
        [f'ms_coverage_{i}' for i in range(n_states)] +
        [f'ms_trans_{i}_{j}' for i in range(n_states) for j in range(n_states)] +
        ['gfp_mean', 'gfp_std']
    )
    correlation_features = [f'corr_ch{i}_ch{j}' for i in range(n_channels) for j in range(i + 1, n_channels)]
    expected_channel_features = len(feature_types)
    expected_total_features = (n_channels * expected_channel_features) + len(microstate_features) + len(correlation_features)
    feature_names = (
        [f"{ft}_ch{ch}" for ch in range(n_channels) for ft in feature_types] +
        microstate_features + correlation_features
    )
    print(f"Expected total features: {expected_total_features}")
    print(f"Generated feature names length: {len(feature_names)}")
    if len(feature_names) != X_features.shape[1]:
        print(f"Adjusting feature names to match X_features.shape[1] ({X_features.shape[1]})")
        feature_names = [f"feature_{i}" for i in range(X_features.shape[1])]
    print(f"Final feature names length: {len(feature_names)}")
    
    print("Selecting features...")
    X_final, initial_model, selected_features, X_train, X_test, y_train, y_test = select_features(X_features, y, feature_names)
    
    class_weight = {0: 1.3, 1: 1.0}
    
    print("Tuning LightGBM...")
    lgb_param_dist = {
        'n_estimators': [500, 1000, 1500],
        'max_depth': [5, 10, 15, 20, -1],
        'learning_rate': [0.001, 0.005, 0.01, 0.05, 0.1],
        'num_leaves': [31, 50, 70, 100, 150],
        'min_child_samples': [5, 10, 20, 30],
        'subsample': [0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
        'min_gain_to_split': [0.0, 0.01, 0.1],
        'reg_alpha': [0, 0.1, 1.0],
        'reg_lambda': [0, 0.1, 1.0]
    }
    lgb_base = LGBMClassifier(random_state=42, class_weight=class_weight)
    lgb_search = RandomizedSearchCV(lgb_base, lgb_param_dist, n_iter=50, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
    lgb_search.fit(X_final, y, eval_set=[(X_test, y_test)], callbacks=[early_stopping(stopping_rounds=20)])
    lgb_model = lgb_search.best_estimator_
    print(f"Best LightGBM parameters: {lgb_search.best_params_}")
    print(f"Best LightGBM CV accuracy: {lgb_search.best_score_:.4f}")
    
    print("Tuning XGBoost...")
    xgb_param_dist = {
        'n_estimators': [500, 1000, 1500],
        'max_depth': [5, 10, 15, 20],
        'learning_rate': [0.001, 0.005, 0.01, 0.05, 0.1],
        'subsample': [0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
        'gamma': [0, 0.1, 1, 5],
        'reg_lambda': [0, 1, 5, 10],
        'reg_alpha': [0, 0.1, 1, 5],
        'min_child_weight': [1, 5, 10],
        'scale_pos_weight': [1, 1.3, 2]
    }
    xgb_base = xgb.XGBClassifier(random_state=42)
    xgb_search = RandomizedSearchCV(xgb_base, xgb_param_dist, n_iter=50, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
    xgb_search.fit(X_final, y)
    xgb_model = xgb_search.best_estimator_
    print(f"Best XGBoost parameters: {xgb_search.best_params_}")
    print(f"Best XGBoost CV accuracy: {xgb_search.best_score_:.4f}")
    
    print("Tuning CatBoost...")
    cat_param_dist = {
        'iterations': [1000, 1500, 2000],
        'depth': [6, 8, 10, 12, 16],
        'learning_rate': [0.001, 0.005, 0.01, 0.05, 0.1],
        'l2_leaf_reg': [1, 3, 5, 7, 9],
        'border_count': [32, 64, 128, 256],
        'bagging_temperature': [0, 0.5, 1],
        'class_weights': [{0: 1.3, 1: 1.0}, {0: 1.5, 1: 1.0}]
    }
    cat_base = CatBoostClassifier(random_state=42, verbose=0)
    cat_search = RandomizedSearchCV(cat_base, cat_param_dist, n_iter=50, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
    cat_search.fit(X_final, y)
    cat_model = cat_search.best_estimator_
    print(f"Best CatBoost parameters: {cat_search.best_params_}")
    print(f"Best CatBoost CV accuracy: {cat_search.best_score_:.4f}")
    
    print("Training Stacking Ensemble...")
    estimators = [('lgb', lgb_model), ('xgb', xgb_model), ('cat', cat_model)]
    stacking_model = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(),
        cv=5
    )
    
    print("Performing 10-fold cross-validation...")
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    cv_scores_lgb = cross_val_score(lgb_model, X_final, y, cv=skf, scoring='accuracy')
    cv_scores_xgb = cross_val_score(xgb_model, X_final, y, cv=skf, scoring='accuracy')
    cv_scores_cat = cross_val_score(cat_model, X_final, y, cv=skf, scoring='accuracy')
    cv_scores_stack = cross_val_score(stacking_model, X_final, y, cv=skf, scoring='accuracy')
    print(f"LightGBM Mean CV accuracy: {cv_scores_lgb.mean():.4f} (+/- {cv_scores_lgb.std() * 2:.4f})")
    print(f"XGBoost Mean CV accuracy: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std() * 2:.4f})")
    print(f"CatBoost Mean CV accuracy: {cv_scores_cat.mean():.4f} (+/- {cv_scores_cat.std() * 2:.4f})")
    print(f"Stacking Ensemble Mean CV accuracy: {cv_scores_stack.mean():.4f} (+/- {cv_scores_stack.std() * 2:.4f})")
    
    print("Evaluating models on test set...")
    evaluate_model(lgb_model, X_train, X_test, y_train, y_test, "LightGBM")
    evaluate_model(xgb_model, X_train, X_test, y_train, y_test, "XGBoost")
    evaluate_model(cat_model, X_train, X_test, y_train, y_test, "CatBoost")
    evaluate_model(stacking_model, X_train, X_test, y_train, y_test, "Stacking Ensemble")
    
    print("Computing SHAP values for LightGBM...")
    explainer = shap.TreeExplainer(lgb_model)
    shap_values = explainer.shap_values(X_test)
    plt.figure()
    shap.summary_plot(shap_values[1] if isinstance(shap_values, list) else shap_values, X_test, feature_names=selected_features, show=False)
    plt.title("SHAP Summary Plot (LightGBM)")
    plt.savefig("shap_summary_lgb.png")
    plt.close()
    
    top_feature_idx = np.argmax(np.abs(shap_values[1] if isinstance(shap_values, list) else shap_values).mean(0))
    plt.figure()
    shap.dependence_plot(top_feature_idx, shap_values[1] if isinstance(shap_values, list) else shap_values, X_test, feature_names=selected_features, show=False)
    plt.title(f"SHAP Dependence Plot for {selected_features[top_feature_idx]}")
    plt.savefig("shap_dependence_top_feature.png")
    plt.close()
    
    print("\nTop 10 selected features:")
    for i, feature in enumerate(selected_features[:10]):
        print(f"{i+1}. {feature}")
    print_memory_usage()

if __name__ == "__main__":
    main()

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
from scipy.io import loadmat
from scipy import signal
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from sklearn.decomposition import PCA
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
import xgboost as xgb
import shap
from scipy.signal import hilbert, find_peaks, iirnotch
from sklearn.cluster import KMeans
from tqdm import tqdm
import psutil
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
import mne
from dtaidistance import dtw
from imblearn.combine import SMOTEENN
from hurst import compute_Hc
import sampen
from sklearn.base import BaseEstimator, ClassifierMixin

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.cluster._kmeans")

def print_memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024 / 1024:.2f} MB")

# Preprocessing Functions
def load_and_segment_file(file_path, window_size=512, stride=256, expected_channels=19):
    mat_data = loadmat(file_path)
    var_names = [k for k in mat_data.keys() if not k.startswith('__')]
    if not var_names:
        raise ValueError(f"No non-metadata variables found in {file_path}")
    possible_var_names = ['data', 'eeg', 'signal', 'EEG', 'Data'] + var_names
    data = None
    for var_name in possible_var_names:
        if var_name in mat_data:
            data = mat_data[var_name]
            break
    if data is None:
        raise ValueError(f"No valid EEG data variable found in {file_path}")
    if data.ndim == 1:
        data = data.reshape(1, -1, 1)
    elif data.ndim == 2:
        data = data[np.newaxis, :, :]
    elif data.ndim == 3:
        pass
    else:
        raise ValueError(f"Unexpected data dimensions {data.ndim}")
    n_segments, n_samples, n_channels = data.shape
    if n_channels != expected_channels:
        if n_channels < expected_channels:
            data = np.pad(data, ((0, 0), (0, 0), (0, expected_channels - n_channels)), mode='constant')
        else:
            data = data[:, :, :expected_channels]
    max_abs_value = np.max(np.abs(data))
    if max_abs_value > 1000:
        data = data / 1000.0
    if max_abs_value > 100:
        scaling_factor = 100 / max_abs_value
        data = data * scaling_factor
    all_segments = []
    for seg in range(n_segments):
        seg_data = data[seg]
        seg_samples = seg_data.shape[0]
        if seg_samples < window_size:
            padded_data = np.pad(seg_data, ((0, window_size - seg_samples), (0, 0)), mode='constant')
            all_segments.append(padded_data[np.newaxis, :, :])
            continue
        n_seg_segments = ((seg_samples - window_size + stride - 1) // stride) + 1
        segments = np.zeros((n_seg_segments, window_size, n_channels))
        for i in range(n_seg_segments):
            start = i * stride
            end = min(start + window_size, seg_samples)
            segment_length = end - start
            if segment_length == window_size:
                segments[i] = seg_data[start:end, :]
            else:
                segments[i] = np.pad(seg_data[start:end, :], ((0, window_size - segment_length), (0, 0)), mode='constant')
        all_segments.append(segments)
    if not all_segments:
        return None
    return np.concatenate(all_segments, axis=0)

def load_from_folder(folder_path, window_size=512, stride=256, expected_channels=19):
    if not os.path.exists(folder_path):
        return None
    mat_files = [f for f in os.listdir(folder_path) if f.endswith('.mat')]
    if not mat_files:
        return None
    all_segments = []
    for file in tqdm(mat_files, desc=f"Loading files from {folder_path}"):
        file_path = os.path.join(folder_path, file)
        segments = load_and_segment_file(file_path, window_size, stride, expected_channels)
        if segments is not None:
            all_segments.append(segments)
    if not all_segments:
        return None
    return np.concatenate(all_segments, axis=0)

def load_all_data(window_size=512, stride=256, expected_channels=19):
    adhd_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1"
    adhd_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part2/ADHD_part2"
    control_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part1/Control_part1"
    control_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part2"
    adhd_segments1 = load_from_folder(adhd_path1, window_size, stride, expected_channels)
    adhd_segments2 = load_from_folder(adhd_path2, window_size, stride, expected_channels)
    if adhd_segments1 is None or adhd_segments2 is None:
        raise ValueError("Failed to load ADHD data")
    adhd_segments = np.concatenate([adhd_segments1, adhd_segments2], axis=0)
    control_segment1 = load_from_folder(control_path1, window_size, stride, expected_channels)
    control_segment2 = load_from_folder(control_path2, window_size, stride, expected_channels)
    if control_segment1 is None or control_segment2 is None:
        raise ValueError("Failed to load Control data")
    control_segments = np.concatenate([control_segment1, control_segment2], axis=0)
    y_adhd = np.ones(adhd_segments.shape[0])
    y_control = np.zeros(control_segments.shape[0])
    X = np.concatenate([adhd_segments, control_segments], axis=0)
    y = np.concatenate([y_adhd, y_control])
    X = notch_filter(X)
    X = clip_spikes(X)
    X = butterworth_filter(X)
    X = apply_ica_and_plot(X)
    print("Data loading and preprocessing completed.")
    return X, y

def notch_filter(data, fs=128, freqs=[50, 60], Q=30):
    filtered_data = data.copy()
    for freq in freqs:
        b, a = signal.iirnotch(freq / (fs / 2), Q)
        filtered_data = signal.filtfilt(b, a, filtered_data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def clip_spikes(data, threshold=500):
    clipped_data = data.copy()
    clipped_data[np.abs(clipped_data) > threshold] = np.sign(clipped_data[np.abs(clipped_data) > threshold]) * threshold
    fp2_threshold = 30
    clipped_data[:, :, 1] = np.where(np.abs(clipped_data[:, :, 1]) > fp2_threshold,
                                     np.sign(clipped_data[:, :, 1]) * fp2_threshold,
                                     clipped_data[:, :, 1])
    return clipped_data

def butterworth_filter(data, fs=128, lowcut=1, highcut=45, order=4):
    nyquist = fs / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = signal.butter(order, [low, high], btype='band')
    filtered_data = signal.filtfilt(b, a, data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def apply_ica_and_plot(data, fs=128, n_components=None, segments_to_plot=3, channels_to_plot=19):
    n_segments, n_samples, n_channels = data.shape
    if n_components is None or n_components > n_channels:
        n_components = n_channels
    total_samples = n_segments * n_samples
    concatenated_data = data.reshape(total_samples, n_channels)
    ch_names = [f"EEG {i}" for i in range(n_channels)]
    montage_labels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2'][:n_channels]
    info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types="eeg")
    raw = mne.io.RawArray(concatenated_data.T, info)
    channel_mapping = {f"EEG {i}": montage_labels[i] for i in range(n_channels)}
    raw.rename_channels(channel_mapping)
    try:
        theta_values = [-18, 18, -54, -39, 0, 39, 54, -90, -90, 90, 90, 90, -126, -141, 180, 141, 126, -162, 162][:n_channels]
        radius_values_cm = [51.1, 51.1, 51.1, 33.3, 25.6, 33.3, 51.1, 51.1, 25.6, 0, 25.6, 51.1, 51.1, 33.3, 25.6, 33.3, 51.1, 51.1, 51.1][:n_channels]
        radius_values_m = [r / 100 for r in radius_values_cm]
        ch_pos = {label: [radius * np.cos(np.deg2rad(theta)), radius * np.sin(np.deg2rad(theta)), 0.0]
                  for label, theta, radius in zip(montage_labels, theta_values, radius_values_m)}
        montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='head')
        raw.set_montage(montage)
    except Exception:
        montage = mne.channels.make_standard_montage('standard_1020')
        raw.set_montage(montage)
    raw.filter(l_freq=0.5, h_freq=None, method='iir')
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=42, max_iter=500)
    ica.fit(raw)
    ica_sources = ica.get_sources(raw).get_data().T
    kurtosis_values = np.array([kurtosis(ica_sources[:, i]) for i in range(n_components)])
    variance_values = np.var(ica_sources, axis=0)
    peak_amplitude = np.max(np.abs(ica_sources), axis=0)
    fp2_weight = np.max(np.abs(ica_sources[total_samples // 2:total_samples // 2 + n_samples, 1])) / np.max(peak_amplitude) if n_channels > 1 else 0
    kurtosis_norm = (kurtosis_values - kurtosis_values.min()) / (kurtosis_values.max() - kurtosis_values.min() + 1e-10)
    variance_norm = (variance_values - variance_values.min()) / (variance_values.max() - variance_values.min() + 1e-10)
    peak_norm = (peak_amplitude - peak_amplitude.min()) / (peak_amplitude.max() - peak_amplitude.min() + 1e-10)
    combined_score = 0.45 * kurtosis_norm + 0.25 * variance_norm + 0.2 * peak_norm + 0.1 * fp2_weight
    top_artifact_indices = np.argsort(combined_score)[-min(10, n_components):]
    ica.exclude = top_artifact_indices.tolist()
    raw_clean = ica.apply(raw.copy())
    cleaned_data_raw = raw_clean.get_data()
    cleaned_data = cleaned_data_raw.T.reshape(n_segments, n_samples, n_channels)
    plot_channels = [6, 15, 16] if n_channels >= 17 else range(min(3, n_channels))
    for ch in plot_channels:
        plt.figure(figsize=(10, 5))
        plt.plot(data[0, :, ch], label=f'Raw ch{ch}')
        plt.plot(cleaned_data[0, :, ch], label=f'Cleaned ch{ch}')
        plt.title(f"Raw vs. Cleaned Data for Channel {ch}")
        plt.legend()
        plt.savefig(f"ica_comparison_ch{ch}.png")
        plt.close()
    return cleaned_data

# Feature Extraction Functions
def normalize_data(data):
    scaler = StandardScaler()
    return scaler.fit_transform(data)

def calculate_engagement_index(data, fs=128):
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    alpha = calculate_psd(data, fs, freq_range=(8, 13))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    return np.mean((theta + alpha) / (beta + 1e-10))

def calculate_psd(data, fs=128, freq_range=None):
    nperseg = min(len(data), 512)
    f, psd = signal.welch(data, fs=fs, nperseg=nperseg, axis=0)
    if freq_range:
        freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
        return np.mean(psd[freq_mask], axis=0) if freq_mask.any() else 0
    return psd.mean(axis=0)

def phase_amplitude_coupling(data, fs=128):
    theta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 4, 8)[0, :, 0]
    alpha = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 8, 13)[0, :, 0]
    beta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 13, 30)[0, :, 0]
    theta_phase = np.angle(hilbert(theta))
    alpha_phase = np.angle(hilbert(alpha))
    beta_phase = np.angle(hilbert(beta))
    theta_amp = np.abs(hilbert(theta))
    alpha_amp = np.abs(hilbert(alpha))
    beta_amp = np.abs(hilbert(beta))
    pac_theta_alpha = np.abs(np.mean(alpha_amp * np.exp(1j * theta_phase), axis=-1))
    pac_theta_beta = np.abs(np.mean(beta_amp * np.exp(1j * theta_phase), axis=-1))
    pac_alpha_theta = np.abs(np.mean(theta_amp * np.exp(1j * alpha_phase), axis=-1))
    pac_alpha_beta = np.abs(np.mean(beta_amp * np.exp(1j * alpha_phase), axis=-1))
    pac_beta_theta = np.abs(np.mean(theta_amp * np.exp(1j * beta_phase), axis=-1))
    pac_beta_alpha = np.abs(np.mean(alpha_amp * np.exp(1j * beta_phase), axis=-1))
    pac_features = []
    for pac in [pac_theta_alpha, pac_theta_beta, pac_alpha_theta, pac_alpha_beta, pac_beta_theta, pac_beta_alpha]:
        pac_features.extend([np.mean(pac), np.std(pac), np.max(pac)])
    return np.array(pac_features)

def calculate_sprint(data, fs=128):
    nperseg = min(len(data), 512)
    noverlap = nperseg // 2
    f, t, Zxx = signal.stft(data, fs=fs, nperseg=nperseg, noverlap=noverlap)
    power = np.abs(Zxx)**2
    return np.mean(power)

def wavelet_features(data):
    scales = np.arange(1, 7)
    coeffs, _ = pywt.cwt(data, scales, 'mexh', axis=0)
    features = []
    for c in coeffs:
        features.extend([np.mean(np.abs(c)), np.std(c), np.max(np.abs(c))])
    return np.array(features)

def hjorth_parameters(data):
    diff1 = np.diff(data)
    diff2 = np.diff(diff1)
    activity = np.var(data)
    mobility = np.sqrt(np.var(diff1) / activity) if activity > 0 else 0
    complexity = np.sqrt(np.var(diff2) / np.var(diff1)) / mobility if mobility > 0 else 0
    return np.array([activity, mobility, complexity])

def line_length(data):
    return np.sum(np.abs(np.diff(data)))

def hurst_exponent(data):
    H, _, _ = compute_Hc(data, kind='change', simplified=True)
    return H

def zero_crossing_rate(data, prev_zcr=None):
    if len(data) <= 1 or np.any(np.isnan(data)) or np.std(data) == 0:  # Insufficient or invalid data
        return prev_zcr if prev_zcr is not None else 0
    return len(np.where(np.diff(np.sign(data)))[0]) / len(data)

def sample_entropy(data, prev_sampen=None):
    if len(data) <= 2 or np.any(np.isnan(data)) or np.std(data) == 0:  # Insufficient or invalid data
        return prev_sampen if prev_sampen is not None else 0
    return sampen.sampen2(data, mm=2, r=0.2 * np.std(data))[-1][1]

def theta_beta_ratio(data, fs=128):
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    return theta / (beta + 1e-10)

def channel_coherence(data, fs=128):
    n_channels = data.shape[1]
    coherence_features = []
    nperseg = min(data.shape[0], 512)
    for band, freq_range in [('theta', (4, 8)), ('alpha', (8, 13)), ('beta', (13, 30))]:
        for i in range(n_channels):
            for j in range(i + 1, n_channels):
                f, coh = signal.coherence(data[:, i], data[:, j], fs=fs, nperseg=nperseg)
                freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
                coherence_features.append(np.mean(coh[freq_mask]) if freq_mask.any() else 0)
    return np.array(coherence_features)

def channel_correlation(data):
    n_channels = data.shape[1]
    corr_features = []
    for i in range(n_channels):
        for j in range(i + 1, n_channels):
            corr = np.corrcoef(data[:, i], data[:, j])[0, 1]
            corr_features.append(corr)
    return np.array(corr_features)

def microstate_analysis(data, fs=128, n_states=4):
    gfp = np.std(data, axis=1)
    peaks, _ = find_peaks(gfp, distance=fs//10)
    if len(peaks) < n_states:
        peak_data = data
    else:
        peak_data = data[peaks]
    kmeans = KMeans(n_clusters=n_states, random_state=42, n_init=10)
    kmeans.fit(peak_data)
    templates = kmeans.cluster_centers_
    distances = np.zeros((data.shape[0], n_states))
    for i, template in enumerate(templates):
        template_norm = np.linalg.norm(template)
        data_norm = np.linalg.norm(data, axis=1)
        distances[:, i] = np.abs(np.sum(data * template, axis=1) / (data_norm * template_norm + 1e-10))
    labels = np.argmax(distances, axis=1)
    durations = np.zeros(n_states)
    occurrences = np.zeros(n_states)
    gfp_peaks = gfp[peaks] if len(peaks) > 0 else gfp
    gfp_mean = np.mean(gfp_peaks)
    gfp_std = np.std(gfp_peaks)
    for state in range(n_states):
        state_indices = np.where(labels == state)[0]
        if len(state_indices) > 0:
            diff = np.diff(state_indices)
            breaks = np.where(diff > 1)[0]
            segment_lengths = np.diff(np.concatenate([[0], breaks + 1, [len(state_indices)]]))
            durations[state] = np.mean(segment_lengths) / fs * 1000 if len(segment_lengths) > 0 else 0
            occurrences[state] = len(breaks) + 1 if len(breaks) > 0 else 1 if len(state_indices) > 0 else 0
    coverage = np.bincount(labels, minlength=n_states).astype(float) / len(labels)
    transitions = np.zeros((n_states, n_states))
    for t in range(len(labels)-1):
        transitions[labels[t], labels[t+1]] += 1
    transition_sums = transitions.sum(axis=1, keepdims=True)
    transitions = transitions / (transition_sums + 1e-10)
    microstate_features = np.hstack([durations, occurrences, coverage, transitions.ravel(), [gfp_mean, gfp_std]])
    return microstate_features

def extract_features_single(sample, idx, prev_features=None):
    n_channels = sample.shape[1]
    normalized = normalize_data(sample)
    microstates = microstate_analysis(normalized)
    correlations = channel_correlation(normalized)
    coherences = channel_coherence(normalized)
    channel_features = []

    # Initialize previous feature placeholders per channel if not provided
    if prev_features is None:
        prev_zcr = [0] * n_channels
        prev_sampen = [0] * n_channels
    else:
        prev_per_channel = prev_features[:n_channels * 50]  # Assuming 50 features per channel from previous calc
        prev_zcr = [prev_per_channel[ch * 50 + 43] for ch in range(n_channels)]  # 43rd index is zcr
        prev_sampen = [prev_per_channel[ch * 50 + 47] for ch in range(n_channels)]  # 47th index is sample_entropy

    for ch in range(n_channels):
        channel_data = normalized[:, ch]
        engagement = calculate_engagement_index(channel_data)
        theta = calculate_psd(channel_data, freq_range=(4, 8))
        alpha = calculate_psd(channel_data, freq_range=(8, 13))
        beta = calculate_psd(channel_data, freq_range=(13, 30))
        pac = phase_amplitude_coupling(channel_data)
        sprint = calculate_sprint(channel_data)
        wavelet = wavelet_features(channel_data)
        ll = line_length(channel_data)
        he = hurst_exponent(channel_data)
        zcr = zero_crossing_rate(channel_data, prev_zcr[ch])
        hjorth = hjorth_parameters(channel_data)
        tbr = theta_beta_ratio(channel_data)
        sentropy = sample_entropy(channel_data, prev_sampen[ch])
        skewness = skew(channel_data)
        kurt = kurtosis(channel_data)
        ch_features = np.concatenate([
            [engagement], [theta], [alpha], [beta], pac, [sprint], wavelet,
            [ll], [he], [zcr], hjorth, [tbr], [sentropy], [skewness], [kurt]
        ])
        channel_features.append(ch_features)
    per_channel_features = np.array(channel_features)
    sample_features = np.concatenate([per_channel_features.flatten(), microstates, correlations, coherences])
    return sample_features

def extract_all_features(data, chunk_size=100):
    n_samples, n_samples_per_segment, n_channels = data.shape
    feature_matrix = []
    prev_features = None  # Initialize previous features as None for the first segment

    with tqdm(total=n_samples, desc="Extracting features", unit="segment",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]") as pbar:
        for i in range(n_samples):
            sample_features = extract_features_single(data[i], i, prev_features)
            feature_matrix.append(sample_features)
            prev_features = sample_features  # Update previous features for the next iteration
            pbar.update(1)
    feature_matrix = np.array(feature_matrix)
    print(f"Feature matrix shape: {feature_matrix.shape}")
    return feature_matrix

# Feature Selection and Evaluation
def select_features(X, y, feature_names):
    smoteenn = SMOTEENN(random_state=42)
    X_balanced, y_balanced = smoteenn.fit_resample(X, y)
    print(f"After SMOTEENN: {X_balanced.shape[0]} samples, {np.bincount(y_balanced.astype(int))} classes")

    X_temp, X_test, y_temp, y_test = train_test_split(X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

    initial_model = xgb.XGBClassifier(random_state=42, n_estimators=100, use_label_encoder=False, eval_metric='logloss')
    initial_model.fit(X_train, y_train)
    explainer = shap.TreeExplainer(initial_model)
    shap_values = explainer.shap_values(X_train)
    shap_importance = np.abs(shap_values).mean(axis=0)
    n_features_to_select = max(1, int(X_train.shape[1] * 0.25))
    important_indices = np.argsort(shap_importance)[-n_features_to_select:]
    X_train_shap = X_train[:, important_indices]
    X_val_shap = X_val[:, important_indices]
    X_test_shap = X_test[:, important_indices]
    f_values, p_values = f_classif(X_train_shap, y_train)
    significant_features = p_values < 0.01
    if not significant_features.any():
        significant_features = np.ones_like(p_values, dtype=bool)
    X_train_anova = X_train_shap[:, significant_features]
    X_val_anova = X_val_shap[:, significant_features]
    X_test_anova = X_test_shap[:, significant_features]
    pca = PCA(n_components=min(0.95, X_train_anova.shape[1]))
    X_train_pca = pca.fit_transform(X_train_anova)
    X_val_pca = pca.transform(X_val_anova)
    X_test_pca = pca.transform(X_test_anova)
    selected_features = np.array(feature_names)[important_indices][significant_features]
    return X_train_pca, X_val_pca, X_test_pca, y_train, y_val, y_test, selected_features, initial_model, shap_values

def evaluate_model(model, X, y, dataset_name):
    y_pred = model.predict(X)
    cm = confusion_matrix(y, y_pred)
    report = classification_report(y, y_pred, output_dict=True)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = report['weighted avg']['f1-score']
    print(f"\n{dataset_name} Metrics: Sensitivity={sensitivity:.4f}, Specificity={specificity:.4f}, F1={f1:.4f}")
    return sensitivity, specificity, f1

# Updated KNNDTWClassifier
class KNNDTWClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self

    def predict(self, X):
        predictions = []
        for x in tqdm(X, desc="Predicting with KNN-DTW"):
            distances = [dtw.distance(x, x_train) for x_train in self.X_train]
            nearest_indices = np.argsort(distances)[:self.n_neighbors]
            nearest_labels = self.y_train[nearest_indices]
            prediction = np.bincount(nearest_labels.astype(int)).argmax()
            predictions.append(prediction)
        return np.array(predictions)

    def get_params(self, deep=True):
        """Return parameters as a dictionary."""
        return {"n_neighbors": self.n_neighbors}

    def set_params(self, **params):
        """Set parameters and return self."""
        for param, value in params.items():
            setattr(self, param, value)
        return self

# Main Execution
def main():
    X, y = load_all_data()
    X_features = extract_all_features(X)

    n_channels = X.shape[2]
    n_samples_per_segment = X.shape[1]

    feature_types = [
        'engagement', 'theta', 'alpha', 'beta',
        'pac_theta_alpha_mean', 'pac_theta_alpha_std', 'pac_theta_alpha_max',
        'pac_theta_beta_mean', 'pac_theta_beta_std', 'pac_theta_beta_max',
        'pac_alpha_theta_mean', 'pac_alpha_theta_std', 'pac_alpha_theta_max',
        'pac_alpha_beta_mean', 'pac_alpha_beta_std', 'pac_alpha_beta_max',
        'pac_beta_theta_mean', 'pac_beta_theta_std', 'pac_beta_theta_max',
        'pac_beta_alpha_mean', 'pac_beta_alpha_std', 'pac_beta_alpha_max',
        'sprint',
        'wavelet_mean1', 'wavelet_std1', 'wavelet_max1',
        'wavelet_mean2', 'wavelet_std2', 'wavelet_max2',
        'wavelet_mean3', 'wavelet_std3', 'wavelet_max3',
        'wavelet_mean4', 'wavelet_std4', 'wavelet_max4',
        'wavelet_mean5', 'wavelet_std5', 'wavelet_max5',
        'wavelet_mean6', 'wavelet_std6', 'wavelet_max6',
        'line_length', 'hurst_exponent', 'zero_crossing_rate',
        'hjorth_activity', 'hjorth_mobility', 'hjorth_complexity',
        'theta_beta_ratio', 'sample_entropy', 'skewness', 'kurtosis'
    ]
    n_states = 4
    n_wavelet_scales = 6
    n_pac_features = 18

    per_channel_feature_count = (
        1 + 1 + 1 + 1 + n_pac_features + 1 + n_wavelet_scales * 3 +
        1 + 1 + 1 + 3 + 1 + 1 + 1 + 1
    )
    microstate_feature_count = n_states + n_states + n_states + n_states * n_states + 2
    correlation_feature_count = n_channels * (n_channels - 1) // 2
    coherence_feature_count = 3 * (n_channels * (n_channels - 1) // 2)

    expected_feature_count = (
        n_channels * per_channel_feature_count +
        microstate_feature_count +
        correlation_feature_count +
        coherence_feature_count
    )

    print(f"Expected feature count: {expected_feature_count}")
    if X_features.shape[1] != expected_feature_count:
        raise ValueError(f"Feature dimension mismatch: {X_features.shape[1]} vs {expected_feature_count}")

    feature_names = (
        [f"{ft}_ch{ch}" for ch in range(n_channels) for ft in feature_types] +
        [f'ms_duration_{i}' for i in range(n_states)] +
        [f'ms_occurrence_{i}' for i in range(n_states)] +
        [f'ms_coverage_{i}' for i in range(n_states)] +
        [f'ms_trans_{i}_{j}' for i in range(n_states) for j in range(n_states)] +
        ['gfp_mean', 'gfp_std'] +
        [f'corr_ch{i}_ch{j}' for i in range(n_channels) for j in range(i + 1, n_channels)] +
        [f'coh_theta_ch{i}_ch{j}' for i in range(n_channels) for j in range(i + 1, n_channels)] +
        [f'coh_alpha_ch{i}_ch{j}' for i in range(n_channels) for j in range(i + 1, n_channels)] +
        [f'coh_beta_ch{i}_ch{j}' for i in range(n_channels) for j in range(i + 1, n_channels)]
    )

    print("Selecting features...")
    X_train_pca, X_val_pca, X_test_pca, y_train, y_val, y_test, selected_features, initial_model, initial_shap_values = select_features(X_features, y, feature_names)

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_results = []

    X_combined = np.vstack([X_train_pca, X_val_pca])
    y_combined = np.hstack([y_train, y_val])
    if X_combined.shape[0] != y_combined.shape[0]:
        raise ValueError(f"Dimension mismatch in CV split: X={X_combined.shape[0]}, y={y_combined.shape[0]}")

    for fold, (train_idx, val_idx) in enumerate(tqdm(cv.split(X_combined, y_combined), total=10, desc="Cross-validation folds")):
        X_train_fold = X_combined[train_idx]
        y_train_fold = y_combined[train_idx]
        X_val_fold = X_combined[val_idx]
        y_val_fold = y_combined[val_idx]

        cat_param_dist = {
            'iterations': [100, 200, 300],
            'depth': [4, 6, 8],
            'learning_rate': [0.01, 0.05, 0.1],
            'l2_leaf_reg': [1, 3, 5],
            'border_count': [32, 64, 128]
        }
        cat_base = CatBoostClassifier(random_state=42, verbose=0)
        cat_search = RandomizedSearchCV(cat_base, cat_param_dist, n_iter=20, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
        cat_search.fit(X_train_fold, y_train_fold)
        cat_model = cat_search.best_estimator_

        knn_dtw_model = KNNDTWClassifier(n_neighbors=5)
        knn_dtw_model.fit(X_train_fold, y_train_fold)

        gb_param_dist = {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.05, 0.1],
            'subsample': [0.6, 0.8, 1.0]
        }
        gb_base = GradientBoostingClassifier(random_state=42)
        gb_search = RandomizedSearchCV(gb_base, gb_param_dist, n_iter=20, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
        gb_search.fit(X_train_fold, y_train_fold)
        gb_model = gb_search.best_estimator_

        estimators = [('cat', cat_model), ('knn_dtw', knn_dtw_model), ('gb', gb_model)]
        stack_model = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(random_state=42), cv=5)
        stack_model.fit(X_train_fold, y_train_fold)

        cat_val_sens, cat_val_spec, cat_val_f1 = evaluate_model(cat_model, X_val_fold, y_val_fold, f"Fold {fold+1} CatBoost Val")
        stack_val_sens, stack_val_spec, stack_val_f1 = evaluate_model(stack_model, X_val_fold, y_val_fold, f"Fold {fold+1} Stack Val")
        cat_test_sens, cat_test_spec, cat_test_f1 = evaluate_model(cat_model, X_test_pca, y_test, f"Fold {fold+1} CatBoost Test")
        stack_test_sens, stack_test_spec, stack_test_f1 = evaluate_model(stack_model, X_test_pca, y_test, f"Fold {fold+1} Stack Test")

        fold_results.append({
            'fold': fold + 1,
            'cat_val': (cat_val_sens, cat_val_spec, cat_val_f1),
            'stack_val': (stack_val_sens, stack_val_spec, stack_val_f1),
            'cat_test': (cat_test_sens, cat_test_spec, cat_test_f1),
            'stack_test': (stack_test_sens, stack_test_spec, stack_test_f1)
        })

    for model in ['cat', 'stack']:
        val_sens = np.mean([r[f'{model}_val'][0] for r in fold_results])
        val_spec = np.mean([r[f'{model}_val'][1] for r in fold_results])
        val_f1 = np.mean([r[f'{model}_val'][2] for r in fold_results])
        test_sens = np.mean([r[f'{model}_test'][0] for r in fold_results])
        test_spec = np.mean([r[f'{model}_test'][1] for r in fold_results])
        test_f1 = np.mean([r[f'{model}_test'][2] for r in fold_results])
        print(f"\n{model.upper()} Model Average Metrics:")
        print(f"Validation - Sensitivity: {val_sens:.4f}, Specificity: {val_spec:.4f}, F1: {val_f1:.4f}")
        print(f"Test - Sensitivity: {test_sens:.4f}, Specificity: {test_spec:.4f}, F1: {test_f1:.4f}")

    print("\nComputing SHAP values for XGBoost (initial model)...")
    explainer = shap.TreeExplainer(initial_model)
    shap_values = explainer.shap_values(X_test_pca)
    plt.figure()
    shap.summary_plot(shap_values, X_test_pca, feature_names=[f"PCA_{i}" for i in range(X_test_pca.shape[1])], show=False)
    plt.title("SHAP Summary Plot (XGBoost - PCA Features)")
    plt.savefig("shap_summary_xgboost_pca.png")
    plt.close()

    plt.figure()
    shap.summary_plot(initial_shap_values, X_train[:, important_indices][:, significant_features],
                      feature_names=selected_features, plot_type="bar", max_display=20, show=False)
    plt.title("SHAP Bar Plot (XGBoost - Original Features)")
    plt.savefig("shap_bar_xgboost_original.png")
    plt.close()

    print("\nTop 10 Selected Features:")
    for i, feature in enumerate(selected_features[:min(10, len(selected_features))]):
        print(f"{i+1}. {feature}")
    print_memory_usage()

if __name__ == "__main__":
    main()

Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1: 100%|█| 30/30 [00:00<00:
Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part2/ADHD_part2: 100%|█| 31/31 [00:00<00:
Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part1/Control_part1: 100%|█| 30/30 [00:
Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part2: 100%|█| 30/30 [00:00<00:00, 121.


Creating RawArray with float64 data, n_channels=19, n_times=4300288
    Range : 0 ... 4300287 =      0.000 ... 33595.992 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 0.5 Hz

IIR filter parameters
---------------------
Butterworth highpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 0.50 Hz: -6.02 dB

Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Fitting ICA took 177.8s.
Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 10 ICA components
    Projecting back using 19 PCA components
Data loading and preprocessing completed.


Extracting features: 100%|██████████████████████████████████████████████████| 8399/8399 [4:59:49<00:00,  2.14s/segment]


Feature matrix shape: (8399, 1683)
Expected feature count: 1683
Selecting features...
After SMOTEENN: 6395 samples, [3337 3058] classes


C:\Users\ADMIN\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [23:57:14] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-06abd128ca6c1688d-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
Predicting with KNN-DTW: 100%|███████████████████████████████████████████████████████| 921/921 [09:07<00:00,  1.68it/s]

Predicting with KNN-DTW: 100%|███████████████████████████████████████████████████████| 921/921 [10:07<00:00,  1.52it/s]

Predicting with KNN-DTW: 100%|███████████████████████████████████████████████████████| 921/921 [10:10<00:00,  1.51it/s]

Predicting with KNN-DTW: 100%|███████████████████████████████████████████████████████| 921/921 [10:55<00:00,  1.41it/s]

Predicting with KNN-DTW:   0%|                                                                 | 0/920 [00:00<?, ?it/s]

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
from scipy.io import loadmat
from scipy import signal
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import shap
from scipy.signal import hilbert, find_peaks, iirnotch
from sklearn.cluster import KMeans
from tqdm import tqdm
import psutil
from sklearn.metrics import confusion_matrix, classification_report, recall_score, f1_score
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
import mne
from dtaidistance import dtw
from imblearn.combine import SMOTEENN
from hurst import compute_Hc
import sampen
from sklearn.base import BaseEstimator, ClassifierMixin

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.cluster._kmeans")

def print_memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024 / 1024:.2f} MB")

# Preprocessing Functions (unchanged)
def load_and_segment_file(file_path, window_size=512, stride=256, expected_channels=19):
    mat_data = loadmat(file_path)
    var_names = [k for k in mat_data.keys() if not k.startswith('__')]
    if not var_names:
        raise ValueError(f"No non-metadata variables found in {file_path}")
    possible_var_names = ['data', 'eeg', 'signal', 'EEG', 'Data'] + var_names
    data = None
    for var_name in possible_var_names:
        if var_name in mat_data:
            data = mat_data[var_name]
            break
    if data is None:
        raise ValueError(f"No valid EEG data variable found in {file_path}")
    if data.ndim == 1:
        data = data.reshape(1, -1, 1)
    elif data.ndim == 2:
        data = data[np.newaxis, :, :]
    elif data.ndim == 3:
        pass
    else:
        raise ValueError(f"Unexpected data dimensions {data.ndim}")
    n_segments, n_samples, n_channels = data.shape
    if n_channels != expected_channels:
        if n_channels < expected_channels:
            data = np.pad(data, ((0, 0), (0, 0), (0, expected_channels - n_channels)), mode='constant')
        else:
            data = data[:, :, :expected_channels]
    max_abs_value = np.max(np.abs(data))
    if max_abs_value > 1000:
        data = data / 1000.0
    if max_abs_value > 100:
        scaling_factor = 100 / max_abs_value
        data = data * scaling_factor
    all_segments = []
    for seg in range(n_segments):
        seg_data = data[seg]
        seg_samples = seg_data.shape[0]
        if seg_samples < window_size:
            padded_data = np.pad(seg_data, ((0, window_size - seg_samples), (0, 0)), mode='constant')
            all_segments.append(padded_data[np.newaxis, :, :])
            continue
        n_seg_segments = ((seg_samples - window_size + stride - 1) // stride) + 1
        segments = np.zeros((n_seg_segments, window_size, n_channels))
        for i in range(n_seg_segments):
            start = i * stride
            end = min(start + window_size, seg_samples)
            segment_length = end - start
            if segment_length == window_size:
                segments[i] = seg_data[start:end, :]
            else:
                segments[i] = np.pad(seg_data[start:end, :], ((0, window_size - segment_length), (0, 0)), mode='constant')
        all_segments.append(segments)
    if not all_segments:
        return None
    return np.concatenate(all_segments, axis=0)

def load_from_folder(folder_path, window_size=512, stride=256, expected_channels=19):
    if not os.path.exists(folder_path):
        return None
    mat_files = [f for f in os.listdir(folder_path) if f.endswith('.mat')]
    if not mat_files:
        return None
    all_segments = []
    for file in tqdm(mat_files, desc=f"Loading files from {folder_path}"):
        file_path = os.path.join(folder_path, file)
        segments = load_and_segment_file(file_path, window_size, stride, expected_channels)
        if segments is not None:
            all_segments.append(segments)
    if not all_segments:
        return None
    return np.concatenate(all_segments, axis=0)

def load_all_data(window_size=512, stride=256, expected_channels=19):
    adhd_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1"
    adhd_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part2/ADHD_part2"
    control_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part1/Control_part1"
    control_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part2"
    adhd_segments1 = load_from_folder(adhd_path1, window_size, stride, expected_channels)
    adhd_segments2 = load_from_folder(adhd_path2, window_size, stride, expected_channels)
    if adhd_segments1 is None or adhd_segments2 is None:
        raise ValueError("Failed to load ADHD data")
    adhd_segments = np.concatenate([adhd_segments1, adhd_segments2], axis=0)
    control_segment1 = load_from_folder(control_path1, window_size, stride, expected_channels)
    control_segment2 = load_from_folder(control_path2, window_size, stride, expected_channels)
    if control_segment1 is None or control_segment2 is None:
        raise ValueError("Failed to load Control data")
    control_segments = np.concatenate([control_segment1, control_segment2], axis=0)
    y_adhd = np.ones(adhd_segments.shape[0])
    y_control = np.zeros(control_segments.shape[0])
    X = np.concatenate([adhd_segments, control_segments], axis=0)
    y = np.concatenate([y_adhd, y_control])
    X = notch_filter(X)
    X = clip_spikes(X)
    X = butterworth_filter(X)
    X = apply_ica_and_plot(X)
    print("Data loading and preprocessing completed.")
    return X, y

def notch_filter(data, fs=128, freqs=[50, 60], Q=30):
    filtered_data = data.copy()
    for freq in freqs:
        b, a = signal.iirnotch(freq / (fs / 2), Q)
        filtered_data = signal.filtfilt(b, a, filtered_data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def clip_spikes(data, threshold=500):
    clipped_data = data.copy()
    clipped_data[np.abs(clipped_data) > threshold] = np.sign(clipped_data[np.abs(clipped_data) > threshold]) * threshold
    fp2_threshold = 30
    clipped_data[:, :, 1] = np.where(np.abs(clipped_data[:, :, 1]) > fp2_threshold,
                                     np.sign(clipped_data[:, :, 1]) * fp2_threshold,
                                     clipped_data[:, :, 1])
    return clipped_data

def butterworth_filter(data, fs=128, lowcut=1, highcut=45, order=4):
    nyquist = fs / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = signal.butter(order, [low, high], btype='band')
    filtered_data = signal.filtfilt(b, a, data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def apply_ica_and_plot(data, fs=128, n_components=None, segments_to_plot=3, channels_to_plot=19):
    n_segments, n_samples, n_channels = data.shape
    if n_components is None or n_components > n_channels:
        n_components = n_channels
    total_samples = n_segments * n_samples
    concatenated_data = data.reshape(total_samples, n_channels)
    ch_names = [f"EEG {i}" for i in range(n_channels)]
    montage_labels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2'][:n_channels]
    info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types="eeg")
    raw = mne.io.RawArray(concatenated_data.T, info)
    channel_mapping = {f"EEG {i}": montage_labels[i] for i in range(n_channels)}
    raw.rename_channels(channel_mapping)
    try:
        theta_values = [-18, 18, -54, -39, 0, 39, 54, -90, -90, 90, 90, 90, -126, -141, 180, 141, 126, -162, 162][:n_channels]
        radius_values_cm = [51.1, 51.1, 51.1, 33.3, 25.6, 33.3, 51.1, 51.1, 25.6, 0, 25.6, 51.1, 51.1, 33.3, 25.6, 33.3, 51.1, 51.1, 51.1][:n_channels]
        radius_values_m = [r / 100 for r in radius_values_cm]
        ch_pos = {label: [radius * np.cos(np.deg2rad(theta)), radius * np.sin(np.deg2rad(theta)), 0.0]
                  for label, theta, radius in zip(montage_labels, theta_values, radius_values_m)}
        montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='head')
        raw.set_montage(montage)
    except Exception:
        montage = mne.channels.make_standard_montage('standard_1020')
        raw.set_montage(montage)
    raw.filter(l_freq=0.5, h_freq=None, method='iir')
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=42, max_iter=500)
    ica.fit(raw)
    ica_sources = ica.get_sources(raw).get_data().T
    kurtosis_values = np.array([kurtosis(ica_sources[:, i]) for i in range(n_components)])
    variance_values = np.var(ica_sources, axis=0)
    peak_amplitude = np.max(np.abs(ica_sources), axis=0)
    fp2_weight = np.max(np.abs(ica_sources[total_samples // 2:total_samples // 2 + n_samples, 1])) / np.max(peak_amplitude) if n_channels > 1 else 0
    kurtosis_norm = (kurtosis_values - kurtosis_values.min()) / (kurtosis_values.max() - kurtosis_values.min() + 1e-10)
    variance_norm = (variance_values - variance_values.min()) / (variance_values.max() - variance_values.min() + 1e-10)
    peak_norm = (peak_amplitude - peak_amplitude.min()) / (peak_amplitude.max() - peak_amplitude.min() + 1e-10)
    combined_score = 0.45 * kurtosis_norm + 0.25 * variance_norm + 0.2 * peak_norm + 0.1 * fp2_weight
    top_artifact_indices = np.argsort(combined_score)[-min(10, n_components):]
    ica.exclude = top_artifact_indices.tolist()
    raw_clean = ica.apply(raw.copy())
    cleaned_data_raw = raw_clean.get_data()
    cleaned_data = cleaned_data_raw.T.reshape(n_segments, n_samples, n_channels)
    plot_channels = [6, 15, 16] if n_channels >= 17 else range(min(3, n_channels))
    for ch in plot_channels:
        plt.figure(figsize=(10, 5))
        plt.plot(data[0, :, ch], label=f'Raw ch{ch}')
        plt.plot(cleaned_data[0, :, ch], label=f'Cleaned ch{ch}')
        plt.title(f"Raw vs. Cleaned Data for Channel {ch}")
        plt.legend()
        plt.savefig(f"ica_comparison_ch{ch}.png")
        plt.close()
    return cleaned_data

# Feature Extraction Functions with Fallbacks
def normalize_data(data):
    scaler = StandardScaler()
    return scaler.fit_transform(data)

def calculate_engagement_index(data, fs=128, prev_value=None):
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    alpha = calculate_psd(data, fs, freq_range=(8, 13))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    if np.any(np.isnan([theta, alpha, beta])) or beta == 0:
        return prev_value if prev_value is not None else 0
    return np.mean((theta + alpha) / (beta + 1e-10))

def calculate_psd(data, fs=128, freq_range=None):
    nperseg = min(len(data), 512)
    f, psd = signal.welch(data, fs=fs, nperseg=nperseg, axis=0)
    if freq_range:
        freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
        return np.mean(psd[freq_mask], axis=0) if freq_mask.any() else 0
    return psd.mean(axis=0)

def phase_amplitude_coupling(data, fs=128, prev_pac=None):
    theta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 4, 8)[0, :, 0]
    alpha = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 8, 13)[0, :, 0]
    beta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 13, 30)[0, :, 0]
    if np.any(np.isnan([theta, alpha, beta])) or np.std([theta, alpha, beta]) == 0:
        return prev_pac if prev_pac is not None else np.zeros(18)
    theta_phase = np.angle(hilbert(theta))
    alpha_phase = np.angle(hilbert(alpha))
    beta_phase = np.angle(hilbert(beta))
    theta_amp = np.abs(hilbert(theta))
    alpha_amp = np.abs(hilbert(alpha))
    beta_amp = np.abs(hilbert(beta))
    pac_theta_alpha = np.abs(np.mean(alpha_amp * np.exp(1j * theta_phase), axis=-1))
    pac_theta_beta = np.abs(np.mean(beta_amp * np.exp(1j * theta_phase), axis=-1))
    pac_alpha_theta = np.abs(np.mean(theta_amp * np.exp(1j * alpha_phase), axis=-1))
    pac_alpha_beta = np.abs(np.mean(beta_amp * np.exp(1j * alpha_phase), axis=-1))
    pac_beta_theta = np.abs(np.mean(theta_amp * np.exp(1j * beta_phase), axis=-1))
    pac_beta_alpha = np.abs(np.mean(alpha_amp * np.exp(1j * beta_phase), axis=-1))
    pac_features = []
    for pac in [pac_theta_alpha, pac_theta_beta, pac_alpha_theta, pac_alpha_beta, pac_beta_theta, pac_beta_alpha]:
        pac_features.extend([np.mean(pac), np.std(pac), np.max(pac)])
    return np.array(pac_features)

def calculate_sprint(data, fs=128, prev_value=None):
    nperseg = min(len(data), 512)
    noverlap = nperseg // 2
    f, t, Zxx = signal.stft(data, fs=fs, nperseg=nperseg, noverlap=noverlap)
    power = np.abs(Zxx)**2
    if np.any(np.isnan(power)):
        return prev_value if prev_value is not None else 0
    return np.mean(power)

def wavelet_features(data, prev_wavelet=None):
    scales = np.arange(1, 7)
    coeffs, _ = pywt.cwt(data, scales, 'mexh', axis=0)
    if np.any(np.isnan(coeffs)):
        return prev_wavelet if prev_wavelet is not None else np.zeros(18)
    features = []
    for c in coeffs:
        features.extend([np.mean(np.abs(c)), np.std(c), np.max(np.abs(c))])
    return np.array(features)

def hjorth_parameters(data, prev_hjorth=None):
    diff1 = np.diff(data)
    diff2 = np.diff(diff1)
    activity = np.var(data)
    if np.any(np.isnan([activity, np.var(diff1), np.var(diff2)])) or activity == 0:
        return prev_hjorth if prev_hjorth is not None else np.zeros(3)
    mobility = np.sqrt(np.var(diff1) / activity)
    complexity = np.sqrt(np.var(diff2) / np.var(diff1)) / mobility if mobility > 0 else 0
    return np.array([activity, mobility, complexity])

def line_length(data, prev_value=None):
    diff = np.diff(data)
    if np.any(np.isnan(diff)):
        return prev_value if prev_value is not None else 0
    return np.sum(np.abs(diff))

def hurst_exponent(data, prev_value=None):
    if len(data) < 10 or np.any(np.isnan(data)) or np.std(data) == 0:
        return prev_value if prev_value is not None else 0.5  # Default Hurst = 0.5
    H, _, _ = compute_Hc(data, kind='change', simplified=True)
    return H

def zero_crossing_rate(data, prev_zcr=None):
    if len(data) <= 1 or np.any(np.isnan(data)) or np.std(data) == 0:
        return prev_zcr if prev_zcr is not None else 0
    return len(np.where(np.diff(np.sign(data)))[0]) / len(data)

def sample_entropy(data, prev_sampen=None):
    if len(data) <= 2 or np.any(np.isnan(data)) or np.std(data) == 0:
        return prev_sampen if prev_sampen is not None else 0
    return sampen.sampen2(data, mm=2, r=0.2 * np.std(data))[-1][1]

def theta_beta_ratio(data, fs=128, prev_value=None):
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    if np.any(np.isnan([theta, beta])) or beta == 0:
        return prev_value if prev_value is not None else 0
    return theta / (beta + 1e-10)

def channel_coherence(data, fs=128, prev_coherence=None):
    n_channels = data.shape[1]
    coherence_features = []
    nperseg = min(data.shape[0], 512)
    expected_length = 3 * (n_channels * (n_channels - 1) // 2)
    for band, freq_range in [('theta', (4, 8)), ('alpha', (8, 13)), ('beta', (13, 30))]:
        for i in range(n_channels):
            for j in range(i + 1, n_channels):
                f, coh = signal.coherence(data[:, i], data[:, j], fs=fs, nperseg=nperseg)
                freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
                coh_value = np.mean(coh[freq_mask]) if freq_mask.any() else 0
                if np.isnan(coh_value):
                    return prev_coherence if prev_coherence is not None else np.zeros(expected_length)
                coherence_features.append(coh_value)
    return np.array(coherence_features)

def channel_correlation(data, prev_correlation=None):
    n_channels = data.shape[1]
    corr_features = []
    expected_length = n_channels * (n_channels - 1) // 2
    for i in range(n_channels):
        for j in range(i + 1, n_channels):
            corr = np.corrcoef(data[:, i], data[:, j])[0, 1]
            if np.isnan(corr):
                return prev_correlation if prev_correlation is not None else np.zeros(expected_length)
            corr_features.append(corr)
    return np.array(corr_features)

def microstate_analysis(data, fs=128, n_states=4, prev_microstates=None):
    expected_length = n_states + n_states + n_states + n_states * n_states + 2
    gfp = np.std(data, axis=1)
    if np.any(np.isnan(gfp)):
        return prev_microstates if prev_microstates is not None else np.zeros(expected_length)
    peaks, _ = find_peaks(gfp, distance=fs//10)
    if len(peaks) < n_states:
        peak_data = data
    else:
        peak_data = data[peaks]
    kmeans = KMeans(n_clusters=n_states, random_state=42, n_init=10)
    kmeans.fit(peak_data)
    templates = kmeans.cluster_centers_
    distances = np.zeros((data.shape[0], n_states))
    for i, template in enumerate(templates):
        template_norm = np.linalg.norm(template)
        data_norm = np.linalg.norm(data, axis=1)
        distances[:, i] = np.abs(np.sum(data * template, axis=1) / (data_norm * template_norm + 1e-10))
    labels = np.argmax(distances, axis=1)
    durations = np.zeros(n_states)
    occurrences = np.zeros(n_states)
    gfp_peaks = gfp[peaks] if len(peaks) > 0 else gfp
    gfp_mean = np.mean(gfp_peaks)
    gfp_std = np.std(gfp_peaks)
    for state in range(n_states):
        state_indices = np.where(labels == state)[0]
        if len(state_indices) > 0:
            diff = np.diff(state_indices)
            breaks = np.where(diff > 1)[0]
            segment_lengths = np.diff(np.concatenate([[0], breaks + 1, [len(state_indices)]]))
            durations[state] = np.mean(segment_lengths) / fs * 1000 if len(segment_lengths) > 0 else 0
            occurrences[state] = len(breaks) + 1 if len(breaks) > 0 else 1 if len(state_indices) > 0 else 0
    coverage = np.bincount(labels, minlength=n_states).astype(float) / len(labels)
    transitions = np.zeros((n_states, n_states))
    for t in range(len(labels)-1):
        transitions[labels[t], labels[t+1]] += 1
    transition_sums = transitions.sum(axis=1, keepdims=True)
    transitions = transitions / (transition_sums + 1e-10)
    microstate_features = np.hstack([durations, occurrences, coverage, transitions.ravel(), [gfp_mean, gfp_std]])
    return microstate_features

def extract_features_single(sample, idx, prev_features=None):
    n_channels = sample.shape[1]
    normalized = normalize_data(sample)
    channel_features = []

    # Define feature indices for easier access (based on concatenation order)
    features_per_channel = 50  # From original concatenation
    total_per_channel = n_channels * features_per_channel
    prev_per_channel = prev_features[:total_per_channel] if prev_features is not None else None

    for ch in range(n_channels):
        channel_data = normalized[:, ch]
        offset = ch * features_per_channel
        prev_values = prev_per_channel[offset:offset + features_per_channel] if prev_per_channel is not None else None

        engagement = calculate_engagement_index(channel_data, prev_value=prev_values[0] if prev_values is not None else None)
        theta = calculate_psd(channel_data, freq_range=(4, 8))
        alpha = calculate_psd(channel_data, freq_range=(8, 13))
        beta = calculate_psd(channel_data, freq_range=(13, 30))
        pac = phase_amplitude_coupling(channel_data, prev_pac=prev_values[4:22] if prev_values is not None else None)
        sprint = calculate_sprint(channel_data, prev_value=prev_values[22] if prev_values is not None else None)
        wavelet = wavelet_features(channel_data, prev_wavelet=prev_values[23:41] if prev_values is not None else None)
        ll = line_length(channel_data, prev_value=prev_values[41] if prev_values is not None else None)
        he = hurst_exponent(channel_data, prev_value=prev_values[42] if prev_values is not None else None)
        zcr = zero_crossing_rate(channel_data, prev_zcr=prev_values[43] if prev_values is not None else None)
        hjorth = hjorth_parameters(channel_data, prev_hjorth=prev_values[44:47] if prev_values is not None else None)
        tbr = theta_beta_ratio(channel_data, prev_value=prev_values[47] if prev_values is not None else None)
        sentropy = sample_entropy(channel_data, prev_sampen=prev_values[48] if prev_values is not None else None)
        skewness = skew(channel_data) if not np.any(np.isnan(channel_data)) else (prev_values[49] if prev_values is not None else 0)
        kurt = kurtosis(channel_data) if not np.any(np.isnan(channel_data)) else (prev_values[50] if prev_values is not None else 0)

        ch_features = np.concatenate([
            [engagement], [theta], [alpha], [beta], pac, [sprint], wavelet,
            [ll], [he], [zcr], hjorth, [tbr], [sentropy], [skewness], [kurt]
        ])
        channel_features.append(ch_features)

    per_channel_features = np.array(channel_features).flatten()
    microstates = microstate_analysis(normalized, prev_microstates=prev_features[total_per_channel:total_per_channel + 54] if prev_features is not None else None)
    correlations = channel_correlation(normalized, prev_correlation=prev_features[total_per_channel + 54:total_per_channel + 54 + 171] if prev_features is not None else None)
    coherences = channel_coherence(normalized, prev_coherence=prev_features[total_per_channel + 54 + 171:] if prev_features is not None else None)
    
    sample_features = np.concatenate([per_channel_features, microstates, correlations, coherences])
    return sample_features

def extract_all_features(data, chunk_size=100):
    n_samples, n_samples_per_segment, n_channels = data.shape
    feature_matrix = []
    prev_features = None

    with tqdm(total=n_samples, desc="Extracting features", unit="segment") as pbar:
        for i in range(n_samples):
            sample_features = extract_features_single(data[i], i, prev_features)
            feature_matrix.append(sample_features)
            prev_features = sample_features
            pbar.update(1)
    feature_matrix = np.array(feature_matrix)
    print(f"Feature matrix shape: {feature_matrix.shape}")
    return feature_matrix

# Feature Selection with MO-DTW-AS-SHAP-XGRFE (unchanged)
def select_features(X, y, feature_names, min_window_size=50):
    smoteenn = SMOTEENN(random_state=42)
    X_balanced, y_balanced = smoteenn.fit_resample(X, y)
    print(f"After SMOTEENN: {X_balanced.shape[0]} samples, {np.bincount(y_balanced.astype(int))} classes")

    X_temp, X_test, y_temp, y_test = train_test_split(X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

    X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
    X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)
    X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

    xgb_model = xgb.XGBClassifier(random_state=42, n_estimators=100, use_label_encoder=False, eval_metric='logloss')
    xgb_model.fit(X_train, y_train)
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_train)

    def evaluate_metrics(model, X, y):
        y_pred = model.predict(X)
        sensitivity = recall_score(y, y_pred, pos_label=1, zero_division=0)
        specificity = recall_score(y, y_pred, pos_label=0, zero_division=0)
        f1 = f1_score(y, y_pred, zero_division=0)
        return sensitivity, specificity, f1
    
    base_sens, base_spec, base_f1 = evaluate_metrics(xgb_model, X_val, y_val)

    gfp = np.mean(X_train, axis=1)
    peaks, _ = find_peaks(gfp, distance=min_window_size, prominence=0.1 * np.max(gfp))
    if len(peaks) == 0:
        peaks = np.array([len(gfp) // 2])
    boundaries = np.concatenate([[0], peaks, [len(gfp)]])
    sub_windows = [(boundaries[i], boundaries[i+1]) for i in range(len(boundaries)-1)]

    ref_pattern_0 = np.mean(X_train[y_train == 0], axis=0)
    ref_pattern_1 = np.mean(X_train[y_train == 1], axis=0)

    n_features = X_train.shape[1]
    mo_shap_importance = np.zeros(n_features)
    dtw_weights_total = 0

    for start_idx, end_idx in sub_windows:
        if end_idx <= start_idx:
            continue
        sub_X = X_train[start_idx:end_idx]
        sub_shap = shap_values[start_idx:end_idx]
        sub_mean = np.mean(sub_X, axis=0)

        dtw_dist_0 = dtw.distance(sub_mean, ref_pattern_0)
        dtw_dist_1 = dtw.distance(sub_mean, ref_pattern_1)
        dtw_weight = 1 / (min(dtw_dist_0, dtw_dist_1) + 1e-10)

        sub_sens_impact = np.zeros(n_features)
        sub_spec_impact = np.zeros(n_features)
        sub_f1_impact = np.zeros(n_features)
        X_perturbed_full = X_train.copy()
        for f in range(n_features):
            X_perturbed_full[start_idx:end_idx, f] = np.random.permutation(sub_X[:, f])
            xgb_model.fit(X_perturbed_full, y_train)
            sens, spec, f1 = evaluate_metrics(xgb_model, X_val, y_val)
            sub_sens_impact[f] = max(base_sens - sens, 0)
            sub_spec_impact[f] = max(base_spec - spec, 0)
            sub_f1_impact[f] = max(base_f1 - f1, 0)

        mo_score = 0.4 * sub_sens_impact + 0.2 * sub_spec_impact + 0.4 * sub_f1_impact
        mo_shap_importance += mo_score * dtw_weight
        dtw_weights_total += dtw_weight

    if dtw_weights_total > 0:
        mo_shap_importance /= dtw_weights_total
    else:
        mo_shap_importance = np.ones(n_features) / n_features

    n_features_to_select = max(20, int(n_features * 0.25))
    ranking = np.argsort(mo_shap_importance)[::-1]
    selected_indices = ranking[:n_features_to_select]
    mask = np.zeros(n_features, dtype=bool)
    mask[selected_indices] = True

    X_train_selected = X_train[:, mask]
    X_val_selected = X_val[:, mask]
    X_test_selected = X_test[:, mask]

    selected_features = np.array(feature_names)[mask]

    shap_values_selected = explainer.shap_values(X_train_selected)
    plt.figure()
    shap.summary_plot(shap_values_selected, X_train_selected, feature_names=selected_features, plot_type="bar", max_display=20, show=False)
    plt.title("Mean SHAP Values of Top 20 Features (MO-DTW-AS-SHAP-XGRFE)")
    plt.savefig("mo_dtw_as_shap_xgrfe_top20.png")
    plt.close()

    return X_train_selected, X_val_selected, X_test_selected, y_train, y_val, y_test, selected_features, xgb_model, shap_values

# Model Evaluation (unchanged)
def evaluate_model(model, X, y, dataset_name):
    y_pred = model.predict(X)
    cm = confusion_matrix(y, y_pred)
    report = classification_report(y, y_pred, output_dict=True)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = report['weighted avg']['f1-score']
    print(f"\n{dataset_name} Metrics: Sensitivity={sensitivity:.4f}, Specificity={specificity:.4f}, F1={f1:.4f}")
    return sensitivity, specificity, f1

# KNNDTWClassifier (unchanged)
class KNNDTWClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self

    def predict(self, X):
        predictions = []
        for x in tqdm(X, desc="Predicting with KNN-DTW"):
            distances = [dtw.distance(x, x_train) for x_train in self.X_train]
            nearest_indices = np.argsort(distances)[:self.n_neighbors]
            nearest_labels = self.y_train[nearest_indices]
            prediction = np.bincount(nearest_labels.astype(int)).argmax()
            predictions.append(prediction)
        return np.array(predictions)

    def get_params(self, deep=True):
        return {"n_neighbors": self.n_neighbors}

    def set_params(self, **params):
        for param, value in params.items():
            setattr(self, param, value)
        return self

# Main Execution (unchanged)
def main():
    X, y = load_all_data()
    X_features = extract_all_features(X)

    n_channels = X.shape[2]
    n_samples_per_segment = X.shape[1]

    feature_types = [
        'engagement', 'theta', 'alpha', 'beta',
        'pac_theta_alpha_mean', 'pac_theta_alpha_std', 'pac_theta_alpha_max',
        'pac_theta_beta_mean', 'pac_theta_beta_std', 'pac_theta_beta_max',
        'pac_alpha_theta_mean', 'pac_alpha_theta_std', 'pac_alpha_theta_max',
        'pac_alpha_beta_mean', 'pac_alpha_beta_std', 'pac_alpha_beta_max',
        'pac_beta_theta_mean', 'pac_beta_theta_std', 'pac_beta_theta_max',
        'pac_beta_alpha_mean', 'pac_beta_alpha_std', 'pac_beta_alpha_max',
        'sprint',
        'wavelet_mean1', 'wavelet_std1', 'wavelet_max1',
        'wavelet_mean2', 'wavelet_std2', 'wavelet_max2',
        'wavelet_mean3', 'wavelet_std3', 'wavelet_max3',
        'wavelet_mean4', 'wavelet_std4', 'wavelet_max4',
        'wavelet_mean5', 'wavelet_std5', 'wavelet_max5',
        'wavelet_mean6', 'wavelet_std6', 'wavelet_max6',
        'line_length', 'hurst_exponent', 'zero_crossing_rate',
        'hjorth_activity', 'hjorth_mobility', 'hjorth_complexity',
        'theta_beta_ratio', 'sample_entropy', 'skewness', 'kurtosis'
    ]
    n_states = 4
    n_wavelet_scales = 6
    n_pac_features = 18

    per_channel_feature_count = (
        1 + 1 + 1 + 1 + n_pac_features + 1 + n_wavelet_scales * 3 +
        1 + 1 + 1 + 3 + 1 + 1 + 1 + 1
    )
    microstate_feature_count = n_states + n_states + n_states + n_states * n_states + 2
    correlation_feature_count = n_channels * (n_channels - 1) // 2
    coherence_feature_count = 3 * (n_channels * (n_channels - 1) // 2)

    expected_feature_count = (
        n_channels * per_channel_feature_count +
        microstate_feature_count +
        correlation_feature_count +
        coherence_feature_count
    )

    print(f"Expected feature count: {expected_feature_count}")
    if X_features.shape[1] != expected_feature_count:
        raise ValueError(f"Feature dimension mismatch: {X_features.shape[1]} vs {expected_feature_count}")

    feature_names = (
        [f"{ft}_ch{ch}" for ch in range(n_channels) for ft in feature_types] +
        [f'ms_duration_{i}' for i in range(n_states)] +
        [f'ms_occurrence_{i}' for i in range(n_states)] +
        [f'ms_coverage_{i}' for i in range(n_states)] +
        [f'ms_trans_{i}_{j}' for i in range(n_states) for j in range(n_states)] +
        ['gfp_mean', 'gfp_std'] +
        [f'corr_ch{i}_ch{j}' for i in range(n_channels) for j in range(i + 1, n_channels)] +
        [f'coh_theta_ch{i}_ch{j}' for i in range(n_channels) for j in range(i + 1, n_channels)] +
        [f'coh_alpha_ch{i}_ch{j}' for i in range(n_channels) for j in range(i + 1, n_channels)] +
        [f'coh_beta_ch{i}_ch{j}' for i in range(n_channels) for j in range(i + 1, n_channels)]
    )

    print("Selecting features with MO-DTW-AS-SHAP-XGRFE...")
    X_train_selected, X_val_selected, X_test_selected, y_train, y_val, y_test, selected_features, initial_model, initial_shap_values = select_features(X_features, y, feature_names)

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_results = []

    X_combined = np.vstack([X_train_selected, X_val_selected])
    y_combined = np.hstack([y_train, y_val])
    if X_combined.shape[0] != y_combined.shape[0]:
        raise ValueError(f"Dimension mismatch in CV split: X={X_combined.shape[0]}, y={y_combined.shape[0]}")

    for fold, (train_idx, val_idx) in enumerate(tqdm(cv.split(X_combined, y_combined), total=10, desc="Cross-validation folds")):
        X_train_fold = X_combined[train_idx]
        y_train_fold = y_combined[train_idx]
        X_val_fold = X_combined[val_idx]
        y_val_fold = y_combined[val_idx]

        cat_param_dist = {
            'iterations': [100, 200, 300],
            'depth': [4, 6, 8],
            'learning_rate': [0.01, 0.05, 0.1],
            'l2_leaf_reg': [1, 3, 5],
            'border_count': [32, 64, 128]
        }
        cat_base = CatBoostClassifier(random_state=42, verbose=0)
        cat_search = RandomizedSearchCV(cat_base, cat_param_dist, n_iter=20, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
        cat_search.fit(X_train_fold, y_train_fold)
        cat_model = cat_search.best_estimator_

        knn_dtw_model = KNNDTWClassifier(n_neighbors=5)
        knn_dtw_model.fit(X_train_fold, y_train_fold)

        gb_param_dist = {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.05, 0.1],
            'subsample': [0.6, 0.8, 1.0]
        }
        gb_base = GradientBoostingClassifier(random_state=42)
        gb_search = RandomizedSearchCV(gb_base, gb_param_dist, n_iter=20, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
        gb_search.fit(X_train_fold, y_train_fold)
        gb_model = gb_search.best_estimator_

        estimators = [('cat', cat_model), ('knn_dtw', knn_dtw_model), ('gb', gb_model)]
        stack_model = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(random_state=42), cv=5)
        stack_model.fit(X_train_fold, y_train_fold)

        cat_val_sens, cat_val_spec, cat_val_f1 = evaluate_model(cat_model, X_val_fold, y_val_fold, f"Fold {fold+1} CatBoost Val")
        stack_val_sens, stack_val_spec, stack_val_f1 = evaluate_model(stack_model, X_val_fold, y_val_fold, f"Fold {fold+1} Stack Val")
        cat_test_sens, cat_test_spec, cat_test_f1 = evaluate_model(cat_model, X_test_selected, y_test, f"Fold {fold+1} CatBoost Test")
        stack_test_sens, stack_test_spec, stack_test_f1 = evaluate_model(stack_model, X_test_selected, y_test, f"Fold {fold+1} Stack Test")

        fold_results.append({
            'fold': fold + 1,
            'cat_val': (cat_val_sens, cat_val_spec, cat_val_f1),
            'stack_val': (stack_val_sens, stack_val_spec, stack_val_f1),
            'cat_test': (cat_test_sens, cat_test_spec, cat_test_f1),
            'stack_test': (stack_test_sens, stack_test_spec, stack_test_f1)
        })

    for model in ['cat', 'stack']:
        val_sens = np.mean([r[f'{model}_val'][0] for r in fold_results])
        val_spec = np.mean([r[f'{model}_val'][1] for r in fold_results])
        val_f1 = np.mean([r[f'{model}_val'][2] for r in fold_results])
        test_sens = np.mean([r[f'{model}_test'][0] for r in fold_results])
        test_spec = np.mean([r[f'{model}_test'][1] for r in fold_results])
        test_f1 = np.mean([r[f'{model}_test'][2] for r in fold_results])
        print(f"\n{model.upper()} Model Average Metrics:")
        print(f"Validation - Sensitivity: {val_sens:.4f}, Specificity: {val_spec:.4f}, F1: {val_f1:.4f}")
        print(f"Test - Sensitivity: {test_sens:.4f}, Specificity: {test_spec:.4f}, F1: {test_f1:.4f}")

    print("\nComputing SHAP values for XGBoost (initial model)...")
    explainer = shap.TreeExplainer(initial_model)
    shap_values_test = explainer.shap_values(X_test_selected)
    plt.figure()
    shap.summary_plot(shap_values_test, X_test_selected, feature_names=selected_features, show=False)
    plt.title("SHAP Summary Plot (XGBoost - MO-DTW-AS-SHAP-XGRFE Features)")
    plt.savefig("shap_summary_xgboost_mo_dtw_as.png")
    plt.close()

    plt.figure()
    shap.summary_plot(initial_shap_values, X_train_selected, feature_names=selected_features, plot_type="bar", max_display=20, show=False)
    plt.title("SHAP Bar Plot (XGBoost - MO-DTW-AS-SHAP-XGRFE Features)")
    plt.savefig("shap_bar_xgboost_mo_dtw_as.png")
    plt.close()

    print("\nTop 10 Selected Features:")
    for i, feature in enumerate(selected_features[:min(10, len(selected_features))]):
        print(f"{i+1}. {feature}")
    print_memory_usage()

if __name__ == "__main__":
    main()

Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1: 100%|█| 30/30 [00:01<00:
Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part2/ADHD_part2: 100%|█| 31/31 [00:01<00:
Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part1/Control_part1: 100%|█| 30/30 [00:
Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part2: 100%|█| 30/30 [00:01<00:00, 29.5


Creating RawArray with float64 data, n_channels=19, n_times=4300288
    Range : 0 ... 4300287 =      0.000 ... 33595.992 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 0.5 Hz

IIR filter parameters
---------------------
Butterworth highpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 0.50 Hz: -6.02 dB

Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Fitting ICA took 357.8s.
Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 10 ICA components
    Projecting back using 19 PCA components
Data loading and preprocessing completed.


Extracting features:   0%|                                                   | 19/8399 [01:06<10:00:35,  4.30s/segment]

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
from scipy.io import loadmat
from scipy import signal
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.feature_selection import VarianceThreshold
import xgboost as xgb
import shap
from scipy.signal import hilbert, find_peaks, iirnotch
from sklearn.cluster import KMeans
from tqdm import tqdm
import psutil
from sklearn.metrics import confusion_matrix, classification_report, recall_score, f1_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
import mne
from dtaidistance import dtw
from imblearn.combine import SMOTEENN
from hurst import compute_Hc
import sampen
from sklearn.base import BaseEstimator, ClassifierMixin
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.cluster._kmeans")

def print_memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024 / 1024:.2f} MB")

# Preprocessing Functions
def load_and_segment_file(file_path, window_size=512, stride=256, expected_channels=19):
    mat_data = loadmat(file_path)
    var_names = [k for k in mat_data.keys() if not k.startswith('__')]
    if not var_names:
        raise ValueError(f"No non-metadata variables found in {file_path}")
    possible_var_names = ['data', 'eeg', 'signal', 'EEG', 'Data'] + var_names
    data = None
    for var_name in possible_var_names:
        if var_name in mat_data:
            data = mat_data[var_name]
            break
    if data is None:
        raise ValueError(f"No valid EEG data variable found in {file_path}")
    if data.ndim == 1:
        data = data.reshape(1, -1, 1)
    elif data.ndim == 2:
        data = data[np.newaxis, :, :]
    elif data.ndim == 3:
        pass
    else:
        raise ValueError(f"Unexpected data dimensions {data.ndim}")
    n_segments, n_samples, n_channels = data.shape
    if n_channels != expected_channels:
        if n_channels < expected_channels:
            data = np.pad(data, ((0, 0), (0, 0), (0, expected_channels - n_channels)), mode='constant')
        else:
            data = data[:, :, :expected_channels]
    max_abs_value = np.max(np.abs(data))
    if max_abs_value > 1000:
        data = data / 1000.0
    if max_abs_value > 100:
        scaling_factor = 100 / max_abs_value
        data = data * scaling_factor
    all_segments = []
    for seg in range(n_segments):
        seg_data = data[seg]
        seg_samples = seg_data.shape[0]
        if seg_samples < window_size:
            padded_data = np.pad(seg_data, ((0, window_size - seg_samples), (0, 0)), mode='constant')
            all_segments.append(padded_data[np.newaxis, :, :])
            continue
        n_seg_segments = ((seg_samples - window_size + stride - 1) // stride) + 1
        segments = np.zeros((n_seg_segments, window_size, n_channels))
        for i in range(n_seg_segments):
            start = i * stride
            end = min(start + window_size, seg_samples)
            segment_length = end - start
            if segment_length == window_size:
                segments[i] = seg_data[start:end, :]
            else:
                segments[i] = np.pad(seg_data[start:end, :], ((0, window_size - segment_length), (0, 0)), mode='constant')
        all_segments.append(segments)
    if not all_segments:
        return None
    return np.concatenate(all_segments, axis=0)

def load_from_folder(folder_path, window_size=512, stride=256, expected_channels=19):
    if not os.path.exists(folder_path):
        return None
    mat_files = [f for f in os.listdir(folder_path) if f.endswith('.mat')]
    if not mat_files:
        return None
    all_segments = []
    for file in tqdm(mat_files, desc=f"Loading files from {folder_path}"):
        file_path = os.path.join(folder_path, file)
        segments = load_and_segment_file(file_path, window_size, stride, expected_channels)
        if segments is not None:
            all_segments.append(segments)
    if not all_segments:
        return None
    return np.concatenate(all_segments, axis=0), mat_files

def load_all_data(window_size=512, stride=256, expected_channels=19):
    adhd_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1"
    adhd_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part2/ADHD_part2"
    control_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part1/Control_part1"
    control_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part2"
    
    # Load data and keep track of patient files
    adhd_segments1, adhd_files1 = load_from_folder(adhd_path1, window_size, stride, expected_channels)
    adhd_segments2, adhd_files2 = load_from_folder(adhd_path2, window_size, stride, expected_channels)
    adhd_segments = []
    if adhd_segments1 is not None:
        adhd_segments.append(adhd_segments1)
    if adhd_segments2 is not None:
        adhd_segments.append(adhd_segments2)
    if not adhd_segments:
        raise ValueError("Failed to load any ADHD data")
    adhd_segments = np.concatenate(adhd_segments, axis=0)
    adhd_files = adhd_files1 + adhd_files2

    control_segment1, control_files1 = load_from_folder(control_path1, window_size, stride, expected_channels)
    control_segment2, control_files2 = load_from_folder(control_path2, window_size, stride, expected_channels)
    control_segments = []
    if control_segment1 is not None:
        control_segments.append(control_segment1)
    if control_segment2 is not None:
        control_segments.append(control_segment2)
    if not control_segments:
        raise ValueError("Failed to load any Control data")
    control_segments = np.concatenate(control_segments, axis=0)
    control_files = control_files1 + control_files2

    # Assign labels based on directory
    y_adhd = np.ones(adhd_segments.shape[0])
    y_control = np.zeros(control_segments.shape[0])
    X = np.concatenate([adhd_segments, control_segments], axis=0)
    y = np.concatenate([y_adhd, y_control])

    # Create patient IDs based on filenames (v<patient_number>)
    adhd_segment_counts = []
    adhd_patient_ids = []
    valid_adhd_files = []
    for f in adhd_files:
        path = os.path.join(adhd_path1, f) if f in adhd_files1 else os.path.join(adhd_path2, f)
        segments = load_and_segment_file(path)
        count = segments.shape[0] if segments is not None else 0
        if count > 0:
            adhd_segment_counts.append(count)
            valid_adhd_files.append(f)
            # Extract patient number from filename (e.g., "v001.mat" -> "v001")
            patient_number = f.split('.')[0]  # Remove '.mat'
            adhd_patient_ids.extend([patient_number] * count)

    control_segment_counts = []
    control_patient_ids = []
    valid_control_files = []
    for f in control_files:
        path = os.path.join(control_path1, f) if f in control_files1 else os.path.join(control_path2, f)
        segments = load_and_segment_file(path)
        count = segments.shape[0] if segments is not None else 0
        if count > 0:
            control_segment_counts.append(count)
            valid_control_files.append(f)
            # Extract patient number from filename (e.g., "v001.mat" -> "v001")
            patient_number = f.split('.')[0]  # Remove '.mat'
            control_patient_ids.extend([patient_number] * count)

    # Combine patient IDs
    patient_ids = adhd_patient_ids + control_patient_ids

    if len(patient_ids) != X.shape[0]:
        raise ValueError(f"Patient IDs length ({len(patient_ids)}) does not match number of segments ({X.shape[0]})")

    print(f"Loaded {X.shape[0]} segments from {len(valid_adhd_files) + len(valid_control_files)} patients")
    return X, y, patient_ids

    

def notch_filter(data, fs=128, freqs=[50, 60], Q=30):
    filtered_data = data.copy()
    for freq in freqs:
        b, a = signal.iirnotch(freq / (fs / 2), Q)
        filtered_data = signal.filtfilt(b, a, filtered_data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def clip_spikes(data, threshold=500):
    clipped_data = data.copy()
    clipped_data[np.abs(clipped_data) > threshold] = np.sign(clipped_data[np.abs(clipped_data) > threshold]) * threshold
    fp2_threshold = 30
    clipped_data[:, :, 1] = np.where(np.abs(clipped_data[:, :, 1]) > fp2_threshold,
                                     np.sign(clipped_data[:, :, 1]) * fp2_threshold,
                                     clipped_data[:, :, 1])
    return clipped_data

def butterworth_filter(data, fs=128, lowcut=1, highcut=45, order=4):
    nyquist = fs / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = signal.butter(order, [low, high], btype='band')
    filtered_data = signal.filtfilt(b, a, data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def apply_ica(data, fs=128, n_components=None):
    if data.ndim == 2:
        data = data[np.newaxis, :, :]
    elif data.ndim != 3:
        raise ValueError(f"Expected 3D data, got {data.ndim}D")
    n_segments, n_samples, n_channels = data.shape
    if n_channels != 19:
        raise ValueError(f"Expected 19 channels, got {n_channels}")
    if n_components is None or n_components > n_channels:
        n_components = min(19, n_channels)
    total_samples = n_segments * n_samples
    concatenated_data = data.reshape(total_samples, n_channels)
    ch_names = [f"EEG {i}" for i in range(n_channels)]
    montage_labels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
    info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types="eeg")
    raw = mne.io.RawArray(concatenated_data.T, info)
    channel_mapping = {f"EEG {i}": montage_labels[i] for i in range(n_channels)}
    raw.rename_channels(channel_mapping)
    try:
        theta_values = [-18, 18, -54, -39, 0, 39, 54, -90, -90, 90, 90, 90, -126, -141, 180, 141, 126, -162, 162]
        radius_values_cm = [51.1, 51.1, 51.1, 33.3, 25.6, 33.3, 51.1, 51.1, 25.6, 0, 25.6, 51.1, 51.1, 33.3, 25.6, 33.3, 51.1, 51.1, 51.1]
        radius_values_m = [r / 100 for r in radius_values_cm]
        ch_pos = {label: [radius * np.cos(np.deg2rad(theta)), radius * np.sin(np.deg2rad(theta)), 0.0]
                  for label, theta, radius in zip(montage_labels, theta_values, radius_values_m)}
        montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='head')
        raw.set_montage(montage)
    except Exception:
        montage = mne.channels.make_standard_montage('standard_1020')
        raw.set_montage(montage)
    raw.filter(l_freq=0.5, h_freq=None, method='iir')
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=42, max_iter=500)
    ica.fit(raw)
    ica_sources = ica.get_sources(raw).get_data().T
    kurtosis_values = np.array([kurtosis(ica_sources[:, i]) for i in range(n_components)])
    variance_values = np.var(ica_sources, axis=0)
    peak_amplitude = np.max(np.abs(ica_sources), axis=0)
    fp2_weight = np.max(np.abs(ica_sources[total_samples // 2:total_samples // 2 + n_samples, 1])) / np.max(peak_amplitude) if n_channels > 1 else 0
    kurtosis_norm = (kurtosis_values - kurtosis_values.min()) / (kurtosis_values.max() - kurtosis_values.min() + 1e-10)
    variance_norm = (variance_values - variance_values.min()) / (variance_values.max() - variance_values.min() + 1e-10)
    peak_norm = (peak_amplitude - peak_amplitude.min()) / (peak_amplitude.max() - peak_amplitude.min() + 1e-10)
    combined_score = 0.45 * kurtosis_norm + 0.25 * variance_norm + 0.2 * peak_norm + 0.1 * fp2_weight
    top_artifact_indices = np.argsort(combined_score)[-min(10, n_components):]
    ica.exclude = top_artifact_indices.tolist()
    raw_clean = ica.apply(raw.copy())
    cleaned_data_raw = raw_clean.get_data()
    cleaned_data = cleaned_data_raw.T.reshape(n_segments, n_samples, n_channels)
    return cleaned_data, data

# Feature Extraction Functions
def normalize_data(data):
    scaler = StandardScaler()
    return scaler.fit_transform(data)

def calculate_engagement_index(data, fs=128, prev_value=None):
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    alpha = calculate_psd(data, fs, freq_range=(8, 13))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    if np.any(np.isnan([theta, alpha, beta])) or beta == 0:
        return prev_value if prev_value is not None else 0
    return np.mean((theta + alpha) / (beta + 1e-10))

def calculate_psd(data, fs=128, freq_range=None):
    nperseg = min(len(data), 512)
    f, psd = signal.welch(data, fs=fs, nperseg=nperseg, axis=0)
    if freq_range:
        freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
        return np.mean(psd[freq_mask], axis=0) if freq_mask.any() else 0
    return psd.mean(axis=0)

def phase_amplitude_coupling(data, fs=128, prev_pac=None):
    theta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 4, 8)[0, :, 0]
    alpha = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 8, 13)[0, :, 0]
    beta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 13, 30)[0, :, 0]
    if np.any(np.isnan([theta, alpha, beta])) or np.std([theta, alpha, beta]) == 0:
        return prev_pac if prev_pac is not None else np.zeros(3)
    theta_phase = np.angle(hilbert(theta))
    alpha_phase = np.angle(hilbert(alpha))
    beta_phase = np.angle(hilbert(beta))
    alpha_amp = np.abs(hilbert(alpha))
    beta_amp = np.abs(hilbert(beta))
    theta_amp = np.abs(hilbert(theta))
    pac_theta_alpha = np.abs(np.mean(alpha_amp * np.exp(1j * theta_phase)))
    pac_theta_beta = np.abs(np.mean(beta_amp * np.exp(1j * theta_phase)))
    pac_alpha_beta = np.abs(np.mean(beta_amp * np.exp(1j * alpha_phase)))
    return np.array([pac_theta_alpha, pac_theta_beta, pac_alpha_beta])

def calculate_sprint(data, fs=128, prev_value=None):
    nperseg = min(len(data), 512)
    noverlap = nperseg // 2
    f, t, Zxx = signal.stft(data, fs=fs, nperseg=nperseg, noverlap=noverlap)
    power = np.abs(Zxx)**2
    if np.any(np.isnan(power)):
        return prev_value if prev_value is not None else 0
    return np.mean(power)

def wavelet_features(data, prev_wavelet=None):
    scales = np.arange(1, 4)  # Reduced to 3 scales
    coeffs, _ = pywt.cwt(data, scales, 'mexh', axis=0)
    if np.any(np.isnan(coeffs)):
        return prev_wavelet if prev_wavelet is not None else np.zeros(6)
    features = []
    for c in coeffs:
        features.extend([np.mean(np.abs(c)), np.max(np.abs(c))])  # Only mean and max
    return np.array(features)

def hjorth_parameters(data, prev_hjorth=None):
    diff1 = np.diff(data)
    diff2 = np.diff(diff1)
    activity = np.var(data)
    if np.any(np.isnan([activity, np.var(diff1), np.var(diff2)])) or activity == 0:
        return prev_hjorth if prev_hjorth is not None else np.zeros(3)
    mobility = np.sqrt(np.var(diff1) / activity)
    complexity = np.sqrt(np.var(diff2) / np.var(diff1)) / mobility if mobility > 0 else 0
    return np.array([activity, mobility, complexity])

def line_length(data, prev_value=None):
    diff = np.diff(data)
    if np.any(np.isnan(diff)):
        return prev_value if prev_value is not None else 0
    return np.sum(np.abs(diff))

def hurst_exponent(data, prev_value=None):
    if len(data) < 10 or np.any(np.isnan(data)) or np.std(data) == 0:
        return prev_value if prev_value is not None else 0.5
    H, _, _ = compute_Hc(data, kind='change', simplified=True)
    return H

def zero_crossing_rate(data, prev_zcr=None):
    if len(data) <= 1 or np.any(np.isnan(data)) or np.std(data) == 0:
        return prev_zcr if prev_zcr is not None else 0
    return len(np.where(np.diff(np.sign(data)))[0]) / len(data)

def sample_entropy(data, prev_sampen=None):
    if len(data) <= 2 or np.any(np.isnan(data)) or np.std(data) == 0:
        return prev_sampen if prev_sampen is not None else 0
    return sampen.sampen2(data, mm=2, r=0.2 * np.std(data))[-1][1]

def theta_beta_ratio(data, fs=128, prev_value=None):
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    if np.any(np.isnan([theta, beta])) or beta == 0:
        return prev_value if prev_value is not None else 0
    return theta / (beta + 1e-10)

def channel_coherence(data, fs=128, prev_coherence=None):
    n_channels = data.shape[1]
    coherence_features = []
    nperseg = min(data.shape[0], 512)
    expected_length = 3 * (n_channels * (n_channels - 1) // 2)
    for band, freq_range in [('theta', (4, 8)), ('alpha', (8, 13)), ('beta', (13, 30))]:
        for i in range(n_channels):
            for j in range(i + 1, n_channels):
                f, coh = signal.coherence(data[:, i], data[:, j], fs=fs, nperseg=nperseg)
                freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
                coh_value = np.mean(coh[freq_mask]) if freq_mask.any() else 0
                if np.isnan(coh_value):
                    return prev_coherence if prev_coherence is not None else np.zeros(expected_length)
                coherence_features.append(coh_value)
    return np.array(coherence_features)

def channel_correlation(data, prev_correlation=None):
    n_channels = data.shape[1]
    corr_features = []
    expected_length = n_channels * (n_channels - 1) // 2
    for i in range(n_channels):
        for j in range(i + 1, n_channels):
            corr = np.corrcoef(data[:, i], data[:, j])[0, 1]
            if np.isnan(corr):
                return prev_correlation if prev_correlation is not None else np.zeros(expected_length)
            corr_features.append(corr)
    return np.array(corr_features)

def microstate_analysis(data, fs=128, n_states=4, prev_microstates=None):
    expected_length = n_states + n_states + 2  # durations, coverage, gfp_mean, gfp_std
    gfp = np.std(data, axis=1)
    if np.any(np.isnan(gfp)):
        return prev_microstates if prev_microstates is not None else np.zeros(expected_length)
    peaks, _ = find_peaks(gfp, distance=fs//10)
    if len(peaks) < n_states:
        peak_data = data
    else:
        peak_data = data[peaks]
    kmeans = KMeans(n_clusters=n_states, random_state=42, n_init=10)
    kmeans.fit(peak_data)
    templates = kmeans.cluster_centers_
    distances = np.zeros((data.shape[0], n_states))
    for i, template in enumerate(templates):
        template_norm = np.linalg.norm(template)
        data_norm = np.linalg.norm(data, axis=1)
        distances[:, i] = np.abs(np.sum(data * template, axis=1) / (data_norm * template_norm + 1e-10))
    labels = np.argmax(distances, axis=1)
    durations = np.zeros(n_states)
    gfp_peaks = gfp[peaks] if len(peaks) > 0 else gfp
    gfp_mean = np.mean(gfp_peaks)
    gfp_std = np.std(gfp_peaks)
    for state in range(n_states):
        state_indices = np.where(labels == state)[0]
        if len(state_indices) > 0:
            diff = np.diff(state_indices)
            breaks = np.where(diff > 1)[0]
            segment_lengths = np.diff(np.concatenate([[0], breaks + 1, [len(state_indices)]]))
            durations[state] = np.mean(segment_lengths) / fs * 1000 if len(segment_lengths) > 0 else 0
    coverage = np.bincount(labels, minlength=n_states).astype(float) / len(labels)
    microstate_features = np.hstack([durations, coverage, [gfp_mean, gfp_std]])
    return microstate_features
       
def extract_features_single(sample, idx, prev_features=None):
    n_channels = sample.shape[1]
    normalized = np.zeros_like(sample)
    for ch in range(n_channels):
        channel_data = sample[:, ch]
        if np.std(channel_data) > 0:
            normalized[:, ch] = (channel_data - np.mean(channel_data)) / np.std(channel_data)
        else:
            normalized[:, ch] = channel_data
    channel_features = []

    features_per_channel = 14  # engagement, theta, alpha, beta, 3 PAC, 6 wavelet, theta_beta_ratio
    total_per_channel = n_channels * features_per_channel
    microstate_len = 10  # 4 durations, 4 coverages, gfp_mean, gfp_std
    corr_len = (n_channels * (n_channels - 1)) // 2
    coh_len = 3 * corr_len  # 3 frequency bands
    total_prev_len = total_per_channel + microstate_len + corr_len + coh_len

    if prev_features is not None and len(prev_features) != total_prev_len:
        raise ValueError(f"Expected prev_features length {total_prev_len}, got {len(prev_features)}")

    for ch in range(n_channels):
        channel_data = normalized[:, ch]
        offset = ch * features_per_channel
        prev_values = prev_features[offset:offset + features_per_channel] if prev_features is not None else None

        engagement = calculate_engagement_index(channel_data, prev_value=prev_values[0] if prev_values is not None else None)
        theta = calculate_psd(channel_data, freq_range=(4, 8))
        alpha = calculate_psd(channel_data, freq_range=(8, 13))
        beta = calculate_psd(channel_data, freq_range=(13, 30))
        pac = phase_amplitude_coupling(channel_data, prev_pac=prev_values[4:7] if prev_values is not None else None)
        wavelet = wavelet_features(channel_data, prev_wavelet=prev_values[7:13] if prev_values is not None else None)
        tbr = theta_beta_ratio(channel_data, prev_value=prev_values[13] if prev_values is not None else None)

        ch_features = np.concatenate([
            [engagement], [theta], [alpha], [beta], pac, wavelet, [tbr]
        ])
        channel_features.append(ch_features)

    per_channel_features = np.array(channel_features).flatten()
    microstates = microstate_analysis(normalized, prev_microstates=prev_features[total_per_channel:total_per_channel + microstate_len] if prev_features is not None else None)
    correlations = channel_correlation(normalized, prev_correlation=prev_features[total_per_channel + microstate_len:total_per_channel + microstate_len + corr_len] if prev_features is not None else None)
    coherences = channel_coherence(normalized, prev_coherence=prev_features[total_per_channel + microstate_len + corr_len:] if prev_features is not None else None)
    
    sample_features = np.concatenate([per_channel_features, microstates, correlations, coherences])
    return sample_features

       
    
def extract_all_features(data, chunk_size=100):
    n_samples, n_samples_per_segment, n_channels = data.shape
    feature_matrix = []
    prev_features = None

    with tqdm(total=n_samples, desc="Extracting features", unit="segment") as pbar:
        for i in range(n_samples):
            sample_features = extract_features_single(data[i], i, prev_features)
            feature_matrix.append(sample_features)
            prev_features = sample_features
            pbar.update(1)
    feature_matrix = np.array(feature_matrix)
    print(f"Feature matrix shape: {feature_matrix.shape}")
    return feature_matrix

# Feature Selection with MO-DTW-AS-SHAP-XGRFE
def remove_highly_correlated_features(X, feature_names, threshold=0.85):
    corr_matrix = np.corrcoef(X.T)
    to_remove = set()
    for i in range(len(corr_matrix)):
        for j in range(i + 1, len(corr_matrix)):
            if abs(corr_matrix[i, j]) > threshold:
                to_remove.add(j)
    keep_indices = [i for i in range(X.shape[1]) if i not in to_remove]
    return X[:, keep_indices], np.array(feature_names)[keep_indices].tolist()

def select_features(X, y, feature_names, min_window_size=50):
    outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    selected_features_list = []
    X_test_selected_list = []
    y_test_list = []
    X_train_val_selected_list = []
    y_train_val_list = []

    # Compute class weights based on class distribution using the input y
    n_negative = np.sum(y == 0)
    n_positive = np.sum(y == 1)
    class_weights = {0: (n_positive + n_negative) / (2 * n_negative), 1: (n_positive + n_negative) / (2 * n_positive)}
    print(f"Computed class weights: {class_weights}")

    for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        X_train_val, X_test_fold = X[train_val_idx], X[test_idx]
        y_train_val, y_test_fold = y[train_val_idx], y[test_idx]

        scaler = StandardScaler()
        X_train_val = scaler.fit_transform(X_train_val)
        X_test_fold = scaler.transform(X_test_fold)
        X_train_val = np.nan_to_num(X_train_val, nan=0.0, posinf=0.0, neginf=0.0)
        X_test_fold = np.nan_to_num(X_test_fold, nan=0.0, posinf=0.0, neginf=0.0)

        # Apply SMOTEENN to balance the training data
        smote_enn = SMOTEENN(random_state=42, sampling_strategy='auto')
        X_train_val_balanced, y_train_val_balanced = smote_enn.fit_resample(X_train_val, y_train_val)
        # Fix for previous TypeError
        y_train_val_balanced = np.rint(y_train_val_balanced).astype(np.int64)
        if not np.all(np.isin(y_train_val_balanced, [0, 1])):
            raise ValueError(f"Non-binary labels found after SMOTEENN: {np.unique(y_train_val_balanced)}")
        print(f"Fold {outer_fold+1} After SMOTEENN: {X_train_val_balanced.shape[0]} samples, Class distribution: {np.bincount(y_train_val_balanced)}")

        selector = VarianceThreshold(threshold=0.01)
        X_train_val_reduced = selector.fit_transform(X_train_val_balanced)
        X_test_fold_reduced = selector.transform(X_test_fold)
        mask = selector.get_support()
        feature_names_reduced = np.array(feature_names)[mask].tolist()
        print(f"Fold {outer_fold+1} After Variance Thresholding: {X_train_val_reduced.shape[1]} features remaining")

        # Correlation filtering
        X_train_val_reduced, feature_names_reduced = remove_highly_correlated_features(X_train_val_reduced, feature_names_reduced, threshold=0.85)
        kept_indices = [i for i, fname in enumerate(feature_names) if fname in feature_names_reduced]
        X_test_fold_reduced = X_test_fold[:, kept_indices]
        feature_indices = [feature_names.index(fname) for fname in feature_names_reduced]
        X_test_fold_reduced = X_test_fold_reduced[:, [kept_indices.index(idx) for idx in feature_indices]]
        print(f"Fold {outer_fold+1} After Correlation Filtering: {X_train_val_reduced.shape[1]} features remaining")

        # MO-DTW-AS-SHAP-XGRFE Feature Selection with nested CV
        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        xgb_model = xgb.XGBClassifier(
            random_state=42,
            n_estimators=20,
            eval_metric='logloss',
            scale_pos_weight=class_weights[1] / class_weights[0],
            max_depth=1,
            min_child_weight=10,
            gamma=5.0,
            reg_alpha=2.0,
            reg_lambda=3.0
        )

        sensitivities, specificities, f1_scores, accuracies = [], [], [], []
        best_thresholds = []
        for fold, (train_idx, val_idx) in enumerate(inner_cv.split(X_train_val_reduced, y_train_val_balanced)):
            X_train, X_val = X_train_val_reduced[train_idx], X_train_val_reduced[val_idx]
            y_train, y_val = y_train_val_balanced[train_idx], y_train_val_balanced[val_idx]
            xgb_model.fit(X_train, y_train)

            # Threshold tuning
            y_scores = xgb_model.predict_proba(X_val)[:, 1]
            thresholds = np.arange(0.3, 0.7, 0.05)
            best_f1 = 0
            best_threshold = 0.5
            for thresh in thresholds:
                y_pred = (y_scores > thresh).astype(int)
                f1 = f1_score(y_val, y_pred, zero_division=0)
                if f1 > best_f1:
                    best_f1 = f1
                    best_threshold = thresh
            best_thresholds.append(best_threshold)

            sens, spec, f1, acc, _ = evaluate_metrics(xgb_model, X_val, y_val, threshold=best_threshold)
            print(f"Fold {outer_fold+1}.{fold+1} Initial XGBoost: Sensitivity={sens:.4f}, Specificity={spec:.4f}, F1={f1:.4f}, Accuracy={acc:.4f}")
            sensitivities.append(sens)
            specificities.append(spec)
            f1_scores.append(f1)
            accuracies.append(acc)

        avg_threshold = np.mean(best_thresholds)
        print(f"Fold {outer_fold+1} Initial XGBoost CV: Sensitivity={np.mean(sensitivities):.4f}, Specificity={np.mean(specificities):.4f}, F1={np.mean(f1_scores):.4f}, Accuracy={np.mean(accuracies):.4f}, Avg Threshold={avg_threshold:.2f}")

        # MO-DTW-AS-SHAP-XGRFE
        X_train_mo, X_val_mo, y_train_mo, y_val_mo = train_test_split(X_train_val_reduced, y_train_val_balanced, test_size=0.2, stratify=y_train_val_balanced, random_state=42)
        xgb_model.fit(X_train_mo, y_train_mo)
        explainer = shap.TreeExplainer(xgb_model)
        shap_values = explainer.shap_values(X_train_mo)
        shap_importance = np.abs(shap_values).mean(axis=0)

        # DTW-based sub-window analysis with error handling
        dtw_scores = []
        for feature_idx in range(X_train_val_reduced.shape[1]):
            feature_series = X_train_val_reduced[:, feature_idx]
            if np.any(np.isnan(feature_series)) or np.std(feature_series) == 0:
                dtw_scores.append(0)
                continue
            sub_windows = [feature_series[i:i+min_window_size] for i in range(0, len(feature_series)-min_window_size, min_window_size)]
            if len(sub_windows) < 2:
                dtw_scores.append(0)
                continue
            try:
                dtw_sum = 0
                count = 0
                for i in range(len(sub_windows)):
                    for j in range(i+1, len(sub_windows)):
                        dist = dtw.distance(sub_windows[i], sub_windows[j])
                        if np.isnan(dist):
                            continue
                        dtw_sum += dist
                        count += 1
                dtw_scores.append(dtw_sum / count if count > 0 else 0)
            except Exception as e:
                print(f"Error in DTW computation for feature {feature_idx}: {e}")
                dtw_scores.append(0)
        dtw_scores = np.array(dtw_scores)

        # Use all metrics to adjust SHAP scores
        avg_sensitivity = np.mean(sensitivities)
        avg_specificity = np.mean(specificities)
        avg_f1 = np.mean(f1_scores)
        avg_accuracy = np.mean(accuracies)

        # Combine metrics into an array
        metrics = np.array([avg_sensitivity, avg_specificity, avg_f1, avg_accuracy])
        print(f"Fold {outer_fold+1} Metrics: Sensitivity={avg_sensitivity:.4f}, Specificity={avg_specificity:.4f}, F1={avg_f1:.4f}, Accuracy={avg_accuracy:.4f}")

        # Normalize metrics to [0, 1] based on their range
        metrics_norm = (metrics - metrics.min()) / (metrics.max() - metrics.min() + 1e-10)

        # Compute weights inversely proportional to normalized metrics
        metric_weights = 1 - metrics_norm  # Inverse: lower metric -> higher weight
        metric_weights = metric_weights / metric_weights.sum()  # Normalize weights to sum to 1
        print(f"Fold {outer_fold+1} Metric Weights: Sensitivity={metric_weights[0]:.4f}, Specificity={metric_weights[1]:.4f}, F1={metric_weights[2]:.4f}, Accuracy={metric_weights[3]:.4f}")

        # Normalize SHAP and DTW scores
        shap_norm = (shap_importance - shap_importance.min()) / (shap_importance.max() - shap_importance.min() + 1e-10)
        dtw_norm = (dtw_scores - dtw_scores.min()) / (dtw_scores.max() - dtw_scores.min() + 1e-10)

        # Adjust SHAP scores by weighting with metric weights
        balanced_weight = np.mean(metric_weights)  # Average weight to scale SHAP scores
        adjusted_shap_norm = shap_norm * (1 + balanced_weight)  # Amplify SHAP scores based on balance

        # Combine scores with multi-objective weighting
        mo_score = 0.5 * adjusted_shap_norm + 0.3 * dtw_norm
        print(f"Fold {outer_fold+1} MO Score Range: min={mo_score.min():.4f}, max={mo_score.max():.4f}")

        # Diversity penalty with balanced feature selection
        feature_types = []
        for fname in feature_names_reduced:
            if 'ch' in fname:
                feature_types.append('channel_feature')
            elif 'corr' in fname:
                feature_types.append('correlation')
            elif 'coh' in fname:
                feature_types.append('coherence')
            elif 'gfp' in fname or 'microstate' in fname:
                feature_types.append('microstate')
            else:
                feature_types.append('other')
        feature_types = np.array(feature_types)

        diversity_penalty = 0.6
        type_counts = {t: 0 for t in set(feature_types)}
        selected_indices = []
        n_features_to_select = max(1, int(0.20 * X_train_val_reduced.shape[1]))  
        for _ in range(n_features_to_select):
            adjusted_scores = mo_score.copy()
            for idx in selected_indices:
                if idx in range(len(feature_types)):
                    type_counts[feature_types[idx]] += 1
            for idx in range(len(adjusted_scores)):
                if idx in selected_indices:
                    adjusted_scores[idx] = -np.inf
                else:
                    type_count = type_counts[feature_types[idx]]
                    adjusted_scores[idx] -= diversity_penalty * type_count
            best_idx = np.argmax(adjusted_scores)
            selected_indices.append(best_idx)

        # Select features
        selected_features = [feature_names_reduced[idx] for idx in selected_indices]
        selected_indices = [list(feature_names_reduced).index(f) for f in selected_features]
        X_train_val_selected = X_train_val_reduced[:, selected_indices]
        X_test_selected = X_test_fold_reduced[:, selected_indices]

        # Distribution of selected feature types
        selected_types = [feature_types[idx] for idx in selected_indices]
        type_distribution = {t: selected_types.count(t) for t in set(selected_types)}
        print("\nDistribution of Selected Feature Types:")
        for t, count in type_distribution.items():
            print(f"{t}: {count} features")

        selected_features_list.append(selected_features)
        X_test_selected_list.append(X_test_selected)
        y_test_list.append(y_test_fold)
        X_train_val_selected_list.append(X_train_val_selected)
        y_train_val_list.append(y_train_val)

    return selected_features_list, X_train_val_selected_list, y_train_val_list, X_test_selected_list, y_test_list
        
# Model Evaluation
def evaluate_metrics(model, X, y, threshold=0.5):
    if hasattr(model, 'predict_proba'):
        y_scores = model.predict_proba(X)[:, 1]
        y_pred = (y_scores > threshold).astype(int)
    else:
        y_pred = model.predict(X)
    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = f1_score(y, y_pred, zero_division=0)
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    return sensitivity, specificity, f1, accuracy, y_pred

# KNNDTWClassifier
class KNNDTWClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self

    def predict(self, X):
        predictions = []
        for x in tqdm(X, desc="Predicting with KNN-DTW"):
            distances = [dtw.distance(x, x_train) for x_train in self.X_train]
            nearest_indices = np.argsort(distances)[:self.n_neighbors]
            nearest_labels = self.y_train[nearest_indices]
            prediction = np.bincount(nearest_labels.astype(int)).argmax()
            predictions.append(prediction)
        return np.array(predictions)

    def get_params(self, deep=True):
        return {"n_neighbors": self.n_neighbors}

    def set_params(self, **params):
        for param, value in params.items():
            setattr(self, param, value)
        return self

# Main Execution
def main():
    # Load data with patient IDs
    window_size = 512
    stride = 256
    expected_channels = 19
    X, y, patient_ids = load_all_data(window_size=window_size, stride=stride, expected_channels=expected_channels)
    
    # Patient-level splitting
    unique_patients = np.unique(patient_ids)

    # Map each patient to their label by checking the y values of their segments
    patient_labels = []
    for patient in unique_patients:
        # Get indices of segments belonging to this patient
        patient_indices = [i for i, pid in enumerate(patient_ids) if pid == patient]
        # Get the label of this patient (all segments for a patient should have the same label)
        patient_label = y[patient_indices[0]]  # Take the label of the first segment
        patient_labels.append(patient_label)
    patient_labels = np.array(patient_labels)

    # Split into train+val and test
    train_val_patients, test_patients = train_test_split(unique_patients, test_size=0.2, stratify=patient_labels, random_state=42)

    # Compute labels for train_val_patients for the second split
    train_val_labels = []
    for patient in train_val_patients:
        patient_indices = [i for i, pid in enumerate(patient_ids) if pid == patient]
        patient_label = y[patient_indices[0]]
        train_val_labels.append(patient_label)

    # Split train_val into train and val
    train_patients, val_patients = train_test_split(train_val_patients, test_size=0.25, stratify=train_val_labels, random_state=42)

    train_idx = [i for i, pid in enumerate(patient_ids) if pid in train_patients]
    val_idx = [i for i, pid in enumerate(patient_ids) if pid in val_patients]
    test_idx = [i for i, pid in enumerate(patient_ids) if pid in test_patients]

    X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
    y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

    # Preprocess each split independently
    print("Preprocessing training data...")
    X_train = notch_filter(X_train)
    X_train = clip_spikes(X_train)
    X_train = butterworth_filter(X_train)
    X_train_cleaned, X_train_raw = apply_ica(X_train)

    print("Preprocessing validation data...")
    X_val = notch_filter(X_val)
    X_val = clip_spikes(X_val)
    X_val = butterworth_filter(X_val)
    X_val_cleaned, X_val_raw = apply_ica(X_val)

    print("Preprocessing test data...")
    X_test = notch_filter(X_test)
    X_test = clip_spikes(X_test)
    X_test = butterworth_filter(X_test)
    X_test_cleaned, X_test_raw = apply_ica(X_test)

    # Plot raw vs. cleaned for specified channels
    channels_to_plot = [0, 2, 3, 5, 16, 17]
    for ch in channels_to_plot:
        plt.figure(figsize=(10, 5))
        plt.plot(X_train_raw[0, :, ch], label=f'Raw ch{ch}')
        plt.plot(X_train_cleaned[0, :, ch], label=f'Cleaned ch{ch}')
        plt.title(f"Raw vs. Cleaned Data for Channel {ch} (Training Set)")
        plt.legend()
        plt.savefig(f"ica_comparison_ch{ch}.png")
        plt.show()
        plt.close()

    # Extract features
    print("Extracting features for training data...")
    X_train_features = extract_all_features(X_train_cleaned)
    print("Extracting features for validation data...")
    X_val_features = extract_all_features(X_val_cleaned)
    print("Extracting features for test data...")
    X_test_features = extract_all_features(X_test_cleaned)

    # Combine train and validation for CV
    X_train_val_features = np.vstack([X_train_features, X_val_features])
    y_train_val = np.hstack([y_train, y_val])
    X_test_features = X_test_features

    # Compute class weights based on y_train_val
    n_negative = np.sum(y_train_val == 0)
    n_positive = np.sum(y_train_val == 1)
    class_weights = {0: (n_positive + n_negative) / (2 * n_negative), 1: (n_positive + n_negative) / (2 * n_positive)}
    print(f"Computed class weights for training: {class_weights}")

    # Define feature names
    n_channels = X.shape[2]
    feature_names = []
    for ch in range(n_channels):
        feature_names.extend([
            f'engagement_ch{ch}', f'theta_ch{ch}', f'alpha_ch{ch}', f'beta_ch{ch}',
            f'pac_theta_alpha_ch{ch}', f'pac_theta_beta_ch{ch}', f'pac_alpha_beta_ch{ch}',
            f'wavelet_mean1_ch{ch}', f'wavelet_max1_ch{ch}',
            f'wavelet_mean2_ch{ch}', f'wavelet_max2_ch{ch}',
            f'wavelet_mean3_ch{ch}', f'wavelet_max3_ch{ch}',
            f'theta_beta_ratio_ch{ch}'
        ])
    microstate_names = [f'microstate_duration_{i}' for i in range(4)] + [f'microstate_coverage_{i}' for i in range(4)] + ['gfp_mean', 'gfp_std']
    corr_names = [f'corr_ch{i}_ch{j}' for i in range(n_channels) for j in range(i+1, n_channels)]
    coh_names = [f'coh_{band}_ch{i}_ch{j}' for band in ['theta', 'alpha', 'beta'] for i in range(n_channels) for j in range(i+1, n_channels)]
    feature_names.extend(microstate_names)
    feature_names.extend(corr_names)
    feature_names.extend(coh_names)

    # Verify feature names length
    expected_feature_count = 14 * n_channels + 10 + (n_channels * (n_channels - 1) // 2) + (3 * (n_channels * (n_channels - 1) // 2))
    if len(feature_names) != expected_feature_count:
        raise ValueError(f"Feature names length mismatch: {len(feature_names)} vs {expected_feature_count}")

    # Feature selection
    print("Selecting features with MO-DTW-AS-SHAP-XGRFE...")
    selected_features_list, X_train_val_selected_list, y_train_val_list, X_test_selected_list, y_test_list = select_features(X_train_val_features, y_train_val, feature_names)

    # Model training and evaluation
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_results = []

    for fold, (X_train_val_selected, y_train_val, X_test_selected, y_test) in enumerate(zip(X_train_val_selected_list, y_train_val_list, X_test_selected_list, y_test_list)):
        print(f"\nCross-validation fold {fold+1}/3")
        
        # CatBoost with hyperparameter tuning
        cat_param_dist = {
            'iterations': [50, 100],
            'depth': [2, 4],
            'learning_rate': [0.01, 0.05],
            'l2_leaf_reg': [3, 5],
            'border_count': [32, 64]
        }
        cat_base = CatBoostClassifier(random_state=42, verbose=0, class_weights=class_weights)
        cat_search = RandomizedSearchCV(cat_base, cat_param_dist, n_iter=10, cv=3, scoring='f1', random_state=42, n_jobs=-1)
        cat_search.fit(X_train_val_selected, y_train_val)
        cat_model = cat_search.best_estimator_

        # KNN-DTW
        knn_dtw_model = KNNDTWClassifier(n_neighbors=5)
        knn_dtw_model.fit(X_train_val_selected, y_train_val)

        # Gradient Boosting
        gb_param_dist = {
            'n_estimators': [50, 100],
            'max_depth': [2, 3],
            'learning_rate': [0.01, 0.05],
            'subsample': [0.8, 1.0]
        }
        gb_base = GradientBoostingClassifier(random_state=42)
        gb_search = RandomizedSearchCV(gb_base, gb_param_dist, n_iter=10, cv=3, scoring='f1', random_state=42, n_jobs=-1)
        gb_search.fit(X_train_val_selected, y_train_val)
        gb_model = gb_search.best_estimator_

        # Stacking Classifier
        estimators = [('cat', cat_model), ('knn_dtw', knn_dtw_model), ('gb', gb_model)]
        stack_model = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(random_state=42, C=0.1), cv=3)
        stack_model.fit(X_train_val_selected, y_train_val)

        # Threshold tuning for CatBoost
        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        best_threshold = 0.5
        best_f1 = 0
        for inner_fold, (train_idx, val_idx) in enumerate(inner_cv.split(X_train_val_selected, y_train_val)):
            X_train_inner, X_val_inner = X_train_val_selected[train_idx], X_train_val_selected[val_idx]
            y_train_inner, y_val_inner = y_train_val[train_idx], y_train_val[val_idx]
            cat_model_inner = CatBoostClassifier(**cat_search.best_params_, random_state=42, verbose=0, class_weights=class_weights)
            cat_model_inner.fit(X_train_inner, y_train_inner)
            thresholds = np.arange(0.3, 0.7, 0.05)
            for thresh in thresholds:
                y_pred_val = (cat_model_inner.predict_proba(X_val_inner)[:, 1] > thresh).astype(int)
                f1 = f1_score(y_val_inner, y_pred_val, zero_division=0)
                if f1 > best_f1:
                    best_f1 = f1
                    best_threshold = thresh
        print(f"Fold {fold+1} Best Threshold: {best_threshold:.2f}")

        # Evaluate models
        cat_val_sens, cat_val_spec, cat_val_f1, cat_val_acc, _ = evaluate_metrics(cat_model, X_train_val_selected, y_train_val, threshold=best_threshold)
        stack_val_sens, stack_val_spec, stack_val_f1, stack_val_acc, _ = evaluate_metrics(stack_model, X_train_val_selected, y_train_val, threshold=best_threshold)
        cat_test_sens, cat_test_spec, cat_test_f1, cat_test_acc, y_pred_test_cat = evaluate_metrics(cat_model, X_test_selected, y_test, threshold=best_threshold)
        stack_test_sens, stack_test_spec, stack_test_f1, stack_test_acc, y_pred_test_stack = evaluate_metrics(stack_model, X_test_selected, y_test, threshold=best_threshold)

        print(f"Fold {fold+1} CatBoost Val Metrics: Sensitivity={cat_val_sens:.4f}, Specificity={cat_val_spec:.4f}, F1={cat_val_f1:.4f}, Accuracy={cat_val_acc:.4f}")
        print(f"Fold {fold+1} Stack Val Metrics: Sensitivity={stack_val_sens:.4f}, Specificity={stack_val_spec:.4f}, F1={stack_val_f1:.4f}, Accuracy={stack_val_acc:.4f}")
        print(f"Fold {fold+1} CatBoost Test Metrics: Sensitivity={cat_test_sens:.4f}, Specificity={cat_test_spec:.4f}, F1={cat_test_f1:.4f}, Accuracy={cat_test_acc:.4f}")
        print(f"Fold {fold+1} Stack Test Metrics: Sensitivity={stack_test_sens:.4f}, Specificity={stack_test_spec:.4f}, F1={stack_test_f1:.4f}, Accuracy={stack_test_acc:.4f}")

        fold_results.append({
            'fold': fold + 1,
            'cat_val': (cat_val_sens, cat_val_spec, cat_val_f1, cat_val_acc),
            'stack_val': (stack_val_sens, stack_val_spec, stack_val_f1, stack_val_acc),
            'cat_test': (cat_test_sens, cat_test_spec, cat_test_f1, cat_test_acc),
            'stack_test': (stack_test_sens, stack_test_spec, stack_test_f1, stack_test_acc),
            'y_pred_test_cat': y_pred_test_cat,
            'y_pred_test_stack': y_pred_test_stack,
            'y_test': y_test
        })

    # Average metrics for CatBoost and Stack
    for model in ['cat', 'stack']:
        val_sens = np.mean([r[f'{model}_val'][0] for r in fold_results])
        val_spec = np.mean([r[f'{model}_val'][1] for r in fold_results])
        val_f1 = np.mean([r[f'{model}_val'][2] for r in fold_results])
        val_acc = np.mean([r[f'{model}_val'][3] for r in fold_results])
        test_sens = np.mean([r[f'{model}_test'][0] for r in fold_results])
        test_spec = np.mean([r[f'{model}_test'][1] for r in fold_results])
        test_f1 = np.mean([r[f'{model}_test'][2] for r in fold_results])
        test_acc = np.mean([r[f'{model}_test'][3] for r in fold_results])
        print(f"\n{model.upper()} Model Average Metrics:")
        print(f"Validation - Sensitivity: {val_sens:.4f}, Specificity: {val_spec:.4f}, F1: {val_f1:.4f}, Accuracy: {val_acc:.4f}")
        print(f"Test - Sensitivity: {test_sens:.4f}, Specificity: {test_spec:.4f}, F1: {test_f1:.4f}, Accuracy: {test_acc:.4f}")

    # Additional Models for Comparison
    print("\nTraining additional models for comparison...")
    final_models = {
        'Logistic Regression': LogisticRegression(random_state=42, class_weight=class_weights, max_iter=1000, C=0.1),
        'Random Forest': RandomForestClassifier(random_state=42, class_weight=class_weights, n_estimators=50, max_depth=5),
        'SVM': SVC(random_state=42, class_weight=class_weights, probability=True, kernel='rbf', C=1.0)
    }
    final_results = {}

    # Use the last fold for final evaluation and visualizations
    X_train_val_selected = X_train_val_selected_list[-1]
    y_train_val = y_train_val_list[-1]
    X_test_selected = X_test_selected_list[-1]
    y_test = y_test_list[-1]
    selected_features = selected_features_list[-1]

    for model_name, model in final_models.items():
        model.fit(X_train_val_selected, y_train_val)
        sens, spec, f1, acc, y_pred = evaluate_metrics(model, X_test_selected, y_test, threshold=0.5)
        print(f"\n{model_name} Test Metrics: Sensitivity={sens:.4f}, Specificity={spec:.4f}, F1={f1:.4f}, Accuracy={acc:.4f}")
        final_results[model_name] = (sens, spec, f1, acc, y_pred)

    # Visualizations for CatBoost (last fold)
    print("\nComputing SHAP values for CatBoost...")
    cat_model.fit(X_train_val_selected, y_train_val)
    if np.any(np.isnan(X_test_selected)):
        print("Skipping SHAP computation due to NaN values")
    else:
        explainer = shap.TreeExplainer(cat_model)
        # Limit the data size for SHAP
        X_test_subset = X_test_selected[:100] if X_test_selected.shape[0] > 100 else X_test_selected
        shap_values_test = explainer.shap_values(X_test_subset)

        plt.figure()
        shap.summary_plot(shap_values_test, X_test_subset, feature_names=selected_features, plot_type="dot", max_display=20, show=False)
        plt.title("SHAP Summary Plot (CatBoost - MO-DTW-AS-SHAP-XGRFE Features)")
        plt.show()
        plt.close()

        plt.figure()
        shap.summary_plot(shap_values_test, X_test_subset, feature_names=selected_features, plot_type="bar", max_display=20, show=False)
        plt.title("Mean SHAP Values of Top 20 Features (CatBoost - MO-DTW-AS-SHAP-XGRFE)")
        plt.show()
        plt.close()

    # Confusion Matrix for CatBoost
    y_pred_test_cat = fold_results[-1]['y_pred_test_cat']
    cm = confusion_matrix(y_test, y_pred_test_cat)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Control', 'ADHD'], yticklabels=['Control', 'ADHD'])
    plt.title("Confusion Matrix (CatBoost - Test Set)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()
    plt.close()

    # ROC-AUC Curve for CatBoost
    y_scores = cat_model.predict_proba(X_test_selected)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC-AUC Curve (CatBoost - Test Set)')
    plt.legend(loc="lower right")
    plt.show()
    plt.close()

    print("\nTop 10 Selected Features:")
    for i, feature in enumerate(selected_features[:min(10, len(selected_features))]):
        print(f"{i+1}. {feature}")
    print_memory_usage()

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'hurst'

In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
from scipy.io import loadmat
from scipy import signal
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.feature_selection import VarianceThreshold
import xgboost as xgb
import shap
from scipy.signal import hilbert, find_peaks, iirnotch
from sklearn.cluster import KMeans
from tqdm import tqdm
import psutil
from sklearn.metrics import confusion_matrix, f1_score, roc_curve, auc, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kurtosis
import mne
from dtaidistance import dtw
from imblearn.combine import SMOTEENN
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.cluster._kmeans")

def print_memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024 / 1024:.2f} MB")

# Preprocessing Functions (unchanged)
def load_and_segment_file(file_path, window_size=512, stride=256, expected_channels=19):
    mat_data = loadmat(file_path)
    var_names = [k for k in mat_data.keys() if not k.startswith('__')]
    if not var_names:
        raise ValueError(f"No non-metadata variables found in {file_path}")
    possible_var_names = ['data', 'eeg', 'signal', 'EEG', 'Data'] + var_names
    data = None
    for var_name in possible_var_names:
        if var_name in mat_data:
            data = mat_data[var_name]
            break
    if data is None:
        raise ValueError(f"No valid EEG data variable found in {file_path}")
    if data.ndim == 1:
        data = data.reshape(1, -1, 1)
    elif data.ndim == 2:
        data = data[np.newaxis, :, :]
    elif data.ndim == 3:
        pass
    else:
        raise ValueError(f"Unexpected data dimensions {data.ndim}")
    n_segments, n_samples, n_channels = data.shape
    if n_channels != expected_channels:
        if n_channels < expected_channels:
            data = np.pad(data, ((0, 0), (0, 0), (0, expected_channels - n_channels)), mode='constant')
        else:
            data = data[:, :, :expected_channels]
    max_abs_value = np.max(np.abs(data))
    if max_abs_value > 1000:
        data = data / 1000.0
    if max_abs_value > 100:
        scaling_factor = 100 / max_abs_value
        data = data * scaling_factor
    all_segments = []
    for seg in range(n_segments):
        seg_data = data[seg]
        seg_samples = seg_data.shape[0]
        if seg_samples < window_size:
            padded_data = np.pad(seg_data, ((0, window_size - seg_samples), (0, 0)), mode='constant')
            all_segments.append(padded_data[np.newaxis, :, :])
            continue
        n_seg_segments = ((seg_samples - window_size + stride - 1) // stride) + 1
        segments = np.zeros((n_seg_segments, window_size, n_channels))
        for i in range(n_seg_segments):
            start = i * stride
            end = min(start + window_size, seg_samples)
            segment_length = end - start
            if segment_length == window_size:
                segments[i] = seg_data[start:end, :]
            else:
                segments[i] = np.pad(seg_data[start:end, :], ((0, window_size - segment_length), (0, 0)), mode='constant')
        all_segments.append(segments)
    if not all_segments:
        return None
    return np.concatenate(all_segments, axis=0)

def load_from_folder(folder_path, window_size=512, stride=256, expected_channels=19):
    if not os.path.exists(folder_path):
        return None
    mat_files = [f for f in os.listdir(folder_path) if f.endswith('.mat')]
    if not mat_files:
        return None
    all_segments = []
    for file in tqdm(mat_files, desc=f"Loading files from {folder_path}"):
        file_path = os.path.join(folder_path, file)
        segments = load_and_segment_file(file_path, window_size, stride, expected_channels)
        if segments is not None:
            all_segments.append(segments)
    if not all_segments:
        return None
    return np.concatenate(all_segments, axis=0), mat_files

def load_all_data(window_size=512, stride=256, expected_channels=19):
    adhd_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1"
    adhd_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part2/ADHD_part2"
    control_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part1/Control_part1"
    control_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part2"
    adhd_segments1, adhd_files1 = load_from_folder(adhd_path1, window_size, stride, expected_channels)
    adhd_segments2, adhd_files2 = load_from_folder(adhd_path2, window_size, stride, expected_channels)
    adhd_segments = []
    if adhd_segments1 is not None:
        adhd_segments.append(adhd_segments1)
    if adhd_segments2 is not None:
        adhd_segments.append(adhd_segments2)
    if not adhd_segments:
        raise ValueError("Failed to load any ADHD data")
    adhd_segments = np.concatenate(adhd_segments, axis=0)
    adhd_files = adhd_files1 + adhd_files2

    control_segment1, control_files1 = load_from_folder(control_path1, window_size, stride, expected_channels)
    control_segment2, control_files2 = load_from_folder(control_path2, window_size, stride, expected_channels)
    control_segments = []
    if control_segment1 is not None:
        control_segments.append(control_segment1)
    if control_segment2 is not None:
        control_segments.append(control_segment2)
    if not control_segments:
        raise ValueError("Failed to load any Control data")
    control_segments = np.concatenate(control_segments, axis=0)
    control_files = control_files1 + control_files2

    y_adhd = np.ones(adhd_segments.shape[0])
    y_control = np.zeros(control_segments.shape[0])
    X = np.concatenate([adhd_segments, control_segments], axis=0)
    y = np.concatenate([y_adhd, y_control])

    adhd_segment_counts = []
    adhd_patient_ids = []
    valid_adhd_files = []
    for f in adhd_files:
        path = os.path.join(adhd_path1, f) if f in adhd_files1 else os.path.join(adhd_path2, f)
        segments = load_and_segment_file(path)
        count = segments.shape[0] if segments is not None else 0
        if count > 0:
            adhd_segment_counts.append(count)
            valid_adhd_files.append(f)
            patient_number = f.split('.')[0]
            adhd_patient_ids.extend([patient_number] * count)

    control_segment_counts = []
    control_patient_ids = []
    valid_control_files = []
    for f in control_files:
        path = os.path.join(control_path1, f) if f in control_files1 else os.path.join(control_path2, f)
        segments = load_and_segment_file(path)
        count = segments.shape[0] if segments is not None else 0
        if count > 0:
            control_segment_counts.append(count)
            valid_control_files.append(f)
            patient_number = f.split('.')[0]
            control_patient_ids.extend([patient_number] * count)

    patient_ids = adhd_patient_ids + control_patient_ids

    if len(patient_ids) != X.shape[0]:
        raise ValueError(f"Patient IDs length ({len(patient_ids)}) does not match number of segments ({X.shape[0]})")

    print(f"Loaded {X.shape[0]} segments from {len(valid_adhd_files) + len(valid_control_files)} patients")
    return X, y, patient_ids

def notch_filter(data, fs=128, freqs=[50, 60], Q=30):
    filtered_data = data.copy()
    for freq in freqs:
        b, a = signal.iirnotch(freq / (fs / 2), Q)
        filtered_data = signal.filtfilt(b, a, filtered_data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def clip_spikes(data, threshold=500):
    clipped_data = data.copy()
    clipped_data[np.abs(clipped_data) > threshold] = np.sign(clipped_data[np.abs(clipped_data) > threshold]) * threshold
    fp2_threshold = 30
    clipped_data[:, :, 1] = np.where(np.abs(clipped_data[:, :, 1]) > fp2_threshold,
                                     np.sign(clipped_data[:, :, 1]) * fp2_threshold,
                                     clipped_data[:, :, 1])
    return clipped_data

def butterworth_filter(data, fs=128, lowcut=1, highcut=45, order=4):
    nyquist = fs / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = signal.butter(order, [low, high], btype='band')
    filtered_data = signal.filtfilt(b, a, data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def apply_ica(data, fs=128, n_components=None):
    if data.ndim == 2:
        data = data[np.newaxis, :, :]
    elif data.ndim != 3:
        raise ValueError(f"Expected 3D data, got {data.ndim}D")
    n_segments, n_samples, n_channels = data.shape
    if n_channels != 19:
        raise ValueError(f"Expected 19 channels, got {n_channels}")
    if n_components is None or n_components > n_channels:
        n_components = min(19, n_channels)
    total_samples = n_segments * n_samples
    concatenated_data = data.reshape(total_samples, n_channels)
    ch_names = [f"EEG {i}" for i in range(n_channels)]
    montage_labels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
    info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types="eeg")
    raw = mne.io.RawArray(concatenated_data.T, info)
    channel_mapping = {f"EEG {i}": montage_labels[i] for i in range(n_channels)}
    raw.rename_channels(channel_mapping)
    try:
        theta_values = [-18, 18, -54, -39, 0, 39, 54, -90, -90, 90, 90, 90, -126, -141, 180, 141, 126, -162, 162]
        radius_values_cm = [51.1, 51.1, 51.1, 33.3, 25.6, 33.3, 51.1, 51.1, 25.6, 0, 25.6, 51.1, 51.1, 33.3, 25.6, 33.3, 51.1, 51.1, 51.1]
        radius_values_m = [r / 100 for r in radius_values_cm]
        ch_pos = {label: [radius * np.cos(np.deg2rad(theta)), radius * np.sin(np.deg2rad(theta)), 0.0]
                  for label, theta, radius in zip(montage_labels, theta_values, radius_values_m)}
        montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='head')
        raw.set_montage(montage)
    except Exception:
        montage = mne.channels.make_standard_montage('standard_1020')
        raw.set_montage(montage)
    raw.filter(l_freq=0.5, h_freq=None, method='iir')
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=42, max_iter=500)
    ica.fit(raw)
    ica_sources = ica.get_sources(raw).get_data().T
    kurtosis_values = np.array([kurtosis(ica_sources[:, i]) for i in range(n_components)])
    variance_values = np.var(ica_sources, axis=0)
    peak_amplitude = np.max(np.abs(ica_sources), axis=0)
    fp2_weight = np.max(np.abs(ica_sources[total_samples // 2:total_samples // 2 + n_samples, 1])) / np.max(peak_amplitude) if n_channels > 1 else 0
    kurtosis_norm = (kurtosis_values - kurtosis_values.min()) / (kurtosis_values.max() - kurtosis_values.min() + 1e-10)
    variance_norm = (variance_values - variance_values.min()) / (variance_values.max() - variance_values.min() + 1e-10)
    peak_norm = (peak_amplitude - peak_amplitude.min()) / (peak_amplitude.max() - peak_amplitude.min() + 1e-10)
    combined_score = 0.45 * kurtosis_norm + 0.25 * variance_norm + 0.2 * peak_norm + 0.1 * fp2_weight
    top_artifact_indices = np.argsort(combined_score)[-min(10, n_components):]
    ica.exclude = top_artifact_indices.tolist()
    raw_clean = ica.apply(raw.copy())
    cleaned_data_raw = raw_clean.get_data()
    cleaned_data = cleaned_data_raw.T.reshape(n_segments, n_samples, n_channels)
    return cleaned_data, data

# Feature Extraction Functions (unchanged)
def normalize_data(data):
    scaler = StandardScaler()
    return scaler.fit_transform(data)

def calculate_engagement_index(data, fs=128, prev_value=None):
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    alpha = calculate_psd(data, fs, freq_range=(8, 13))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    if np.any(np.isnan([theta, alpha, beta])) or beta == 0:
        return prev_value if prev_value is not None else 0
    return np.mean((theta + alpha) / (beta + 1e-10))

def calculate_psd(data, fs=128, freq_range=None):
    nperseg = min(len(data), 512)
    f, psd = signal.welch(data, fs=fs, nperseg=nperseg, axis=0)
    if freq_range:
        freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
        return np.mean(psd[freq_mask], axis=0) if freq_mask.any() else 0
    return psd.mean(axis=0)

def phase_amplitude_coupling(data, fs=128, prev_pac=None):
    theta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 4, 8)[0, :, 0]
    alpha = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 8, 13)[0, :, 0]
    beta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 13, 30)[0, :, 0]
    if np.any(np.isnan([theta, alpha, beta])) or np.std([theta, alpha, beta]) == 0:
        return prev_pac if prev_pac is not None else np.zeros(3)
    theta_phase = np.angle(hilbert(theta))
    alpha_phase = np.angle(hilbert(alpha))
    beta_phase = np.angle(hilbert(beta))
    alpha_amp = np.abs(hilbert(alpha))
    beta_amp = np.abs(hilbert(beta))
    theta_amp = np.abs(hilbert(theta))
    pac_theta_alpha = np.abs(np.mean(alpha_amp * np.exp(1j * theta_phase)))
    pac_theta_beta = np.abs(np.mean(beta_amp * np.exp(1j * theta_phase)))
    pac_alpha_beta = np.abs(np.mean(beta_amp * np.exp(1j * alpha_phase)))
    return np.array([pac_theta_alpha, pac_theta_beta, pac_alpha_beta])

def wavelet_features(data, prev_wavelet=None):
    scales = np.arange(1, 4)
    coeffs, _ = pywt.cwt(data, scales, 'mexh', axis=0)
    if np.any(np.isnan(coeffs)):
        return prev_wavelet if prev_wavelet is not None else np.zeros(6)
    features = []
    for c in coeffs:
        features.extend([np.mean(np.abs(c)), np.max(np.abs(c))])
    return np.array(features)

def theta_beta_ratio(data, fs=128, prev_value=None):
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    if np.any(np.isnan([theta, beta])) or beta == 0:
        return prev_value if prev_value is not None else 0
    return theta / (beta + 1e-10)

def channel_coherence(data, fs=128, prev_coherence=None):
    n_channels = data.shape[1]
    coherence_features = []
    nperseg = min(data.shape[0], 512)
    expected_length = 3 * (n_channels * (n_channels - 1) // 2)
    for band, freq_range in [('theta', (4, 8)), ('alpha', (8, 13)), ('beta', (13, 30))]:
        for i in range(n_channels):
            for j in range(i + 1, n_channels):
                f, coh = signal.coherence(data[:, i], data[:, j], fs=fs, nperseg=nperseg)
                freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
                coh_value = np.mean(coh[freq_mask]) if freq_mask.any() else 0
                if np.isnan(coh_value):
                    return prev_coherence if prev_coherence is not None else np.zeros(expected_length)
                coherence_features.append(coh_value)
    return np.array(coherence_features)

def channel_correlation(data, prev_correlation=None):
    n_channels = data.shape[1]
    corr_features = []
    expected_length = n_channels * (n_channels - 1) // 2
    for i in range(n_channels):
        for j in range(i + 1, n_channels):
            corr = np.corrcoef(data[:, i], data[:, j])[0, 1]
            if np.isnan(corr):
                return prev_correlation if prev_correlation is not None else np.zeros(expected_length)
            corr_features.append(corr)
    return np.array(corr_features)

def microstate_analysis(data, fs=128, n_states=4, prev_microstates=None):
    expected_length = n_states + n_states + 2
    gfp = np.std(data, axis=1)
    if np.any(np.isnan(gfp)):
        return prev_microstates if prev_microstates is not None else np.zeros(expected_length)
    peaks, _ = find_peaks(gfp, distance=fs//10)
    if len(peaks) < n_states:
        peak_data = data
    else:
        peak_data = data[peaks]
    kmeans = KMeans(n_clusters=n_states, random_state=42, n_init=10)
    kmeans.fit(peak_data)
    templates = kmeans.cluster_centers_
    distances = np.zeros((data.shape[0], n_states))
    for i, template in enumerate(templates):
        template_norm = np.linalg.norm(template)
        data_norm = np.linalg.norm(data, axis=1)
        distances[:, i] = np.abs(np.sum(data * template, axis=1) / (data_norm * template_norm + 1e-10))
    labels = np.argmax(distances, axis=1)
    durations = np.zeros(n_states)
    gfp_peaks = gfp[peaks] if len(peaks) > 0 else gfp
    gfp_mean = np.mean(gfp_peaks)
    gfp_std = np.std(gfp_peaks)
    for state in range(n_states):
        state_indices = np.where(labels == state)[0]
        if len(state_indices) > 0:
            diff = np.diff(state_indices)
            breaks = np.where(diff > 1)[0]
            segment_lengths = np.diff(np.concatenate([[0], breaks + 1, [len(state_indices)]]))
            durations[state] = np.mean(segment_lengths) / fs * 1000 if len(segment_lengths) > 0 else 0
    coverage = np.bincount(labels, minlength=n_states).astype(float) / len(labels)
    microstate_features = np.hstack([durations, coverage, [gfp_mean, gfp_std]])
    return microstate_features

def extract_features_single(sample, idx, prev_features=None):
    n_channels = sample.shape[1]
    normalized = np.zeros_like(sample)
    for ch in range(n_channels):
        channel_data = sample[:, ch]
        if np.std(channel_data) > 0:
            normalized[:, ch] = (channel_data - np.mean(channel_data)) / np.std(channel_data)
        else:
            normalized[:, ch] = channel_data
    channel_features = []

    features_per_channel = 14
    total_per_channel = n_channels * features_per_channel
    microstate_len = 10
    corr_len = (n_channels * (n_channels - 1)) // 2
    coh_len = 3 * corr_len
    total_prev_len = total_per_channel + microstate_len + corr_len + coh_len

    if prev_features is not None and len(prev_features) != total_prev_len:
        raise ValueError(f"Expected prev_features length {total_prev_len}, got {len(prev_features)}")

    for ch in range(n_channels):
        channel_data = normalized[:, ch]
        offset = ch * features_per_channel
        prev_values = prev_features[offset:offset + features_per_channel] if prev_features is not None else None

        engagement = calculate_engagement_index(channel_data, prev_value=prev_values[0] if prev_values is not None else None)
        theta = calculate_psd(channel_data, freq_range=(4, 8))
        alpha = calculate_psd(channel_data, freq_range=(8, 13))
        beta = calculate_psd(channel_data, freq_range=(13, 30))
        pac = phase_amplitude_coupling(channel_data, prev_pac=prev_values[4:7] if prev_values is not None else None)
        wavelet = wavelet_features(channel_data, prev_wavelet=prev_values[7:13] if prev_values is not None else None)
        tbr = theta_beta_ratio(channel_data, prev_value=prev_values[13] if prev_values is not None else None)

        ch_features = np.concatenate([
            [engagement], [theta], [alpha], [beta], pac, wavelet, [tbr]
        ])
        channel_features.append(ch_features)

    per_channel_features = np.array(channel_features).flatten()
    microstates = microstate_analysis(normalized, prev_microstates=prev_features[total_per_channel:total_per_channel + microstate_len] if prev_features is not None else None)
    correlations = channel_correlation(normalized, prev_correlation=prev_features[total_per_channel + microstate_len:total_per_channel + microstate_len + corr_len] if prev_features is not None else None)
    coherences = channel_coherence(normalized, prev_coherence=prev_features[total_per_channel + microstate_len + corr_len:] if prev_features is not None else None)

    sample_features = np.concatenate([per_channel_features, microstates, correlations, coherences])
    return sample_features

def extract_all_features(data, chunk_size=100):
    n_samples, n_samples_per_segment, n_channels = data.shape
    feature_matrix = []
    prev_features = None

    with tqdm(total=n_samples, desc="Extracting features", unit="segment") as pbar:
        for i in range(n_samples):
            sample_features = extract_features_single(data[i], i, prev_features)
            feature_matrix.append(sample_features)
            prev_features = sample_features
            pbar.update(1)
    feature_matrix = np.array(feature_matrix)
    print(f"Feature matrix shape: {feature_matrix.shape}")
    return feature_matrix

# Feature Selection (functionality unchanged, only n_features_to_select increased)
def remove_highly_correlated_features(X, feature_names, threshold=0.85):
    corr_matrix = np.corrcoef(X.T)
    to_remove = set()
    for i in range(len(corr_matrix)):
        for j in range(i + 1, len(corr_matrix)):
            if abs(corr_matrix[i, j]) > threshold:
                to_remove.add(j)
    keep_indices = [i for i in range(X.shape[1]) if i not in to_remove]
    return X[:, keep_indices], np.array(feature_names)[keep_indices].tolist()

def select_features(X, y, feature_names, min_window_size=50):
    outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    selected_features_list = []
    X_test_selected_list = []
    y_test_list = []
    X_train_val_selected_list = []
    y_train_val_list = []

    n_negative = np.sum(y == 0)
    n_positive = np.sum(y == 1)
    class_weights = {0: 1.0, 1: 2.0}  # Adjusted to favor minority class
    print(f"Computed class weights: {class_weights}")

    for outer_fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        X_train_val, X_test_fold = X[train_val_idx], X[test_idx]
        y_train_val, y_test_fold = y[train_val_idx], y[test_idx]

        scaler = StandardScaler()
        X_train_val = scaler.fit_transform(X_train_val)
        X_test_fold = scaler.transform(X_test_fold)
        X_train_val = np.nan_to_num(X_train_val, nan=0.0, posinf=0.0, neginf=0.0)
        X_test_fold = np.nan_to_num(X_test_fold, nan=0.0, posinf=0.0, neginf=0.0)

        smote_enn = SMOTEENN(random_state=42, sampling_strategy='auto')
        X_train_val_balanced, y_train_val_balanced = smote_enn.fit_resample(X_train_val, y_train_val)
        y_train_val_balanced = np.rint(y_train_val_balanced).astype(np.int64)
        if not np.all(np.isin(y_train_val_balanced, [0, 1])):
            raise ValueError(f"Non-binary labels found after SMOTEENN: {np.unique(y_train_val_balanced)}")
        print(f"Fold {outer_fold+1} After SMOTEENN: {X_train_val_balanced.shape[0]} samples, Class distribution: {np.bincount(y_train_val_balanced)}")

        selector = VarianceThreshold(threshold=0.01)
        X_train_val_reduced = selector.fit_transform(X_train_val_balanced)
        X_test_fold_reduced = selector.transform(X_test_fold)
        mask = selector.get_support()
        feature_names_reduced = np.array(feature_names)[mask].tolist()
        print(f"Fold {outer_fold+1} After Variance Thresholding: {X_train_val_reduced.shape[1]} features remaining")

        X_train_val_reduced, feature_names_reduced = remove_highly_correlated_features(X_train_val_reduced, feature_names_reduced, threshold=0.85)
        kept_indices = [i for i, fname in enumerate(feature_names) if fname in feature_names_reduced]
        X_test_fold_reduced = X_test_fold[:, kept_indices]
        feature_indices = [feature_names.index(fname) for fname in feature_names_reduced]
        X_test_fold_reduced = X_test_fold_reduced[:, [kept_indices.index(idx) for idx in feature_indices]]
        print(f"Fold {outer_fold+1} After Correlation Filtering: {X_train_val_reduced.shape[1]} features remaining")

        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        xgb_model = xgb.XGBClassifier(
            random_state=42,
            n_estimators=100,
            eval_metric='logloss',
            scale_pos_weight=class_weights[1] / class_weights[0],
            max_depth=3,
            min_child_weight=5,
            gamma=2.0,
            reg_alpha=1.0,
            reg_lambda=2.0
        )

        sensitivities, specificities, f1_scores, accuracies = [], [], [], []
        best_thresholds = []
        for fold, (train_idx, val_idx) in enumerate(inner_cv.split(X_train_val_reduced, y_train_val_balanced)):
            X_train, X_val = X_train_val_reduced[train_idx], X_train_val_reduced[val_idx]
            y_train, y_val = y_train_val_balanced[train_idx], y_train_val_balanced[val_idx]
            xgb_model.fit(X_train, y_train)

            y_scores = xgb_model.predict_proba(X_val)[:, 1]
            fpr, tpr, thresholds = roc_curve(y_val, y_scores)
            youden_j = tpr + (1 - fpr) - 1
            best_idx = np.argmax(youden_j)
            best_threshold = thresholds[best_idx]
            y_pred = (y_scores > best_threshold).astype(int)
            best_thresholds.append(best_threshold)

            sens, spec, f1, acc, _ = evaluate_metrics(xgb_model, X_val, y_val, threshold=best_threshold)
            print(f"Fold {outer_fold+1}.{fold+1} Initial XGBoost: Sensitivity={sens:.4f}, Specificity={spec:.4f}, F1={f1:.4f}, Accuracy={acc:.4f}")
            sensitivities.append(sens)
            specificities.append(spec)
            f1_scores.append(f1)
            accuracies.append(acc)

        avg_threshold = np.mean(best_thresholds)
        print(f"Fold {outer_fold+1} Initial XGBoost CV: Sensitivity={np.mean(sensitivities):.4f}, Specificity={np.mean(specificities):.4f}, F1={np.mean(f1_scores):.4f}, Accuracy={np.mean(accuracies):.4f}, Avg Threshold={avg_threshold:.2f}")

        X_train_mo, X_val_mo, y_train_mo, y_val_mo = train_test_split(X_train_val_reduced, y_train_val_balanced, test_size=0.2, stratify=y_train_val_balanced, random_state=42)
        xgb_model.fit(X_train_mo, y_train_mo)
        explainer = shap.TreeExplainer(xgb_model)
        X_train_val_balanced_reduced = X_train_val_balanced[:, kept_indices]
        shap_values = explainer.shap_values(X_train_val_balanced_reduced)
        shap_importance = np.abs(shap_values).mean(axis=0)

        dtw_scores = []
        for feature_idx in range(X_train_val_reduced.shape[1]):
            feature_series = X_train_val_reduced[:, feature_idx]
            if np.any(np.isnan(feature_series)) or np.std(feature_series) == 0:
                dtw_scores.append(0)
                continue
            sub_windows = [feature_series[i:i+min_window_size] for i in range(0, len(feature_series)-min_window_size, min_window_size)]
            if len(sub_windows) < 2:
                dtw_scores.append(0)
                continue
            try:
                dtw_sum = 0
                count = 0
                for i in range(len(sub_windows)):
                    for j in range(i+1, len(sub_windows)):
                        dist = dtw.distance(sub_windows[i], sub_windows[j])
                        if np.isnan(dist):
                            continue
                        dtw_sum += dist
                        count += 1
                dtw_scores.append(dtw_sum / count if count > 0 else 0)
            except Exception as e:
                print(f"Error in DTW computation for feature {feature_idx}: {e}")
                dtw_scores.append(0)
        dtw_scores = np.array(dtw_scores)

        avg_sensitivity = np.mean(sensitivities)
        avg_specificity = np.mean(specificities)
        avg_f1 = np.mean(f1_scores)
        avg_accuracy = np.mean(accuracies)
        metrics = np.array([avg_sensitivity, avg_specificity, avg_f1, avg_accuracy])
        print(f"Fold {outer_fold+1} Metrics: Sensitivity={avg_sensitivity:.4f}, Specificity={avg_specificity:.4f}, F1={avg_f1:.4f}, Accuracy={avg_accuracy:.4f}")

        metrics_norm = (metrics - metrics.min()) / (metrics.max() - metrics.min() + 1e-10)
        metric_weights = 1 - metrics_norm
        metric_weights = metric_weights / metric_weights.sum()
        print(f"Fold {outer_fold+1} Metric Weights: Sensitivity={metric_weights[0]:.4f}, Specificity={metric_weights[1]:.4f}, F1={metric_weights[2]:.4f}, Accuracy={metric_weights[3]:.4f}")

        shap_norm = (shap_importance - shap_importance.min()) / (shap_importance.max() - shap_importance.min() + 1e-10)
        dtw_norm = (dtw_scores - dtw_scores.min()) / (dtw_scores.max() - dtw_scores.min() + 1e-10)
        balanced_weight = np.mean(metric_weights)
        adjusted_shap_norm = shap_norm * (1 + balanced_weight)
        mo_score = 0.5 * adjusted_shap_norm + 0.3 * dtw_norm
        print(f"Fold {outer_fold+1} MO Score Range: min={mo_score.min():.4f}, max={mo_score.max():.4f}")

        feature_types = []
        for fname in feature_names_reduced:
            if 'ch' in fname:
                feature_types.append('channel_feature')
            elif 'corr' in fname:
                feature_types.append('correlation')
            elif 'coh' in fname:
                feature_types.append('coherence')
            elif 'gfp' in fname or 'microstate' in fname:
                feature_types.append('microstate')
            else:
                feature_types.append('other')
        feature_types = np.array(feature_types)

        diversity_penalty = 0.6
        type_counts = {t: 0 for t in set(feature_types)}
        selected_indices = []
        n_features_to_select = max(1, int(0.5 * X_train_val_reduced.shape[1]))  # Increased to 50%
        for _ in range(n_features_to_select):
            adjusted_scores = mo_score.copy()
            for idx in selected_indices:
                if idx in range(len(feature_types)):
                    type_counts[feature_types[idx]] += 1
            for idx in range(len(adjusted_scores)):
                if idx in selected_indices:
                    adjusted_scores[idx] = -np.inf
                else:
                    type_count = type_counts[feature_types[idx]]
                    adjusted_scores[idx] -= diversity_penalty * type_count
            best_idx = np.argmax(adjusted_scores)
            selected_indices.append(best_idx)

        selected_features = [feature_names_reduced[idx] for idx in selected_indices]
        selected_indices = [list(feature_names_reduced).index(f) for f in selected_features]
        X_train_val_selected = X_train_val_reduced[:, selected_indices]
        X_test_selected = X_test_fold_reduced[:, selected_indices]

        selected_types = [feature_types[idx] for idx in selected_indices]
        type_distribution = {t: selected_types.count(t) for t in set(selected_types)}
        print("\nDistribution of Selected Feature Types:")
        for t, count in type_distribution.items():
            print(f"{t}: {count} features")

        selected_features_list.append(selected_features)
        X_test_selected_list.append(X_test_selected)
        y_test_list.append(y_test_fold)
        X_train_val_selected_list.append(X_train_val_selected)
        y_train_val_list.append(y_train_val_balanced)

    return selected_features_list, X_train_val_selected_list, y_train_val_list, X_test_selected_list, y_test_list

# Model Evaluation
def evaluate_metrics(model, X, y, threshold=0.5):
    if hasattr(model, 'predict_proba'):
        y_scores = model.predict_proba(X)[:, 1]
        y_pred = (y_scores > threshold).astype(int)
    else:
        y_pred = model.predict(X)
    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = f1_score(y, y_pred, zero_division=0)
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    return sensitivity, specificity, f1, accuracy, y_pred

# Main Execution
def main():
    window_size = 512
    stride = 256
    expected_channels = 19
    X, y, patient_ids = load_all_data(window_size=window_size, stride=stride, expected_channels=expected_channels)

    unique_patients = np.unique(patient_ids)
    patient_labels = []
    for patient in unique_patients:
        patient_indices = [i for i, pid in enumerate(patient_ids) if pid == patient]
        patient_label = y[patient_indices[0]]
        patient_labels.append(patient_label)
    patient_labels = np.array(patient_labels)

    train_val_patients, test_patients = train_test_split(unique_patients, test_size=0.2, stratify=patient_labels, random_state=42)
    train_val_labels = [y[[i for i, pid in enumerate(patient_ids) if pid == patient][0]] for patient in train_val_patients]
    train_patients, val_patients = train_test_split(train_val_patients, test_size=0.25, stratify=train_val_labels, random_state=42)

    train_idx = [i for i, pid in enumerate(patient_ids) if pid in train_patients]
    val_idx = [i for i, pid in enumerate(patient_ids) if pid in val_patients]
    test_idx = [i for i, pid in enumerate(patient_ids) if pid in test_patients]

    X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
    y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

    print("Preprocessing training data...")
    X_train = notch_filter(X_train)
    X_train = clip_spikes(X_train)
    X_train = butterworth_filter(X_train)
    X_train_cleaned, X_train_raw = apply_ica(X_train)

    print("Preprocessing validation data...")
    X_val = notch_filter(X_val)
    X_val = clip_spikes(X_val)
    X_val = butterworth_filter(X_val)
    X_val_cleaned, X_val_raw = apply_ica(X_val)

    print("Preprocessing test data...")
    X_test = notch_filter(X_test)
    X_test = clip_spikes(X_test)
    X_test = butterworth_filter(X_test)
    X_test_cleaned, X_test_raw = apply_ica(X_test)

    print("Extracting features for training data...")
    X_train_features = extract_all_features(X_train_cleaned)
    print("Extracting features for validation data...")
    X_val_features = extract_all_features(X_val_cleaned)
    print("Extracting features for test data...")
    X_test_features = extract_all_features(X_test_cleaned)

    X_train_val_features = np.vstack([X_train_features, X_val_features])
    y_train_val = np.hstack([y_train, y_val])
    X_test_features = X_test_features

    n_negative = np.sum(y_train_val == 0)
    n_positive = np.sum(y_train_val == 1)
    class_weights = {0: 1.0, 1: 2.0}  # Adjusted for stronger minority emphasis
    print(f"Computed class weights for training: {class_weights}")

    n_channels = X.shape[2]
    feature_names = []
    for ch in range(n_channels):
        feature_names.extend([
            f'engagement_ch{ch}', f'theta_ch{ch}', f'alpha_ch{ch}', f'beta_ch{ch}',
            f'pac_theta_alpha_ch{ch}', f'pac_theta_beta_ch{ch}', f'pac_alpha_beta_ch{ch}',
            f'wavelet_mean1_ch{ch}', f'wavelet_max1_ch{ch}',
            f'wavelet_mean2_ch{ch}', f'wavelet_max2_ch{ch}',
            f'wavelet_mean3_ch{ch}', f'wavelet_max3_ch{ch}',
            f'theta_beta_ratio_ch{ch}'
        ])
    microstate_names = [f'microstate_duration_{i}' for i in range(4)] + [f'microstate_coverage_{i}' for i in range(4)] + ['gfp_mean', 'gfp_std']
    corr_names = [f'corr_ch{i}_ch{j}' for i in range(n_channels) for j in range(i+1, n_channels)]
    coh_names = [f'coh_{band}_ch{i}_ch{j}' for band in ['theta', 'alpha', 'beta'] for i in range(n_channels) for j in range(i+1, n_channels)]
    feature_names.extend(microstate_names)
    feature_names.extend(corr_names)
    feature_names.extend(coh_names)

    expected_feature_count = 14 * n_channels + 10 + (n_channels * (n_channels - 1) // 2) + (3 * (n_channels * (n_channels - 1) // 2))
    if len(feature_names) != expected_feature_count:
        raise ValueError(f"Feature names length mismatch: {len(feature_names)} vs {expected_feature_count}")

    print("Selecting features with MO-DTW-AS-SHAP-XGRFE...")
    selected_features_list, X_train_val_selected_list, y_train_val_list, X_test_selected_list, y_test_list = select_features(X_train_val_features, y_train_val, feature_names)

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_results = []

    for fold, (X_train_val_selected, y_train_val, X_test_selected, y_test) in enumerate(zip(X_train_val_selected_list, y_train_val_list, X_test_selected_list, y_test_list)):
        print(f"\nCross-validation fold {fold+1}/3")

        # CatBoost
        cat_param_dist = {
            'iterations': [100, 200, 300],
            'depth': [4, 6, 8],
            'learning_rate': [0.01, 0.05, 0.1],
            'l2_leaf_reg': [1, 3, 5],
            'border_count': [32, 64, 128]
        }
        cat_base = CatBoostClassifier(random_state=42, verbose=0, class_weights=class_weights)
        cat_search = RandomizedSearchCV(cat_base, cat_param_dist, n_iter=20, cv=3, scoring='f1', random_state=42, n_jobs=-1)
        cat_search.fit(X_train_val_selected, y_train_val)
        cat_model = cat_search.best_estimator_

        # SVM
        svm_param_dist = {
            'C': [0.1, 1, 10],
            'gamma': ['scale', 'auto', 0.1],
            'kernel': ['rbf', 'linear']
        }
        svm_base = SVC(random_state=42, class_weight=class_weights, probability=True)
        svm_search = RandomizedSearchCV(svm_base, svm_param_dist, n_iter=20, cv=3, scoring='f1', random_state=42, n_jobs=-1)
        svm_search.fit(X_train_val_selected, y_train_val)
        svm_model = svm_search.best_estimator_

        # Gradient Boosting
        gb_param_dist = {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.05, 0.1],
            'subsample': [0.8, 0.9, 1.0]
        }
        gb_base = GradientBoostingClassifier(random_state=42)
        gb_search = RandomizedSearchCV(gb_base, gb_param_dist, n_iter=20, cv=3, scoring='f1', random_state=42, n_jobs=-1)
        gb_search.fit(X_train_val_selected, y_train_val)
        gb_model = gb_search.best_estimator_

        # Voting Classifier
        voting_model = VotingClassifier(
            estimators=[('cat', cat_model), ('svm', svm_model), ('gb', gb_model)],
            voting='soft',
            weights=[0.4, 0.3, 0.3]
        )
        voting_model.fit(X_train_val_selected, y_train_val)

        # Threshold tuning with Youden’s J
        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        best_threshold = 0.5
        best_youden = 0
        for train_idx, val_idx in inner_cv.split(X_train_val_selected, y_train_val):
            X_train_inner, X_val_inner = X_train_val_selected[train_idx], X_train_val_selected[val_idx]
            y_train_inner, y_val_inner = y_train_val[train_idx], y_train_val[val_idx]
            voting_model.fit(X_train_inner, y_train_inner)
            y_scores = voting_model.predict_proba(X_val_inner)[:, 1]
            fpr, tpr, thresholds = roc_curve(y_val_inner, y_scores)
            youden_j = tpr + (1 - fpr) - 1
            idx = np.argmax(youden_j)
            if youden_j[idx] > best_youden:
                best_youden = youden_j[idx]
                best_threshold = thresholds[idx]
        print(f"Fold {fold+1} Best Threshold (Youden’s J): {best_threshold:.2f}")

        # Evaluate models
        cat_val_sens, cat_val_spec, cat_val_f1, cat_val_acc, _ = evaluate_metrics(cat_model, X_train_val_selected, y_train_val, threshold=best_threshold)
        vote_val_sens, vote_val_spec, vote_val_f1, vote_val_acc, _ = evaluate_metrics(voting_model, X_train_val_selected, y_train_val, threshold=best_threshold)
        cat_test_sens, cat_test_spec, cat_test_f1, cat_test_acc, y_pred_test_cat = evaluate_metrics(cat_model, X_test_selected, y_test, threshold=best_threshold)
        vote_test_sens, vote_test_spec, vote_test_f1, vote_test_acc, y_pred_test_vote = evaluate_metrics(voting_model, X_test_selected, y_test, threshold=best_threshold)

        print(f"Fold {fold+1} CatBoost Val Metrics: Sensitivity={cat_val_sens:.4f}, Specificity={cat_val_spec:.4f}, F1={cat_val_f1:.4f}, Accuracy={cat_val_acc:.4f}")
        print(f"Fold {fold+1} Voting Val Metrics: Sensitivity={vote_val_sens:.4f}, Specificity={vote_val_spec:.4f}, F1={vote_val_f1:.4f}, Accuracy={vote_val_acc:.4f}")
        print(f"Fold {fold+1} CatBoost Test Metrics: Sensitivity={cat_test_sens:.4f}, Specificity={cat_test_spec:.4f}, F1={cat_test_f1:.4f}, Accuracy={cat_test_acc:.4f}")
        print(f"Fold {fold+1} Voting Test Metrics: Sensitivity={vote_test_sens:.4f}, Specificity={vote_test_spec:.4f}, F1={vote_test_f1:.4f}, Accuracy={vote_test_acc:.4f}")

        fold_results.append({
            'fold': fold + 1,
            'cat_val': (cat_val_sens, cat_val_spec, cat_val_f1, cat_val_acc),
            'vote_val': (vote_val_sens, vote_val_spec, vote_val_f1, vote_val_acc),
            'cat_test': (cat_test_sens, cat_test_spec, cat_test_f1, cat_test_acc),
            'vote_test': (vote_test_sens, vote_test_spec, vote_test_f1, vote_test_acc),
            'y_pred_test_cat': y_pred_test_cat,
            'y_pred_test_vote': y_pred_test_vote,
            'y_test': y_test
        })

    # Average metrics
    for model in ['cat', 'vote']:
        val_sens = np.mean([r[f'{model}_val'][0] for r in fold_results])
        val_spec = np.mean([r[f'{model}_val'][1] for r in fold_results])
        val_f1 = np.mean([r[f'{model}_val'][2] for r in fold_results])
        val_acc = np.mean([r[f'{model}_val'][3] for r in fold_results])
        test_sens = np.mean([r[f'{model}_test'][0] for r in fold_results])
        test_spec = np.mean([r[f'{model}_test'][1] for r in fold_results])
        test_f1 = np.mean([r[f'{model}_test'][2] for r in fold_results])
        test_acc = np.mean([r[f'{model}_test'][3] for r in fold_results])
        print(f"\n{model.upper()} Model Average Metrics:")
        print(f"Validation - Sensitivity: {val_sens:.4f}, Specificity: {val_spec:.4f}, F1: {val_f1:.4f}, Accuracy: {val_acc:.4f}")
        print(f"Test - Sensitivity: {test_sens:.4f}, Specificity: {test_spec:.4f}, F1: {test_f1:.4f}, Accuracy: {test_acc:.4f}")

    # SHAP and Visualizations (last fold)
    X_train_val_selected = X_train_val_selected_list[-1]
    y_train_val = y_train_val_list[-1]
    X_test_selected = X_test_selected_list[-1]
    y_test = y_test_list[-1]
    selected_features = selected_features_list[-1]

    print("\nComputing SHAP values for CatBoost...")
    cat_model.fit(X_train_val_selected, y_train_val)
    explainer_cat = shap.TreeExplainer(cat_model)
    X_test_subset = X_test_selected[:100] if X_test_selected.shape[0] > 100 else X_test_selected
    shap_values_cat = explainer_cat.shap_values(X_test_subset)

    plt.figure()
    shap.summary_plot(shap_values_cat, X_test_subset, feature_names=selected_features, plot_type="dot", max_display=20, show=False)
    plt.title("SHAP Summary Plot (CatBoost)")
    plt.show()
    plt.close()

    plt.figure()
    shap.summary_plot(shap_values_cat, X_test_subset, feature_names=selected_features, plot_type="bar", max_display=20, show=False)
    plt.title("Mean SHAP Values (CatBoost)")
    plt.show()
    plt.close()

    print("\nComputing SHAP values for Voting Classifier...")
    voting_model.fit(X_train_val_selected, y_train_val)
    explainer_vote = shap.KernelExplainer(lambda x: voting_model.predict_proba(x)[:, 1], X_train_val_selected[:100])
    shap_values_vote = explainer_vote.shap_values(X_test_subset, nsamples=100)

    plt.figure()
    shap.summary_plot(shap_values_vote, X_test_subset, feature_names=selected_features, plot_type="dot", max_display=20, show=False)
    plt.title("SHAP Summary Plot (Voting Classifier)")
    plt.show()
    plt.close()

    plt.figure()
    shap.summary_plot(shap_values_vote, X_test_subset, feature_names=selected_features, plot_type="bar", max_display=20, show=False)
    plt.title("Mean SHAP Values (Voting Classifier)")
    plt.show()
    plt.close()

    # Confusion Matrix (Voting)
    y_pred_test_vote = fold_results[-1]['y_pred_test_vote']
    cm = confusion_matrix(y_test, y_pred_test_vote)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Control', 'ADHD'], yticklabels=['Control', 'ADHD'])
    plt.title("Confusion Matrix (Voting Classifier - Test Set)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()
    plt.close()

    # ROC-AUC Curve (Voting)
    y_scores = voting_model.predict_proba(X_test_selected)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC-AUC Curve (Voting Classifier - Test Set)')
    plt.legend(loc="lower right")
    plt.show()
    plt.close()

    print("\nTop 10 Selected Features:")
    for i, feature in enumerate(selected_features[:min(10, len(selected_features))]):
        print(f"{i+1}. {feature}")
    print_memory_usage()

if __name__ == "__main__":
    main()

Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1: 100%|█| 30/30 [00:01<00:
Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part2/ADHD_part2: 100%|█| 31/31 [00:01<00:
Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part1/Control_part1: 100%|█| 30/30 [00:
Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part2: 100%|█| 30/30 [00:01<00:00, 24.0


Loaded 8399 segments from 121 patients
Preprocessing training data...


KeyboardInterrupt: 

In [48]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
from scipy.io import loadmat
from scipy import signal
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, RFE, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, f1_score, roc_curve, auc, accuracy_score
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from catboost import CatBoostClassifier
from imblearn.combine import SMOTEENN
import xgboost as xgb
from scipy.signal import hilbert, find_peaks, iirnotch
from scipy.stats import wilcoxon, t, kurtosis
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
import mne
from dtaidistance import dtw
from sklearn.cluster import KMeans
import psutil
import time
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pandas as pd
from tqdm import tqdm
import warnings
import itertools
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.cluster._kmeans")

# Helper Functions
def print_memory_usage():
    """Print current memory usage in MB."""
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024 / 1024:.2f} MB")

def compute_confidence_interval(data, confidence=0.95):
    """Compute 95% confidence interval for a metric."""
    n = len(data)
    mean = np.mean(data)
    std = np.std(data, ddof=1)
    margin = t.ppf((1 + confidence) / 2, df=n-1) * std / np.sqrt(n)
    return mean, (mean - margin, mean + margin)

def mcnemar_test(y_true, y_pred1, y_pred2):
    """Perform McNemar's test for paired predictions."""
    table = np.zeros((2, 2))
    for true, pred1, pred2 in zip(y_true, y_pred1, y_pred2):
        if pred1 == true and pred2 == true:
            table[0, 0] += 1
        elif pred1 == true and pred2 != true:
            table[0, 1] += 1
        elif pred1 != true and pred2 == true:
            table[1, 0] += 1
        else:
            table[1, 1] += 1
    result = mcnemar(table, exact=True)
    return result.pvalue

def compute_vif(X):
    """Compute Variance Inflation Factor to detect collinearity."""
    vif_data = pd.DataFrame()
    vif_data["Feature"] = range(X.shape[1])
    vif_data["VIF"] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    return vif_data

def build_cnn_model(input_shape):
    """Build CNN model with dropout to prevent overfitting."""
    model = Sequential([
        Conv1D(64, kernel_size=3, activation='relu', input_shape=input_shape),
        MaxPooling1D(pool_size=2),
        Conv1D(32, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def apply_ica(data, fs=128, n_components=None, method='weighted', weights=None):
    """Apply ICA with weighted (kurtosis, variance, peak, Fp2, frequency) or standard method."""
    start_time = time.time()
    print(f"Applying ICA (method={method}) with input shape={data.shape}")
    if data.ndim == 2:
        data = data[np.newaxis, :, :]
    elif data.ndim != 3:
        raise ValueError(f"Expected 3D data, got {data.ndim}D")
    n_segments, n_samples, n_channels = data.shape
    if n_channels != 19:
        raise ValueError(f"Expected 19 channels, got {n_channels}")
    if n_components is None or n_components > n_channels:
        n_components = min(19, n_channels)
    print(f"Using n_components={n_components}")
    if n_components < 1:
        print("Warning: n_components < 1, skipping ICA exclusion")
        return data, data
    total_samples = n_segments * n_samples
    concatenated_data = data.reshape(total_samples, n_channels)
    ch_names = [f"EEG {i}" for i in range(n_channels)]
    montage_labels = ['EEG1', 'EEG2', 'EEG3', 'EEG4', 'EEG5', 'EEG6', 'EEG7', 'EEG8', 'EEG9', 'EEG10', 'EEG11', 'EEG12', 'EEG13', 'EEG14', 'EEG15', 'EEG16', 'EEG17', 'EEG18', 'EEG19']
    info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types="eeg")
    raw = mne.io.RawArray(concatenated_data.T, info)
    channel_mapping = {f"EEG {i}": montage_labels[i] for i in range(n_channels)}
    raw.rename_channels(channel_mapping)
    try:
        theta_values = [-18, 18, -54, -39, 0, 39, 54, -90, -90, 90, 90, 90, -126, -141, 180, 141, 126, -162, 162]
        radius_values_cm = [51.3, 51.3, 51.3, 33.3, 25.6, 33.3, 51.3, 51.3, 25.6, 0, 25.6, 51.3, 51.3, 33.3, 25.6, 33.3, 51.3, 51.3, 51.3]
        radius_values_m = [r / 100 for r in radius_values_cm]
        ch_pos = {label: [radius * np.cos(np.deg2rad(theta)), radius * np.sin(np.deg2rad(theta)), 0.0]
                  for label, theta, radius in zip(montage_labels, theta_values, radius_values_m)}
        montage = mne.channels.make_dig_montage(ch_pos, coord_frame='head')
        raw.set_montage(montage)
    except Exception:
        print("Warning: Failed to set custom montage, using standard_1020")
        montage = mne.channels.make_standard_montage('standard_1020')
        raw.set_montage(montage)
    raw.filter(l_freq=0.5, h_freq=45, method='iir', iir_params=dict(order=4, ftype='butter'))
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=42, max_iter=500)
    ica.fit(raw)
    ica_sources = ica.get_sources(raw).get_data().T
    kurtosis_values = np.array([kurtosis(ica_sources[:, i]) for i in range(n_components)])
    variance_values = np.var(ica_sources, axis=0)
    peak_amplitude = np.max(np.abs(ica_sources), axis=0)
    fp2_weight = np.mean(np.abs(ica_sources[:, 1])) / np.mean(np.abs(ica_sources)) if n_channels > 1 else 0
    nperseg = min(total_samples, 512)
    f, psd = signal.welch(ica_sources.T, fs=fs, nperseg=nperseg, axis=1)
    artifact_band = (f >= 45) & (f <= 60)
    artifact_power = np.mean(psd[:, artifact_band], axis=1) + 1e-10
    print(f"Component Metrics:")
    print(f"Kurtosis: {kurtosis_values}")
    print(f"Variance: {variance_values}")
    print(f"Peak Amplitude: {peak_amplitude}")
    print(f"Fp2 Weight: {fp2_weight}")
    print(f"Artifact Power (45-60 Hz): {artifact_power}")
    if method == 'weighted':
        if weights is None:
            weights = {'kurtosis': 0.45, 'variance': 0.15, 'peak': 0.15, 'fp2': 0.10, 'artifact': 0.15}
        print(f"Using weights: {weights}")
        kurtosis_norm = (kurtosis_values - kurtosis_values.min()) / (kurtosis_values.max() - kurtosis_values.min() + 1e-10)
        variance_norm = (variance_values - variance_values.min()) / (variance_values.max() - variance_values.min() + 1e-10)
        peak_norm = (peak_amplitude - peak_amplitude.min()) / (peak_amplitude.max() - peak_amplitude.min() + 1e-10)
        artifact_norm = (artifact_power - artifact_power.min()) / (artifact_power.max() - artifact_power.min() + 1e-10)
        combined_score = (weights['kurtosis'] * kurtosis_norm +
                         weights['variance'] * variance_norm +
                         weights['peak'] * peak_norm +
                         weights['fp2'] * fp2_weight +
                         weights['artifact'] * artifact_norm)
        print(f"Combined score: {combined_score}")
        score_threshold = np.percentile(combined_score, 90)
        top_artifact_indices = np.where(combined_score > score_threshold)[0].tolist()
        if not top_artifact_indices or len(top_artifact_indices) < 2:
            top_artifact_indices = np.argsort(combined_score)[-min(2, n_components):].tolist()
        ica.exclude = top_artifact_indices
        print(f"Weighted ICA: Excluding {len(ica.exclude)} components, indices={ica.exclude}")
        print(f"Excluded Kurtosis: {[kurtosis_values[i] for i in ica.exclude]}")
        print(f"Excluded Variance: {[variance_values[i] for i in ica.exclude]}")
        print(f"Excluded Peak: {[peak_amplitude[i] for i in ica.exclude]}")
        print(f"Excluded Artifact Power: {[artifact_power[i] for i in ica.exclude]}")
    else:
        score_threshold = np.percentile(kurtosis_values, 90)
        top_artifact_indices = np.where(kurtosis_values > score_threshold)[0].tolist()
        if not top_artifact_indices:
            top_artifact_indices = np.argsort(kurtosis_values)[-min(2, n_components):].tolist()
        ica.exclude = top_artifact_indices
        print(f"Standard ICA: Excluding {len(ica.exclude)} components, indices={ica.exclude}, kurtosis values={[kurtosis_values[i] for i in ica.exclude]}")
    raw_clean = ica.apply(raw.copy())
    cleaned_data_raw = raw_clean.get_data()
    cleaned_data = cleaned_data_raw.T.reshape(n_segments, n_samples, n_channels)
    print(f"ICA ({method}): Time={time.time()-start_time:.2f}s, Output shape={cleaned_data.shape}")
    return cleaned_data, data

def compare_ica_methods(X_raw, X_cleaned_ica, X_cleaned_weighted, fs=128):
    """Compare standard ICA and weighted ICA using SNR in multiple EEG bands."""
    nperseg = min(X_raw.shape[1], 512)
    print(f"Input shapes: X_raw={X_raw.shape}, X_cleaned_ica={X_cleaned_ica.shape}, X_cleaned_weighted={X_cleaned_weighted.shape}")
    
    scale_factor = 1.0
    X_raw = X_raw * scale_factor
    X_cleaned_ica = X_cleaned_ica * scale_factor
    X_cleaned_weighted = X_cleaned_weighted * scale_factor
    
    f, psd_raw = signal.welch(X_raw, fs=fs, nperseg=nperseg, axis=1, scaling='density')
    _, psd_ica = signal.welch(X_cleaned_ica, fs=fs, nperseg=nperseg, axis=1, scaling='density')
    _, psd_weighted = signal.welch(X_cleaned_weighted, fs=fs, nperseg=nperseg, axis=1, scaling='density')
    
    signal_bands = [(1, 4), (4, 8), (8, 13), (13, 30)]
    noise_band = (45, 60)
    
    signal_power_ica = 0
    signal_power_weighted = 0
    band_weights = [0.2, 0.3, 0.3, 0.2]
    for (low, high), w in zip(signal_bands, band_weights):
        band_mask = (f >= low) & (f <= high)
        signal_power_ica += np.mean(psd_ica[:, band_mask, :], axis=1) * w
        signal_power_weighted += np.mean(psd_weighted[:, band_mask, :], axis=1) * w
    signal_power_ica = signal_power_ica + 1e-10
    signal_power_weighted = signal_power_weighted + 1e-10
    
    noise_mask = (f >= noise_band[0]) & (f <= noise_band[1])
    noise_power_ica = np.mean(psd_ica[:, noise_mask, :], axis=1) + 1e-9
    noise_power_weighted = np.mean(psd_weighted[:, noise_mask, :], axis=1) + 1e-9
    
    print(f"Standard ICA: signal_power (mean)={np.mean(signal_power_ica):.4e}, noise_power (mean)={np.mean(noise_power_ica):.4e}")
    print(f"Standard ICA: signal_power (Fp2)={np.mean(signal_power_ica[:, 1]):.4e}")
    print(f"Weighted ICA: signal_power (mean)={np.mean(signal_power_weighted):.4e}, noise_power (mean)={np.mean(noise_power_weighted):.4e}")
    print(f"Weighted ICA: signal_power (Fp2)={np.mean(signal_power_weighted[:, 1]):.4e}")
    
    snr_ica = 10 * np.log10(np.clip(signal_power_ica / noise_power_ica, 1e-10, 1e10))
    snr_weighted = 10 * np.log10(np.clip(signal_power_weighted / noise_power_weighted, 1e-10, 1e10))
    
    snr_ica_mean = np.mean(snr_ica)
    snr_weighted_mean = np.mean(snr_weighted)
    
    print(f"Standard ICA SNR (mean): {snr_ica_mean:.4f} dB")
    print(f"Weighted ICA SNR (mean): {snr_weighted_mean:.4f} dB")
    return snr_ica_mean, snr_weighted_mean

def find_optimal_weights(X, X_raw, fs=128, n_components=19):
    """Perform random search to find optimal weights for Weighted ICA, maximizing SNR and difference over Standard ICA."""
    print("Starting random search for optimal weights...")
    start_time = time.time()
    
    n_iterations = 30
    best_snr_weighted = -np.inf
    best_snr_diff = -np.inf
    best_weights = None
    results = []
    
    for i in tqdm(range(n_iterations), desc="Random search progress"):
        weights = {
            'kurtosis': np.random.uniform(0.4, 0.5),
            'variance': np.random.uniform(0.1, 0.2),
            'peak': np.random.uniform(0.15, 0.25),
            'fp2': np.random.uniform(0.05, 0.15),
            'artifact': np.random.uniform(0.15, 0.25)
        }
        total = sum(weights.values())
        weights = {k: v / total for k, v in weights.items()}
        try:
            X_cleaned_weighted, _ = apply_ica(X, fs=fs, n_components=n_components, method='weighted', weights=weights)
            X_cleaned_ica, _ = apply_ica(X, fs=fs, n_components=n_components, method='standard')
            snr_ica, snr_weighted = compare_ica_methods(X_raw, X_cleaned_ica, X_cleaned_weighted, fs=fs)
            snr_diff = snr_weighted - snr_ica
            results.append((weights, snr_weighted, snr_diff))
            if snr_weighted > best_snr_weighted and snr_diff > best_snr_diff:
                best_snr_weighted = snr_weighted
                best_snr_diff = snr_diff
                best_weights = weights
                print(f"New best composite score: {snr_weighted:.4f} (SNR: {snr_weighted:.4f} dB, Diff: {snr_diff:.4f} dB) with weights {best_weights}")
        except Exception as e:
            print(f"Iteration {i+1} failed: {str(e)}")
            continue
    
    print(f"Random search completed in {time.time()-start_time:.2f}s")
    print(f"Best weights: {best_weights}")
    print(f"Best Weighted ICA SNR: {best_snr_weighted:.4f} dB")
    print(f"Best SNR difference (Weighted - Standard): {best_snr_diff:.4f} dB")
    return best_weights, best_snr_weighted, best_snr_diff

def load_and_segment_file(file_path, window_size=512, stride=256, expected_channels=19):
    """Load and segment EEG data from a .mat file."""
    print(f"Loading file: {file_path}")
    try:
        mat_data = loadmat(file_path)
        var_names = [k for k in mat_data.keys() if not k.startswith('__')]
        if not var_names:
            print(f"Error: No non-metadata variables found in {file_path}")
            return np.array([])
        possible_var_names = ['data', 'eeg', 'signal', 'EEG', 'Data'] + var_names
        data = None
        for var_name in possible_var_names:
            if var_name in mat_data:
                data = mat_data[var_name]
                print(f"Found variable '{var_name}' in {file_path}")
                break
        if data is None:
            print(f"Error: No valid EEG data variable found in {file_path}")
            return np.array([])
        if data.ndim == 1:
            data = data.reshape(1, -1, 1)
        elif data.ndim == 2:
            data = data[np.newaxis, :, :]
        elif data.ndim == 3:
            pass
        else:
            print(f"Error: Unexpected data dimensions {data.ndim} in {file_path}")
            return np.array([])
        n_segments, n_samples, n_channels = data.shape
        print(f"Data shape: {data.shape}")
        if n_channels != expected_channels:
            if n_channels < expected_channels:
                data = np.pad(data, ((0, 0), (0, 0), (0, expected_channels - n_channels)), mode='constant')
                print(f"Padded channels from {n_channels} to {expected_channels}")
            else:
                data = data[:, :, :expected_channels]
                print(f"Truncated channels from {n_channels} to {expected_channels}")
        max_abs_value = np.max(np.abs(data))
        print(f"Max absolute value: {max_abs_value:.2f}")
        if max_abs_value > 1000:
            data = data / 1000.0
            print(f"Scaled data by 1/1000 (max_abs={max_abs_value:.2f})")
        if max_abs_value > 100:
            scaling_factor = 100 / max_abs_value
            data = data * scaling_factor
            print(f"Scaled data by {scaling_factor:.4f} (max_abs={max_abs_value:.2f})")
        all_segments = []
        for seg in range(n_segments):
            seg_data = data[seg]
            seg_samples = seg_data.shape[0]
            if seg_samples < window_size:
                padded_data = np.pad(seg_data, ((0, window_size - seg_samples), (0, 0)), mode='constant')
                all_segments.append(padded_data[np.newaxis, :, :])
                print(f"Padded segment {seg} from {seg_samples} to {window_size} samples")
                continue
            n_seg_segments = ((seg_samples - window_size + stride - 1) // stride) + 1
            segments = np.zeros((n_seg_segments, window_size, n_channels))
            for i in range(n_seg_segments):
                start = i * stride
                end = min(start + window_size, seg_samples)
                segment_length = end - start
                if segment_length == window_size:
                    segments[i] = seg_data[start:end, :]
                else:
                    segments[i] = np.pad(seg_data[start:end, :], ((0, window_size - segment_length), (0, 0)), mode='constant')
            all_segments.append(segments)
        if not all_segments:
            print(f"Error: No segments created for {file_path}")
            return np.array([])
        return np.concatenate(all_segments, axis=0)
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")
        return np.array([])

def load_from_folder(folder_path, window_size=512, stride=256, expected_channels=19):
    """Load EEG data from all .mat files in a folder."""
    print(f"Checking folder: {folder_path}")
    if not os.path.exists(folder_path):
        print(f"Error: Folder {folder_path} does not exist")
        return np.array([]), []
    mat_files = [f for f in os.listdir(folder_path) if f.endswith('.mat')]
    if not mat_files:
        print(f"Error: No .mat files found in {folder_path}")
        return np.array([]), []
    print(f"Found {len(mat_files)} .mat files in {folder_path}")
    all_segments = []
    valid_files = []
    for file in tqdm(mat_files, desc=f"Loading files from {folder_path}"):
        file_path = os.path.join(folder_path, file)
        segments = load_and_segment_file(file_path, window_size, stride, expected_channels)
        if segments.size > 0:
            all_segments.append(segments)
            valid_files.append(file)
        else:
            print(f"Warning: No valid segments loaded from {file_path}")
    if not all_segments:
        print(f"Error: No valid segments loaded from any file in {folder_path}")
        return np.array([]), []
    return np.concatenate(all_segments, axis=0), valid_files

def load_all_data(window_size=512, stride=256, expected_channels=19):
    """Load ADHD and control EEG data from specified folders."""
    adhd_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1"
    adhd_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part2/ADHD_part2"
    control_path1 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part1/Control_part1"
    control_path2 = "C:/Users/ADMIN/Downloads/Calcium signalling research/Control_part2"
    print("Loading ADHD data...")
    adhd_segments1, adhd_files1 = load_from_folder(adhd_path1, window_size, stride, expected_channels)
    adhd_segments2, adhd_files2 = load_from_folder(adhd_path2, window_size, stride, expected_channels)
    adhd_segments = []
    if adhd_segments1.size > 0:
        adhd_segments.append(adhd_segments1)
        print(f"Loaded {adhd_segments1.shape[0]} ADHD segments from {adhd_path1}")
    if adhd_segments2.size > 0:
        adhd_segments.append(adhd_segments2)
        print(f"Loaded {adhd_segments2.shape[0]} ADHD segments from {adhd_path2}")
    if not adhd_segments:
        print("Warning: No ADHD data loaded")
    else:
        adhd_segments = np.concatenate(adhd_segments, axis=0)
        print(f"Total ADHD segments: {adhd_segments.shape[0]} from {len(adhd_files1) + len(adhd_files2)} files")

    print("Loading Control data...")
    control_segment1, control_files1 = load_from_folder(control_path1, window_size, stride, expected_channels)
    control_segment2, control_files2 = load_from_folder(control_path2, window_size, stride, expected_channels)
    control_segments = []
    if control_segment1.size > 0:
        control_segments.append(control_segment1)
        print(f"Loaded {control_segment1.shape[0]} Control segments from {control_path1}")
    if control_segment2.size > 0:
        control_segments.append(control_segment2)
        print(f"Loaded {control_segment2.shape[0]} Control segments from {control_path2}")
    if not control_segments:
        print("Warning: No Control data loaded")
    else:
        control_segments = np.concatenate(control_segments, axis=0)
        print(f"Total Control segments: {control_segments.shape[0]} from {len(control_files1) + len(control_files2)} files")

    if len(adhd_segments) == 0 and len(control_segments) == 0:
        raise ValueError("Failed to load both ADHD and Control data. Check folder paths and .mat files.")

    y_adhd = np.ones(adhd_segments.shape[0]) if adhd_segments.size > 0 else np.array([])
    y_control = np.zeros(control_segments.shape[0]) if control_segments.size > 0 else np.array([])
    X = np.concatenate([adhd_segments, control_segments], axis=0) if adhd_segments.size > 0 and control_segments.size > 0 else (adhd_segments if adhd_segments.size > 0 else control_segments)
    y = np.concatenate([y_adhd, y_control]) if y_adhd.size > 0 and y_control.size > 0 else (y_adhd if y_adhd.size > 0 else y_control)

    adhd_segment_counts = []
    adhd_patient_ids = []
    valid_adhd_files = []
    for f in adhd_files1 + adhd_files2:
        path = os.path.join(adhd_path1, f) if f in adhd_files1 else os.path.join(adhd_path2, f)
        segments = load_and_segment_file(path, window_size, stride, expected_channels)
        count = segments.shape[0] if segments.size > 0 else 0
        if count > 0:
            adhd_segment_counts.append(count)
            valid_adhd_files.append(f)
            patient_number = f.split('.')[0]
            adhd_patient_ids.extend([patient_number] * count)

    control_segment_counts = []
    control_patient_ids = []
    valid_control_files = []
    for f in control_files1 + control_files2:
        path = os.path.join(control_path1, f) if f in control_files1 else os.path.join(control_path2, f)
        segments = load_and_segment_file(path, window_size, stride, expected_channels)
        count = segments.shape[0] if segments.size > 0 else 0
        if count > 0:
            control_segment_counts.append(count)
            valid_control_files.append(f)
            patient_number = f.split('_')[0]
            control_patient_ids.extend([patient_number] * count)

    patient_ids = adhd_patient_ids + control_patient_ids

    if len(patient_ids) != X.shape[0]:
        print(f"Warning: Patient IDs length ({len(patient_ids)}) does not match number of segments ({X.shape[0]})")
        if X.shape[0] == 0:
            raise ValueError("No segments loaded. Cannot proceed with empty dataset.")
    else:
        print(f"Loaded {X.shape[0]} segments from {len(valid_adhd_files) + len(valid_control_files)} patients")
    return X, y, patient_ids

def notch_filter(data, fs=128, freqs=[50], Q=30):
    """Apply notch filter at 50 Hz to remove powerline noise (Iran standard)."""
    filtered_data = data.copy()
    for freq in freqs:
        b, a = signal.iirnotch(freq, Q, fs)
        filtered_data = signal.filtfilt(b, a, filtered_data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def clip_spikes(data, threshold=500):
    """Clip extreme values to reduce artifacts, with special threshold for noise."""
    clipped_data = data.copy()
    clipped_data[np.abs(clipped_data) > threshold] = np.sign(clipped_data[np.abs(clipped_data) > threshold]) * threshold
    fp2_threshold = 30
    clipped_data[:, :, 1] = np.where(np.abs(clipped_data[:, :, 1]) > fp2_threshold,
                                     np.sign(clipped_data[:, :, 1]) * fp2_threshold,
                                     clipped_data[:, :, 1])
    return clipped_data

def butterworth_filter(data, fs=128, lowcut=0.5, highcut=45, order=4):
    """Apply Butterworth bandpass filter (0.5-45 Hz)."""
    nyquist = fs / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    if low <= 0 or high >= 1:
        raise ValueError(f"Invalid filter frequencies: lowcut={lowcut}, highcut={highcut}, fs={fs}")
    b, a = signal.butter(order, [low, high], btype='band')
    filtered_data = signal.filtfilt(b, a, data, axis=1)
    filtered_data = np.nan_to_num(filtered_data, nan=0.0, posinf=0.0, neginf=0.0)
    return filtered_data

def normalize_data(data):
    """Normalize data per channel."""
    scaler = StandardScaler()
    return scaler.fit_transform(data)

def calculate_engagement_index(data, fs=128, prev_value=None):
    """Calculate engagement index (theta + alpha) / beta."""
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    alpha = calculate_psd(data, fs, freq_range=(8, 13))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    if np.any(np.isnan([theta, alpha, beta])) or beta == 0:
        return prev_value if prev_value is not None else 0
    return np.mean((theta + alpha) / (beta + 1e-10))

def calculate_psd(data, fs=128, freq_range=None):
    """Calculate power spectral density using Welch’s method."""
    nperseg = min(len(data), 512)
    f, psd = signal.welch(data, fs=fs, nperseg=nperseg, axis=0)
    if freq_range:
        freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
        return np.mean(psd[freq_mask], axis=0) if freq_mask.any() else 0
    return psd.mean(axis=0)

def phase_amplitude_coupling(data, fs=128, prev_pac=None):
    """Calculate phase-amplitude coupling for theta-alpha, theta-beta, alpha-beta."""
    theta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 4, 8)[0, :, 0]
    alpha = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 8, 13)[0, :, 0]
    beta = butterworth_filter(data[np.newaxis, :, np.newaxis], fs, 13, 30)[0, :, 0]
    if np.any(np.isnan([theta, alpha, beta])) or np.std([theta, alpha, beta]) == 0:
        return prev_pac if prev_pac is not None else np.zeros(3)
    theta_phase = np.angle(hilbert(theta))
    alpha_phase = np.angle(hilbert(alpha))
    beta_phase = np.angle(hilbert(beta))
    alpha_amp = np.abs(hilbert(alpha))
    beta_amp = np.abs(hilbert(beta))
    pac_theta_alpha = np.abs(np.mean(alpha_amp * np.exp(1j * theta_phase)))
    pac_theta_beta = np.abs(np.mean(beta_amp * np.exp(1j * theta_phase)))
    pac_alpha_beta = np.abs(np.mean(beta_amp * np.exp(1j * alpha_phase)))
    return np.array([pac_theta_alpha, pac_theta_beta, pac_alpha_beta])

def wavelet_features(data, prev_wavelet=None):
    """Extract wavelet features using Mexican Hat wavelet."""
    scales = np.arange(1, 4)
    coeffs, _ = pywt.cwt(data, scales, 'mexh', axis=0)
    if np.any(np.isnan(coeffs)):
        return prev_wavelet if prev_wavelet is not None else np.zeros(6)
    features = []
    for c in coeffs:
        features.extend([np.mean(np.abs(c)), np.max(np.abs(c))])
    return np.array(features)

def theta_beta_ratio(data, fs=128, prev_value=None):
    """Calculate theta/beta ratio."""
    theta = calculate_psd(data, fs, freq_range=(4, 8))
    beta = calculate_psd(data, fs, freq_range=(13, 30))
    if np.any(np.isnan([theta, beta])) or beta == 0:
        return prev_value if prev_value is not None else 0
    return theta / (beta + 1e-10)

def channel_coherence(data, fs=128, prev_coherence=None):
    """Calculate coherence in theta, alpha, beta between channel pairs."""
    n_channels = data.shape[1]
    coherence_features = []
    nperseg = min(data.shape[0], 512)
    expected_length = 3 * (n_channels * (n_channels - 1) // 2)
    for band, freq_range in [('theta', (4, 8)), ('alpha', (8, 13)), ('beta', (13, 30))]:
        for i in range(n_channels):
            for j in range(i + 1, n_channels):
                f, coh = signal.coherence(data[:, i], data[:, j], fs=fs, nperseg=nperseg)
                freq_mask = (f >= freq_range[0]) & (f <= freq_range[1])
                coh_value = np.mean(coh[freq_mask]) if freq_mask.any() else 0
                if np.isnan(coh_value):
                    return prev_coherence if prev_coherence is not None else np.zeros(expected_length)
                coherence_features.append(coh_value)
    return np.array(coherence_features)

def channel_correlation(data, prev_correlation=None):
    """Calculate correlation between channel pairs."""
    n_channels = data.shape[1]
    corr_features = []
    expected_length = n_channels * (n_channels - 1) // 2
    for i in range(n_channels):
        for j in range(i + 1, n_channels):
            corr = np.corrcoef(data[:, i], data[:, j])[0, 1]
            if np.isnan(corr):
                return prev_correlation if prev_correlation is not None else np.zeros(expected_length)
            corr_features.append(corr)
    return np.array(corr_features)

def microstate_analysis(data, fs=128, n_states=4, prev_microstates=None):
    """Perform microstate analysis with K-means clustering."""
    expected_length = n_states + n_states + 2
    gfp = np.std(data, axis=1)
    if np.any(np.isnan(gfp)):
        return prev_microstates if prev_microstates is not None else np.zeros(expected_length)
    peaks, _ = find_peaks(gfp, distance=fs//10)
    if len(peaks) < n_states:
        peak_data = data
    else:
        peak_data = data[peaks]
    kmeans = KMeans(n_clusters=n_states, random_state=42, n_init=10)
    kmeans.fit(peak_data)
    templates = kmeans.cluster_centers_
    distances = np.zeros((data.shape[0], n_states))
    for i, template in enumerate(templates):
        template_norm = np.linalg.norm(template)
        data_norm = np.linalg.norm(data, axis=1)
        distances[:, i] = np.abs(np.sum(data * template, axis=1) / (data_norm * template_norm + 1e-10))
    labels = np.argmax(distances, axis=1)
    durations = np.zeros(n_states)
    gfp_peaks = gfp[peaks] if len(peaks) > 0 else gfp
    gfp_mean = np.mean(gfp_peaks)
    gfp_std = np.std(gfp_peaks)
    for state in range(n_states):
        state_indices = np.where(labels == state)[0]
        if len(state_indices) > 0:
            diff = np.diff(state_indices)
            breaks = np.where(diff > 1)[0]
            segment_lengths = np.diff(np.concatenate([[0], breaks + 1, [len(state_indices)]]))
            durations[state] = np.mean(segment_lengths) / fs * 1000 if len(segment_lengths) > 0 else 0
    coverage = np.bincount(labels, minlength=n_states).astype(float) / len(labels)
    microstate_features = np.hstack([durations, coverage, [gfp_mean, gfp_std]])
    return microstate_features

def extract_features_single(sample, idx, prev_features=None):
    """Extract features for a single EEG sample."""
    n_channels = sample.shape[1]
    normalized = np.zeros_like(sample)
    for ch in range(n_channels):
        channel_data = sample[:, ch]
        if np.std(channel_data) > 0:
            normalized[:, ch] = (channel_data - np.mean(channel_data)) / np.std(channel_data)
        else:
            normalized[:, ch] = channel_data
    channel_features = []

    features_per_channel = 14
    total_per_channel = n_channels * features_per_channel
    microstate_len = 10
    corr_len = (n_channels * (n_channels - 1) // 2)
    coh_len = 3 * corr_len
    total_prev_len = total_per_channel + microstate_len + corr_len + coh_len

    if prev_features is not None and len(prev_features) != total_prev_len:
        raise ValueError(f"Expected prev_features length {total_prev_len}, got {len(prev_features)}")

    for ch in range(n_channels):
        channel_data = normalized[:, ch]
        offset = ch * features_per_channel
        prev_values = prev_features[offset:offset + features_per_channel] if prev_features is not None else None

        engagement = calculate_engagement_index(channel_data, prev_value=prev_values[0] if prev_values is not None else None)
        theta = calculate_psd(channel_data, freq_range=(4, 8))
        alpha = calculate_psd(channel_data, freq_range=(8, 13))
        beta = calculate_psd(channel_data, freq_range=(13, 30))
        pac = phase_amplitude_coupling(channel_data, prev_pac=prev_values[4:7] if prev_values is not None else None)
        wavelet = wavelet_features(channel_data, prev_wavelet=prev_values[7:13] if prev_values is not None else None)
        tbr = theta_beta_ratio(channel_data, prev_value=prev_values[13] if prev_values is not None else None)

        ch_features = np.concatenate([
            [engagement], [theta], [alpha], [beta], pac, wavelet, [tbr]
        ])
        channel_features.append(ch_features)

    per_channel_features = np.array(channel_features).flatten()
    microstates = microstate_analysis(normalized, prev_microstates=prev_features[total_per_channel:total_per_channel + microstate_len] if prev_features is not None else None)
    correlations = channel_correlation(normalized, prev_correlation=prev_features[total_per_channel + microstate_len:total_per_channel + microstate_len + corr_len] if prev_features is not None else None)
    coherences = channel_coherence(normalized, prev_coherence=prev_features[total_per_channel + microstate_len + corr_len:] if prev_features is not None else None)

    sample_features = np.concatenate([per_channel_features, microstates, correlations, coherences])
    return sample_features

def extract_all_features(data, chunk_size=100):
    """Extract features for all EEG samples with a single progress bar."""
    start_time = time.time()
    n_samples, n_samples_per_segment, n_channels = data.shape
    feature_matrix = []
    prev_features = None

    with tqdm(total=n_samples, desc="Extracting features", unit="segment") as pbar:
        for i in range(n_samples):
            sample_features = extract_features_single(data[i], i, prev_features)
            feature_matrix.append(sample_features)
            prev_features = sample_features
            pbar.update(1)

    feature_matrix = np.array(feature_matrix)
    print(f"Feature matrix shape: {feature_matrix.shape}, Time={time.time()-start_time:.2f}s")
    return feature_matrix

def remove_highly_correlated_features(X, feature_names, threshold=0.85):
    """Remove features with correlation above threshold."""
    corr_matrix = np.corrcoef(X.T)
    to_remove = set()
    for i in range(len(corr_matrix)):
        for j in range(i + 1, len(corr_matrix)):
            if abs(corr_matrix[i, j]) > threshold:
                to_remove.add(j)
    keep_indices = [i for i in range(X.shape[1]) if i not in to_remove]
    return X[:, keep_indices], np.array(feature_names)[keep_indices].tolist()

def select_features_mrs_shap(X, y, feature_names, n_features_to_select=0.6, min_window_size=50, use_dtw=True, use_shap=True):
    """MRS-SHAP feature selection with DTW, SHAP, and mutual information."""
    start_time = time.time()
    process = psutil.Process()
    mem_before = process.memory_info().rss / 1024**2

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

    smote_enn = SMOTEENN(random_state=42, sampling_strategy='auto', enn=dict(n_neighbors=5))
    X_balanced, y_balanced = smote_enn.fit_resample(X_scaled, y)
    print(f"After SMOTE-ENN (k=5): {X_balanced.shape[0]} samples, Class distribution: {np.bincount(y_balanced)}")

    selector = VarianceThreshold(threshold=0.01)
    X_reduced = selector.fit_transform(X_balanced)
    mask = selector.get_support()
    feature_names_reduced = np.array(feature_names)[mask].tolist()
    print(f"After Variance Thresholding: {X_reduced.shape[1]} features")

    X_reduced, feature_names_reduced = remove_highly_correlated_features(X_reduced, feature_names_reduced, threshold=0.85)
    print(f"After Correlation Filtering: {X_reduced.shape[1]} features")

    mi_scores = mutual_info_classif(X_reduced, y_balanced, random_state=42)
    mi_threshold = np.percentile(mi_scores, 20)
    mi_mask = mi_scores > mi_threshold
    X_reduced = X_reduced[:, mi_mask]
    feature_names_reduced = [feature_names_reduced[i] for i in range(len(mi_mask)) if mi_mask[i]]
    print(f"After Mutual Information Filtering: {X_reduced.shape[1]} features")

    vif_data = compute_vif(X_reduced)
    low_vif_mask = vif_data['VIF'] < 10
    X_reduced = X_reduced[:, low_vif_mask]
    feature_names_reduced = [feature_names_reduced[i] for i in range(len(low_vif_mask)) if low_vif_mask[i]]
    print(f"After VIF Filtering: {X_reduced.shape[1]} features")

    xgb_model = xgb.XGBClassifier(random_state=42, n_estimators=100, eval_metric='logloss', reg_lambda=0.1)
    xgb_model.fit(X_reduced, y_balanced)
    shap_importance = np.ones(X_reduced.shape[1])
    if use_shap:
        explainer = shap.TreeExplainer(xgb_model)
        shap_values = explainer.shap_values(X_reduced)
        shap_importance = np.abs(shap_values).mean(axis=0)

    dtw_scores = np.zeros(X_reduced.shape[1])
    if use_dtw:
        for feature_idx in range(X_reduced.shape[1]):
            feature_series = X_reduced[:, feature_idx]
            if np.any(np.isnan(feature_series)) or np.std(feature_series) == 0:
                dtw_scores[feature_idx] = 0
                continue
            sub_windows = [feature_series[i:i+min_window_size] for i in range(0, len(feature_series)-min_window_size, min_window_size)]
            if len(sub_windows) < 2:
                dtw_scores[feature_idx] = 0
                continue
            try:
                dtw_sum = 0
                count = 0
                for i in range(len(sub_windows)):
                    for j in range(i+1, len(sub_windows)):
                        dist = dtw.distance(sub_windows[i], sub_windows[j])
                        if np.isnan(dist):
                            continue
                        dtw_sum += dist
                        count += 1
                dtw_scores[feature_idx] = dtw_sum / count if count > 0 else 0
            except Exception as e:
                print(f"Error in DTW computation for feature {feature_idx}: {e}")
                dtw_scores[feature_idx] = 0

    shap_norm = (shap_importance - shap_importance.min()) / (shap_importance.max() - shap_importance.min() + 1e-10)
    dtw_norm = (dtw_scores - dtw_scores.min()) / (dtw_scores.max() - dtw_scores.min() + 1e-10)
    mo_score = 0.5 * shap_norm + 0.3 * dtw_norm if use_dtw and use_shap else (shap_norm if use_shap else dtw_norm)

    feature_types = []
    for fname in feature_names_reduced:
        if any(k in fname for k in ['theta', 'alpha', 'beta', 'engagement', 'theta_beta_ratio']):
            feature_types.append('spectral')
        elif any(k in fname for k in ['pac', 'wavelet']):
            feature_types.append('nonlinear')
        elif any(k in fname for k in ['corr', 'coh', 'microstate', 'gfp']):
            feature_types.append('global')
        else:
            feature_types.append('other')
    feature_types = np.array(feature_types)

    diversity_penalty = 0.6
    type_counts = {t: 0 for t in set(feature_types)}
    selected_indices = []
    n_select = max(1, int(n_features_to_select * X_reduced.shape[1]))
    for _ in range(n_select):
        adjusted_scores = mo_score.copy()
        for idx in selected_indices:
            if idx in range(len(feature_types)):
                type_counts[feature_types[idx]] += 1
        for idx in range(len(adjusted_scores)):
            if idx in selected_indices:
                adjusted_scores[idx] = -np.inf
            else:
                type_count = type_counts[feature_types[idx]]
                adjusted_scores[idx] -= diversity_penalty * type_count
        best_idx = np.argmax(adjusted_scores)
        selected_indices.append(best_idx)

    selected_features = [feature_names_reduced[idx] for idx in selected_indices]
    selected_indices_orig = [feature_names.index(f) for f in selected_features]
    X_selected = X[:, selected_indices_orig]

    selected_types = [feature_types[idx] for idx in selected_indices]
    type_distribution = {t: selected_types.count(t) for t in set(selected_types)}
    print("\nDistribution of Selected Feature Types:")
    for t, count in type_distribution.items():
        print(f"{t}: {count} features")

    mem_after = process.memory_info().rss / 1024**2
    print(f"MRS-SHAP: Time={time.time()-start_time:.2f}s, Memory={mem_after-mem_before:.2f}MB")
    return selected_features, X_selected, y

def select_features_standard(X, y, feature_names, method='rfe', n_features_to_select=0.6):
    """Standard feature selection using PCA, RFE, or ANOVA."""
    start_time = time.time()
    process = psutil.Process()
    mem_before = process.memory_info().rss / 1024**2

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

    smote_enn = SMOTEENN(random_state=42, sampling_strategy='auto', enn=dict(n_neighbors=5))
    X_balanced, y_balanced = smote_enn.fit_resample(X_scaled, y)

    selector = VarianceThreshold(threshold=0.01)
    X_reduced = selector.fit_transform(X_balanced)
    mask = selector.get_support()
    feature_names_reduced = np.array(feature_names)[mask].tolist()

    if method == 'pca':
        n_features = max(1, int(n_features_to_select * X_reduced.shape[1]))
        pca = PCA(n_components=n_features, random_state=42)
        X_reduced = pca.fit_transform(X_reduced)
        selected_features = feature_names_reduced[:n_features]
    elif method == 'rfe':
        estimator = xgb.XGBClassifier(random_state=42, n_estimators=100, eval_metric='logloss', reg_lambda=0.1)
        n_features = max(1, int(n_features_to_select * X_reduced.shape[1]))
        selector = RFE(estimator, n_features_to_select=n_features, step=1)
        selector.fit(X_reduced, y_balanced)
        selected_mask = selector.support_
        selected_features = [feature_names_reduced[i] for i in range(len(selected_mask)) if selected_mask[i]]
    elif method == 'anova':
        n_features = max(1, int(n_features_to_select * X_reduced.shape[1]))
        selector = SelectKBest(score_func=f_classif, k=n_features)
        selector.fit(X_reduced, y_balanced)
        selected_mask = selector.get_support()
        selected_features = [feature_names_reduced[i] for i in range(len(selected_mask)) if selected_mask[i]]

    selected_indices = [feature_names.index(f) for f in selected_features]
    X_selected = X[:, selected_indices]

    mem_after = process.memory_info().rss / 1024**2
    print(f"{method.upper()}: Time={time.time()-start_time:.2f}s, Memory={mem_after-mem_before:.2f}MB")
    return selected_features, X_selected, y

def evaluate_metrics(model, X, y, threshold=0.5):
    """Evaluate model performance with multiple metrics."""
    if hasattr(model, 'predict_proba'):
        y_scores = model.predict_proba(X)[:, 1]
        y_pred = (y_scores > threshold).astype(int)
    else:
        y_pred = model.predict(X)
        y_scores = model.decision_function(X) if hasattr(model, 'decision_function') else y_pred
    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = f1_score(y, y_pred, zero_division=0)
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    auc_score = auc(*roc_curve(y, y_scores)[:2]) if hasattr(model, 'predict_proba') or hasattr(model, 'decision_function') else 0
    return sensitivity, specificity, f1, accuracy, auc_score, y_pred

def sensitivity_analysis(X, y, feature_names, percentages=[0.4, 0.6, 0.8]):
    """Perform sensitivity analysis for feature retention percentages."""
    results = []
    for p in percentages:
        selected_features, X_selected, y_selected = select_features_mrs_shap(X, y, feature_names, n_features_to_select=p)
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        fold_results = []
        for train_idx, test_idx in skf.split(X_selected, y_selected):
            X_train, X_test = X_selected[train_idx], X_selected[test_idx]
            y_train, y_test = y_selected[train_idx], y_selected[test_idx]
            cat_model = CatBoostClassifier(random_state=42, verbose=0, class_weights={0: 1.0, 1: 2.0})
            cat_model.fit(X_train, y_train)
            metrics = evaluate_metrics(cat_model, X_test, y_test)
            fold_results.append(metrics[:5])
        mean_results = np.mean(fold_results, axis=0)
        results.append((p, mean_results))
    return results

def main():
    """Main function for ADHD classification with MRS-SHAP pipeline."""
    start_time = time.time()
    print("Dataset: Resting-state EEG (Nasrabadi et al., 2021), 61 ADHD + 60 controls, 19 channels, 128 Hz")
    print("Ethics: Approved by Roozbeh Hospital, Tehran")
    window_size = 512
    stride = 256
    expected_channels = 19
    X, y, patient_ids = load_all_data(window_size, stride, expected_channels)
    print(f"Data loading completed: {time.time()-start_time:.2f}s")

    unique_patients = np.unique(patient_ids)
    patient_labels = [y[[i for i, pid in enumerate(patient_ids) if pid == patient][0]] for patient in unique_patients]
    train_val_patients, test_patients = train_test_split(unique_patients, test_size=0.2, stratify=patient_labels, random_state=42)
    train_val_labels = [y[[i for i, pid in enumerate(patient_ids) if pid == patient][0]] for patient in train_val_patients]
    train_patients, val_patients = train_test_split(train_val_patients, test_size=0.25, stratify=train_val_labels, random_state=42)

    train_idx = [i for i, pid in enumerate(patient_ids) if pid in train_patients]
    val_idx = [i for i, pid in enumerate(patient_ids) if pid in val_patients]
    test_idx = [i for i, pid in enumerate(patient_ids) if pid in test_patients]

    X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
    y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

    print("\nPreprocessing data...")
    start_time_pre = time.time()
    X_train = notch_filter(X_train, freqs=[50], Q=30)
    X_train = clip_spikes(X_train)
    X_train = butterworth_filter(X_train)
    
    best_weights, best_snr, weight_results = find_optimal_weights(X_train, X_train, fs=128, n_components=19)
    
    X_train_cleaned, X_train_raw = apply_ica(X_train, method='weighted', weights=best_weights)
    X_train_standard, _ = apply_ica(X_train, method='standard')

    X_val = notch_filter(X_val, freqs=[50], Q=30)
    X_val = clip_spikes(X_val)
    X_val = butterworth_filter(X_val)
    X_val_cleaned, _ = apply_ica(X_val, method='weighted', weights=best_weights)
    X_val_standard, _ = apply_ica(X_val, method='standard')

    X_test = notch_filter(X_test, freqs=[50], Q=30)
    X_test = clip_spikes(X_test)
    X_test = butterworth_filter(X_test)
    X_test_cleaned, _ = apply_ica(X_test, method='weighted', weights=best_weights)
    X_test_standard, _ = apply_ica(X_test, method='standard')

    print(f"Preprocessing completed: {time.time()-start_time_pre:.2f}s")
    print("\nComparing ICA methods...")
    snr_ica, snr_weighted = compare_ica_methods(X_train_raw, X_train_standard, X_train_cleaned)

    print("\nExtracting features...")
    start_time = time.time()
    X_train_features = extract_all_features(X_train_cleaned)
    X_val_features = extract_all_features(X_val_cleaned)
    X_test_features = extract_all_features(X_test_cleaned)
    print(f"Feature extraction completed: {time.time()-start_time:.2f}s")

    n_channels = X.shape[2]
    feature_names = []
    for ch in range(n_channels):
        feature_names.extend([
            f'engagement_ch{ch}', f'theta_ch{ch}', f'alpha_ch{ch}', f'beta_ch{ch}',
            f'pac_theta_alpha_ch{ch}', f'pac_theta_beta_ch{ch}', f'pac_alpha_beta_ch{ch}',
            f'wavelet_mean1_ch{ch}', f'wavelet_max1_ch{ch}',
            f'wavelet_mean2_ch{ch}', f'wavelet_max2_ch{ch}',
            f'wavelet_mean3_ch{ch}', f'wavelet_max3_ch{ch}',
            f'theta_beta_ratio_ch{ch}'
        ])
    microstate_names = [f'microstate_duration_{i}' for i in range(4)] + [f'microstate_coverage_{i}' for i in range(4)] + ['gfp_mean', 'gfp_std']
    corr_names = [f'corr_ch{i}_ch{j}' for i in range(n_channels) for j in range(i+1, n_channels)]
    coh_names = [f'coh_{band}_ch{i}_ch{j}' for band in ['theta', 'alpha', 'beta'] for i in range(n_channels) for j in range(i+1, n_channels)]
    feature_names.extend(microstate_names + corr_names + coh_names)

    expected_feature_count = 14 * n_channels + 10 + (n_channels * (n_channels - 1) // 2) + (3 * (n_channels * (n_channels - 1) // 2))
    if len(feature_names) != expected_feature_count:
        raise ValueError(f"Feature names length mismatch: {len(feature_names)} vs {expected_feature_count}")
    if X_train_features.shape[1] != len(feature_names):
        raise ValueError(f"Feature matrix columns ({X_train_features.shape[1]}) do not match feature names ({len(feature_names)})")

    X_train_val_features = np.vstack([X_train_features, X_val_features])
    y_train_val = np.hstack([y_train, y_val])

    print("\nSelecting features...")
    methods = ['mrs_shap', 'pca', 'rfe', 'anova', 'mrs_shap_no_smote', 'cnn']
    all_results = {method: [] for method in methods}
    all_predictions = {method: [] for method in methods}

    outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_train_val_features, y_train_val)):
        print(f"\nFold {fold+1}/3")
        X_tr, X_val = X_train_val_features[train_idx], X_train_val_features[val_idx]
        y_tr, y_val = y_train_val[train_idx], y_train_val[val_idx]

        # MRS-SHAP
        selected_features, X_tr_selected, y_tr_selected = select_features_mrs_shap(X_tr, y_tr, feature_names)
        X_val_selected = X_val[:, [feature_names.index(f) for f in selected_features]]
        cat_model = CatBoostClassifier(random_state=42, verbose=0, class_weights={0: 1.0, 1: 2.0})
        cat_model.fit(X_tr_selected, y_tr_selected)
        metrics = evaluate_metrics(cat_model, X_val_selected, y_val)
        all_results['mrs_shap'].append(metrics[:5])
        all_predictions['mrs_shap'].append(metrics[5])

        # MRS-SHAP without SMOTE-ENN
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)
        X_tr_scaled = np.nan_to_num(X_tr_scaled, nan=0.0, posinf=0.0, neginf=0.0)
        cat_model.fit(X_tr_scaled, y_tr)
        metrics = evaluate_metrics(cat_model, X_val_scaled, y_val)
        all_results['mrs_shap_no_smote'].append(metrics[:5])
        all_predictions['mrs_shap_no_smote'].append(metrics[5])

        # PCA
        selected_features, X_tr_selected, y_tr_selected = select_features_standard(X_tr, y_tr, feature_names, method='pca')
        X_val_selected = X_val[:, [feature_names.index(f) for f in selected_features]]
        cat_model.fit(X_tr_selected, y_tr_selected)
        metrics = evaluate_metrics(cat_model, X_val_selected, y_val)
        all_results['pca'].append(metrics[:5])
        all_predictions['pca'].append(metrics[5])

        # RFE
        selected_features, X_tr_selected, y_tr_selected = select_features_standard(X_tr, y_tr, feature_names, method='rfe')
        X_val_selected = X_val[:, [feature_names.index(f) for f in selected_features]]
        cat_model.fit(X_tr_selected, y_tr_selected)
        metrics = evaluate_metrics(cat_model, X_val_selected, y_val)
        all_results['rfe'].append(metrics[:5])
        all_predictions['rfe'].append(metrics[5])

        # ANOVA
        selected_features, X_tr_selected, y_tr_selected = select_features_standard(X_tr, y_tr, feature_names, method='anova')
        X_val_selected = X_val[:, [feature_names.index(f) for f in selected_features]]
        cat_model.fit(X_tr_selected, y_tr_selected)
        metrics = evaluate_metrics(cat_model, X_val_selected, y_val)
        all_results['anova'].append(metrics[:5])
        all_predictions['anova'].append(metrics[5])

        # CNN
        cnn_model = build_cnn_model((X_tr.shape[1], X_tr.shape[2]))
        cnn_model.fit(X_tr, y_tr, epochs=10, batch_size=32, verbose=0)
        y_pred = (cnn_model.predict(X_val) > 0.5).astype(int).flatten()
        y_scores = cnn_model.predict(X_val).flatten()
        tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        f1 = f1_score(y_val, y_pred, zero_division=0)
        accuracy = accuracy_score(y_val, y_pred)
        auc_score = auc(*roc_curve(y_val, y_scores)[:2])
        all_results['cnn'].append([sensitivity, specificity, f1, accuracy, auc_score])
        all_predictions['cnn'].append(y_pred)

    # Ablation Studies
    print("\nPerforming ablation studies...")
    ablation_results = []
    for component in ['no_dtw', 'no_shap', 'spectral_only', 'nonlinear_only', 'global_only']:
        if component == 'no_dtw':
            selected_features, X_tr_selected, y_tr_selected = select_features_mrs_shap(X_train_val_features, y_train_val, feature_names, use_dtw=False)
        elif component == 'no_shap':
            selected_features, X_tr_selected, y_tr_selected = select_features_mrs_shap(X_train_val_features, y_train_val, feature_names, use_shap=False)
        elif component == 'spectral_only':
            spectral_features = [f for f in feature_names if any(k in f for k in ['theta', 'alpha', 'beta', 'engagement', 'theta_beta_ratio'])]
            selected_features = spectral_features[:int(0.6*len(spectral_features))]
            X_tr_selected = X_train_val_features[:, [feature_names.index(f) for f in selected_features]]
            y_tr_selected = y_train_val
        elif component == 'nonlinear_only':
            nonlinear_features = [f for f in feature_names if any(k in f for k in ['pac', 'wavelet'])]
            selected_features = nonlinear_features[:int(0.6*len(nonlinear_features))]
            X_tr_selected = X_train_val_features[:, [feature_names.index(f) for f in selected_features]]
            y_tr_selected = y_train_val
        elif component == 'global_only':
            global_features = [f for f in feature_names if any(k in f for k in ['corr', 'coh', 'microstate', 'gfp'])]
            selected_features = global_features[:int(0.6*len(global_features))]
            X_tr_selected = X_train_val_features[:, [feature_names.index(f) for f in selected_features]]
            y_tr_selected = y_train_val

        cat_model = CatBoostClassifier(random_state=42, verbose=0, class_weights={0: 1.0, 1: 2.0})
        cat_model.fit(X_tr_selected, y_tr_selected)
        metrics = evaluate_metrics(cat_model, X_test_features[:, [feature_names.index(f) for f in selected_features]], y_test)
        ablation_results.append((component, metrics[:5]))

    # Sensitivity Analysis
    print("\nPerforming sensitivity analysis...")
    sens_results = sensitivity_analysis(X_train_val_features, y_train_val, feature_names)

    # Aggregate Results
    metrics_names = ['Sensitivity', 'Specificity', 'F1-Score', 'Accuracy', 'AUC']
    result_table = []
    for method in all_results:
        mean_metrics = np.mean(all_results[method], axis=0)
        ci_metrics = [compute_confidence_interval([r[i] for r in all_results[method]]) for i in range(len(metrics_names))]
        result_table.append({
            'Method': method,
            **{m: f"{mean:.3f} ({ci[0]:.3f}--{ci[1]:.3f})" for m, (mean, ci) in zip(metrics_names, zip(mean_metrics, [ci[1] for ci in ci_metrics]))}
        })

    # Statistical Analysis
    print("\nStatistical Analysis:")
    for metric_idx, metric in enumerate(metrics_names):
        print(f"\n{metric}:")
        mrs_shap_scores = [r[metric_idx] for r in all_results['mrs_shap']]
        for method in ['pca', 'rfe', 'anova', 'cnn']:
            other_scores = [r[metric_idx] for r in all_results[method]]
            try:
                stat, p = wilcoxon(mrs_shap_scores, other_scores)
                print(f"Wilcoxon (MRS-SHAP vs. {method}): p={p:.4f} {'(significant)' if p < 0.05 else ''}")
            except:
                print(f"Wilcoxon (MRS-SHAP vs. {method}): Failed, p=1.0000")
            try:
                p_mcnemar = mcnemar_test(y_val, all_predictions['mrs_shap'][fold], all_predictions[method][fold])
                print(f"McNemar (MRS-SHAP vs. {method}): p={p_mcnemar:.4f} {'(significant)' if p_mcnemar < 0.05 else ''}")
            except:
                print(f"McNemar (MRS-SHAP vs. {method}): Failed, p=1.0000")

    p_values = []
    for method in ['pca', 'rfe', 'anova', 'cnn']:
        for i in range(len(metrics_names)):
            mrs_shap_scores = [r[i] for r in all_results['mrs_shap']]
            other_scores = [r[i] for r in all_results[method]]
            try:
                stat, p = wilcoxon(mrs_shap_scores, other_scores)
                p_values.append(p)
            except:
                p_values.append(1.0)
    corrected_p = multipletests(p_values, method='bonferroni')[1]
    print("\nBonferroni-corrected p-values:")
    for i, (method, metric) in enumerate([(m, n) for m in ['pca', 'rfe', 'anova', 'cnn'] for n in metrics_names]):
        print(f"{metric} (MRS-SHAP vs {method}): p={corrected_p[i]:.4f}")

    # Visualizations
    print("\nGenerating visualizations...")
    plt.figure(figsize=(12, 6), dpi=300)
    data = pd.DataFrame({
        'Method': np.repeat(['MRS-SHAP', 'PCA', 'RFE', 'ANOVA', 'CNN'], [len(all_results[m]) for m in ['mrs_shap', 'pca', 'rfe', 'anova', 'cnn']]),
        'Accuracy': np.concatenate([[r[3] for r in all_results[m]] for m in ['mrs_shap', 'pca', 'rfe', 'anova', 'cnn']])
    })
    sns.boxplot(x='Method', y='Accuracy', data=data)
    plt.title('Classification Accuracy Across Methods')
    plt.ylabel('Accuracy')
    plt.xlabel('Method')
    plt.tight_layout()
    plt.savefig('accuracy_comparison.png')
    plt.close()

    # Sensitivity Analysis Visualization
    plt.figure(figsize=(10, 6), dpi=300)
    percentages = [r[0] for r in sens_results]
    accuracies = [r[1][3] for r in sens_results]
    plt.plot(percentages, accuracies, marker='o')
    plt.title('Sensitivity Analysis: Feature Retention vs Accuracy')
    plt.xlabel('Feature Retention Percentage')
    plt.ylabel('Accuracy')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('sensitivity_analysis.png')
    plt.close()

    # Ablation Study Visualization
    plt.figure(figsize=(10, 6), dpi=300)
    components = [r[0] for r in ablation_results]
    accuracies = [r[1][3] for r in ablation_results]
    plt.bar(components, accuracies)
    plt.title('Ablation Study: Component Impact on Accuracy')
    plt.xlabel('Component')
    plt.ylabel('Accuracy')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('ablation_study.png')
    plt.close()

    # Final Evaluation on Test Set
    print("\nFinal Evaluation on Test Set...")
    selected_features, X_train_val_selected, y_train_val_selected = select_features_mrs_shap(X_train_val_features, y_train_val, feature_names)
    X_test_selected = X_test_features[:, [feature_names.index(f) for f in selected_features]]

    # Train ensemble model
    svm = SVC(probability=True, random_state=42, class_weight='balanced')
    lr = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)
    gb = GradientBoostingClassifier(random_state=42)
    cat = CatBoostClassifier(random_state=42, verbose=0, class_weights={0: 1.0, 1: 2.0})
    ensemble = VotingClassifier(
        estimators=[('svm', svm), ('lr', lr), ('gb', gb), ('cat', cat)],
        voting='soft'
    )
    ensemble.fit(X_train_val_selected, y_train_val_selected)
    final_metrics = evaluate_metrics(ensemble, X_test_selected, y_test)
    sensitivity, specificity, f1, accuracy, auc_score, y_pred = final_metrics

    # Display Final Metrics
    print("\nFinal Test Set Metrics (Ensemble with MRS-SHAP):")
    print(f"Sensitivity: {sensitivity:.3f}")
    print(f"Specificity: {specificity:.3f}")
    print(f"F1-Score: {f1:.3f}")
    print(f"Accuracy: {accuracy:.3f}")
    print(f"AUC: {auc_score:.3f}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5), dpi=300)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix (Test Set)')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig('confusion_matrix_test.png')
    plt.close()

    # ROC Curve
    y_scores = ensemble.predict_proba(X_test_selected)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    plt.figure(figsize=(8, 6), dpi=300)
    plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_score:.3f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title('ROC Curve (Test Set)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('roc_curve_test.png')
    plt.close()

    # Feature Importance (SHAP)
    print("\nComputing SHAP feature importance...")
    explainer = shap.KernelExplainer(ensemble.predict_proba, X_train_val_selected)
    shap_values = explainer.shap_values(X_test_selected, nsamples=100)
    plt.figure(figsize=(10, 6), dpi=300)
    shap.summary_plot(shap_values[1], X_test_selected, feature_names=selected_features, show=False)
    plt.title('SHAP Feature Importance (Test Set)')
    plt.tight_layout()
    plt.savefig('shap_importance_test.png')
    plt.close()

        # Save Results
    results_df = pd.DataFrame(result_table)
    results_df.to_csv('classification_results.csv', index=False)
    print("Results saved to 'classification_results.csv'")

    ablation_df = pd.DataFrame(
        [(comp, *metrics) for comp, metrics in ablation_results],
        columns=['Component', 'Sensitivity', 'Specificity', 'F1-Score', 'Accuracy', 'AUC']
    )
    ablation_df.to_csv('ablation_results.csv', index=False)
    print("Ablation results saved to 'ablation_results.csv'")

    sens_df = pd.DataFrame(
        [(p, *metrics) for p, metrics in sens_results],
        columns=['Percentage', 'Sensitivity', 'Specificity', 'F1-Score', 'Accuracy', 'AUC']
    )
    sens_df.to_csv('sensitivity_analysis_results.csv', index=False)
    print("Sensitivity analysis results saved to 'sensitivity_analysis_results.csv'")

    # Save Selected Features
    with open('selected_features.txt', 'w') as f:
        f.write('\n'.join(selected_features))
    print("Selected features saved to 'selected_features.txt'")

    # Save Final Metrics
    final_metrics_df = pd.DataFrame({
        'Metric': ['Sensitivity', 'Specificity', 'F1-Score', 'Accuracy', 'AUC'],
        'Value': [sensitivity, specificity, f1, accuracy, auc_score]
    })
    final_metrics_df.to_csv('final_test_metrics.csv', index=False)
    print("Final test metrics saved to 'final_test_metrics.csv'")

    print(f"\nTotal execution time: {time.time() - start_time:.2f} seconds")
    print_memory_usage()

if __name__ == "__main__":
    main()

Dataset: Resting-state EEG (Nasrabadi et al., 2021), 61 ADHD + 60 controls, 19 channels, 128 Hz
Ethics: Approved by Roozbeh Hospital, Tehran
Loading ADHD data...
Checking folder: C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1
Found 30 .mat files in C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1


Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1:   7%| | 2/30 [00:00<00:0

Loading file: C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1\v10p.mat
Found variable 'v10p' in C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1\v10p.mat
Data shape: (1, 14304, 19)
Max absolute value: 4802.00
Scaled data by 1/1000 (max_abs=4802.00)
Scaled data by 0.0208 (max_abs=4802.00)
Loading file: C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1\v12p.mat
Found variable 'v12p' in C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1\v12p.mat
Data shape: (1, 17604, 19)
Max absolute value: 2891.00
Scaled data by 1/1000 (max_abs=2891.00)
Scaled data by 0.0346 (max_abs=2891.00)
Loading file: C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1\v14p.mat
Found variable 'v14p' in C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1\v14p.mat
Data shape: (1, 17562, 19)
Max absolute value: 3711.00
Scaled data by 1/1000 (max_abs=3711.00)
Scaled data by

Loading files from C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1:  30%|▎| 9/30 [00:00<00:0

Found variable 'v173' in C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1\v173.mat
Data shape: (1, 24241, 19)
Max absolute value: 2260.00
Scaled data by 1/1000 (max_abs=2260.00)
Scaled data by 0.0442 (max_abs=2260.00)
Loading file: C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1\v18p.mat
Found variable 'v18p' in C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1\v18p.mat
Data shape: (1, 25003, 19)
Max absolute value: 4432.00
Scaled data by 1/1000 (max_abs=4432.00)
Scaled data by 0.0226 (max_abs=4432.00)
Loading file: C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1\v19p.mat
Found variable 'v19p' in C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_part1/ADHD_part1\v19p.mat
Data shape: (1, 23063, 19)
Max absolute value: 2849.00
Scaled data by 1/1000 (max_abs=2849.00)
Scaled data by 0.0351 (max_abs=2849.00)
Loading file: C:/Users/ADMIN/Downloads/Calcium signalling research/ADHD_

...

Extracting features: 100%|████████████████████████████████████████████████████| 1657/1657 [16:45<00:00,  1.65segment/s]


Feature matrix shape: (1657, 960), Time=1005.36s


Extracting features: 100%|████████████████████████████████████████████████████| 1685/1685 [27:22<00:00,  1.03segment/s]


Feature matrix shape: (1685, 960), Time=1642.19s
Feature extraction completed: 4789.31s

Selecting features...

Fold 1/3


InvalidParameterError: The 'enn' parameter of SMOTEENN must be an instance of 'imblearn.under_sampling._prototype_selection._edited_nearest_neighbours.EditedNearestNeighbours' or None. Got {'n_neighbors': 5} instead.